In [ ]:
# ==================================================================================================
# LSTM MODEL NOTEBOOK — CELL 1
# STARTUP + FROZEN PROTOCOL VERIFICATION + DATA PATHS
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import shutil
import gc
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    brier_score_loss,
    precision_recall_curve
)


print("=" * 120)
print("LSTM — W10 / H5 + H10")
print("TRAIN + VALIDATION DEVELOPMENT ONLY")
print("TEST IS EMBARGOED")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

P12_2 = (
    ROOT
    / "model_ready"
    / "12_2"
)

P12_3 = (
    ROOT
    / "model_ready"
    / "12_3"
)


READY_12_3 = (
    P12_3
    / "12_3_READY_FOR_MODELING.json"
)

LOCK_12_3 = (
    P12_3
    / "12_3_PROTOCOL_LOCK.json"
)

LOCK_HASH_12_3 = (
    P12_3
    / "12_3_PROTOCOL_LOCK_SHA256.txt"
)

HYPERPARAMETER_POLICY = (
    P12_3
    / "12_3_HYPERPARAMETER_POLICY.json"
)

SEED_POLICY = (
    P12_3
    / "12_3_RANDOM_SEED_POLICY.json"
)

THRESHOLD_POLICY = (
    P12_3
    / "12_3_THRESHOLD_POLICY.json"
)

BINARY_WEIGHTS = (
    P12_2
    / "policy"
    / "BINARY_CLASS_WEIGHTS.csv"
)


# ==================================================================================================
# IMPORTANT:
# NO TEST PATHS ARE DEFINED IN THIS NOTEBOOK.
# ==================================================================================================

X_TRAIN_FILE = (
    P12_2
    / "transformed"
    / "W10"
    / "train"
    / "X_log1p_standardized.npy"
)

X_VAL_FILE = (
    P12_2
    / "transformed"
    / "W10"
    / "validation"
    / "X_log1p_standardized.npy"
)


Y_TRAIN_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)

ELIGIBLE_TRAIN_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)


Y_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "y_binary.npy"
)

ELIGIBLE_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "eligible.npy"
)

VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)


# ==================================================================================================
# OUTPUT
# ==================================================================================================

MODEL_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
)

MODEL_ROOT.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():

        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def read_json(path):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def set_seed(seed):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ==================================================================================================
# VERIFY 12.3
# ==================================================================================================

for path in [
    READY_12_3,
    LOCK_12_3,
    LOCK_HASH_12_3,
    HYPERPARAMETER_POLICY,
    SEED_POLICY,
    THRESHOLD_POLICY,
    BINARY_WEIGHTS,
    X_TRAIN_FILE,
    X_VAL_FILE,
    Y_TRAIN_FILE,
    ELIGIBLE_TRAIN_FILE,
    Y_VAL_FILE,
    ELIGIBLE_VAL_FILE,
    VAL_INDEX_FILE,
]:

    require_file(
        path
    )


lock_hash = sha256_file(
    LOCK_12_3
)

recorded_lock_hash = (

    LOCK_HASH_12_3

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if lock_hash != recorded_lock_hash:

    raise RuntimeError(
        "Phase 12.3 protocol lock failed."
    )


ready = read_json(
    READY_12_3
)


if ready.get(
    "phase_12_3_status"
) != "PASS":

    raise RuntimeError(
        "Phase 12.3 is not PASS."
    )


if ready.get(
    "training_authorized"
) is not True:

    raise RuntimeError(
        "TRAIN/VALIDATION development is not authorized."
    )


if ready.get(
    "test_evaluation_authorized"
) is not False:

    raise RuntimeError(
        "Unexpected TEST authorization."
    )


if ready.get(
    "primary_window_minutes"
) != 10:

    raise RuntimeError(
        "Primary W changed."
    )


if ready.get(
    "primary_horizons_minutes"
) != [
    5,
    10
]:

    raise RuntimeError(
        "Primary horizons changed."
    )


print(
    "✓ Phase 12.3 protocol lock verified"
)

print(
    "✓ TRAIN + VALIDATION modeling authorized"
)

print(
    "✓ TEST evaluation NOT authorized"
)


# ==================================================================================================
# VERIFY FROZEN POLICIES
# ==================================================================================================

hp_policy = read_json(
    HYPERPARAMETER_POLICY
)

seed_policy = read_json(
    SEED_POLICY
)


if hp_policy[
    "LSTM"
][
    "max_trials_per_horizon"
] != 4:

    raise RuntimeError(
        "Frozen LSTM trial budget changed."
    )


if hp_policy[
    "DEEP_COMMON"
][
    "max_epochs"
] != 60:

    raise RuntimeError(
        "Frozen max epoch budget changed."
    )


FINAL_SEEDS = seed_policy[
    "final_replicate_seeds"
]


if FINAL_SEEDS != [
    20260930,
    20260931,
    20260932
]:

    raise RuntimeError(
        "Frozen final replicate seeds changed."
    )


TUNING_SEED = seed_policy[
    "development_tuning_seed"
]


print(
    "✓ LSTM max tuning trials/horizon = 4"
)

print(
    "✓ max epochs = 60"
)

print(
    "✓ tuning seed =",
    TUNING_SEED
)

print(
    "✓ final replicate seeds =",
    FINAL_SEEDS
)


# ==================================================================================================
# DEVICE
# ==================================================================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


print(
    "\nDevice:",
    DEVICE
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        )
    )


# ==================================================================================================
# LOAD MEMORY-MAPPED DATA
# ==================================================================================================

X_TRAIN_ALL = np.load(

    X_TRAIN_FILE,

    mmap_mode="r"
)


X_VAL_ALL = np.load(

    X_VAL_FILE,

    mmap_mode="r"
)


Y_TRAIN_ALL = np.load(

    Y_TRAIN_FILE,

    allow_pickle=False
)


ELIGIBLE_TRAIN_ALL = np.load(

    ELIGIBLE_TRAIN_FILE,

    allow_pickle=False
)


Y_VAL_ALL = np.load(

    Y_VAL_FILE,

    allow_pickle=False
)


ELIGIBLE_VAL_ALL = np.load(

    ELIGIBLE_VAL_FILE,

    allow_pickle=False
)


VAL_INDEX_ALL = pd.read_parquet(
    VAL_INDEX_FILE
)


if X_TRAIN_ALL.shape != (
    59837,
    10,
    14
):

    raise RuntimeError(
        f"Unexpected TRAIN shape: {X_TRAIN_ALL.shape}"
    )


if X_VAL_ALL.shape != (
    12661,
    10,
    14
):

    raise RuntimeError(
        f"Unexpected VALIDATION shape: {X_VAL_ALL.shape}"
    )


if len(
    VAL_INDEX_ALL
) != X_VAL_ALL.shape[
    0
]:

    raise RuntimeError(
        "Validation index alignment failed."
    )


HORIZON_TO_COLUMN = {

    1:
        0,

    5:
        1,

    10:
        2,

    15:
        3,

    30:
        4,
}


WEIGHTS = pd.read_csv(
    BINARY_WEIGHTS
)


def get_horizon_data(H):

    j = HORIZON_TO_COLUMN[
        H
    ]


    train_mask = (
        ELIGIBLE_TRAIN_ALL[
            :,
            j
        ]
        ==
        1
    )


    val_mask = (
        ELIGIBLE_VAL_ALL[
            :,
            j
        ]
        ==
        1
    )


    X_train = np.asarray(

        X_TRAIN_ALL[
            train_mask
        ],

        dtype=np.float32
    )


    y_train = np.asarray(

        Y_TRAIN_ALL[
            train_mask,
            j
        ],

        dtype=np.float32
    )


    X_val = np.asarray(

        X_VAL_ALL[
            val_mask
        ],

        dtype=np.float32
    )


    y_val = np.asarray(

        Y_VAL_ALL[
            val_mask,
            j
        ],

        dtype=np.float32
    )


    val_index = (

        VAL_INDEX_ALL

        .loc[
            val_mask
        ]

        .reset_index(
            drop=True
        )
    )


    if not np.isin(
        y_train,
        [
            0,
            1
        ]
    ).all():

        raise RuntimeError(
            f"H{H}: invalid TRAIN labels."
        )


    if not np.isin(
        y_val,
        [
            0,
            1
        ]
    ).all():

        raise RuntimeError(
            f"H{H}: invalid VALIDATION labels."
        )


    weight_row = WEIGHTS.loc[
        (
            WEIGHTS[
                "History_Window_Minutes"
            ]
            ==
            10
        )

        &

        (
            WEIGHTS[
                "Horizon_Minutes"
            ]
            ==
            H
        )
    ]


    if len(
        weight_row
    ) != 1:

        raise RuntimeError(
            f"H{H}: frozen class weight missing."
        )


    pos_weight = float(

        weight_row[
            "BCE_Pos_Weight"
        ].iloc[
            0
        ]
    )


    return {

        "X_train":
            X_train,

        "y_train":
            y_train,

        "X_val":
            X_val,

        "y_val":
            y_val,

        "val_index":
            val_index,

        "pos_weight":
            pos_weight,
    }


# ==================================================================================================
# VERIFY H5/H10 SUPPORT
# ==================================================================================================

for H in [
    5,
    10
]:

    data = get_horizon_data(
        H
    )


    print(
        f"\nH{H}"
    )

    print(
        "TRAIN:",
        data[
            "X_train"
        ].shape,
        "| positives:",
        int(
            data[
                "y_train"
            ].sum()
        ),
        "| negatives:",
        int(
            (
                data[
                    "y_train"
                ]
                ==
                0
            ).sum()
        )
    )


    print(
        "VALIDATION:",
        data[
            "X_val"
        ].shape,
        "| positives:",
        int(
            data[
                "y_val"
            ].sum()
        ),
        "| negatives:",
        int(
            (
                data[
                    "y_val"
                ]
                ==
                0
            ).sum()
        )
    )


    print(
        "Frozen BCE pos_weight:",
        data[
            "pos_weight"
        ]
    )


    del data

    gc.collect()


print("\n" + "=" * 120)
print("STARTUP COMPLETE")
print("NO TEST DATA LOADED")
print("=" * 120)

In [ ]:
# ==================================================================================================
# LSTM MODEL NOTEBOOK — CELL 2
# MODEL + TRAINING + VALIDATION FUNCTIONS
# ==================================================================================================

import copy
from dataclasses import dataclass


# ==================================================================================================
# FROZEN DEVELOPMENT SETTINGS
# ==================================================================================================

MAX_EPOCHS = 60

PATIENCE = 8

MIN_DELTA = 0.0001

BATCH_SIZE = 256

GRAD_CLIP = 1.0


# ==================================================================================================
# PREDECLARED FOUR LSTM TRIALS
#
# All values are inside the Phase 12.3 frozen search space.
# ==================================================================================================

LSTM_TRIALS = [

    {
        "Config_ID":
            "LSTM_A",

        "hidden_size":
            64,

        "num_layers":
            1,

        "dropout":
            0.1,

        "learning_rate":
            0.001,

        "weight_decay":
            0.0001,
    },


    {
        "Config_ID":
            "LSTM_B",

        "hidden_size":
            128,

        "num_layers":
            1,

        "dropout":
            0.1,

        "learning_rate":
            0.0003,

        "weight_decay":
            0.0001,
    },


    {
        "Config_ID":
            "LSTM_C",

        "hidden_size":
            64,

        "num_layers":
            2,

        "dropout":
            0.3,

        "learning_rate":
            0.001,

        "weight_decay":
            0.001,
    },


    {
        "Config_ID":
            "LSTM_D",

        "hidden_size":
            128,

        "num_layers":
            2,

        "dropout":
            0.3,

        "learning_rate":
            0.0003,

        "weight_decay":
            0.001,
    },
]


if len(
    LSTM_TRIALS
) != 4:

    raise RuntimeError(
        "Frozen LSTM tuning budget requires exactly 4 trials."
    )


# ==================================================================================================
# MODEL
# ==================================================================================================

class LSTMClassifier(
    nn.Module
):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        dropout
    ):

        super().__init__()


        self.lstm = nn.LSTM(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True,

            dropout=(
                dropout

                if num_layers
                >
                1

                else 0.0
            )
        )


        self.output_dropout = nn.Dropout(
            dropout
        )


        self.classifier = nn.Linear(
            hidden_size,
            1
        )


    def forward(
        self,
        x
    ):

        _, (
            h_n,
            _
        ) = self.lstm(
            x
        )


        final_hidden = h_n[
            -1
        ]


        final_hidden = self.output_dropout(
            final_hidden
        )


        logits = self.classifier(
            final_hidden
        ).squeeze(
            1
        )


        return logits


# ==================================================================================================
# DATALOADERS
# ==================================================================================================

def make_loaders(
    X_train,
    y_train,
    X_val,
    y_val,
    seed
):

    train_dataset = TensorDataset(

        torch.from_numpy(
            X_train
        ),

        torch.from_numpy(
            y_train
        )
    )


    val_dataset = TensorDataset(

        torch.from_numpy(
            X_val
        ),

        torch.from_numpy(
            y_val
        )
    )


    generator = torch.Generator()

    generator.manual_seed(
        seed
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        generator=generator,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            ==
            "cuda"
        ),

        drop_last=False
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            ==
            "cuda"
        ),

        drop_last=False
    )


    return (
        train_loader,
        val_loader
    )


# ==================================================================================================
# VALIDATION PREDICTIONS
# ==================================================================================================

@torch.no_grad()
def predict_scores(
    model,
    loader
):

    model.eval()


    all_scores = []

    all_targets = []


    for xb, yb in loader:

        xb = xb.to(
            DEVICE,
            non_blocking=True
        )


        logits = model(
            xb
        )


        scores = torch.sigmoid(
            logits
        )


        all_scores.append(

            scores

            .detach()

            .cpu()

            .numpy()
        )


        all_targets.append(

            yb

            .detach()

            .cpu()

            .numpy()
        )


    scores = np.concatenate(
        all_scores
    )


    targets = np.concatenate(
        all_targets
    )


    return (
        targets.astype(
            np.int8
        ),
        scores.astype(
            np.float64
        )
    )


# ==================================================================================================
# THRESHOLD SELECTION
#
# Phase 12.3:
#
#   maximize VALIDATION F1
#
# ties:
#
#   higher precision
#   higher threshold
# ==================================================================================================

def choose_validation_threshold(
    y_true,
    scores
):

    precision, recall, thresholds = precision_recall_curve(

        y_true,

        scores
    )


    if len(
        thresholds
    ) == 0:

        return 0.5


    precision_t = precision[
        :-1
    ]

    recall_t = recall[
        :-1
    ]


    denom = (
        precision_t
        +
        recall_t
    )


    f1 = np.divide(

        2.0
        *
        precision_t
        *
        recall_t,

        denom,

        out=np.zeros_like(
            denom
        ),

        where=(
            denom
            >
            0
        )
    )


    table = pd.DataFrame(
        {
            "threshold":
                thresholds,

            "precision":
                precision_t,

            "recall":
                recall_t,

            "f1":
                f1,
        }
    )


    best_f1 = table[
        "f1"
    ].max()


    candidates = table.loc[
        np.isclose(
            table[
                "f1"
            ],
            best_f1,
            atol=1e-12,
            rtol=0.0
        )
    ].copy()


    candidates = candidates.sort_values(

        [
            "precision",
            "threshold"
        ],

        ascending=[
            False,
            False
        ]
    )


    return float(

        candidates.iloc[
            0
        ][
            "threshold"
        ]
    )


# ==================================================================================================
# BINARY METRICS
# ==================================================================================================

def calculate_binary_metrics(
    y_true,
    scores,
    threshold
):

    pred = (
        scores
        >=
        threshold
    ).astype(
        np.int8
    )


    return {

        "Average_Precision":
            float(
                average_precision_score(
                    y_true,
                    scores
                )
            ),

        "ROC_AUC":
            float(
                roc_auc_score(
                    y_true,
                    scores
                )
            ),

        "Precision":
            float(
                precision_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "Recall":
            float(
                recall_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    pred
                )
            ),

        "Brier":
            float(
                brier_score_loss(
                    y_true,
                    scores
                )
            ),

        "Threshold":
            float(
                threshold
            ),
    }


# ==================================================================================================
# EVENT METRICS
# ==================================================================================================

def calculate_event_metrics(
    y_true,
    scores,
    threshold,
    val_index
):

    pred = (
        scores
        >=
        threshold
    )


    y_true = np.asarray(
        y_true,
        dtype=np.int8
    )


    episodes = (

        val_index[
            "Episode_ID"
        ]

        .astype(
            str
        )

        .to_numpy()
    )


    tto = pd.to_numeric(

        val_index[
            "Minutes_To_Next_Onset"
        ],

        errors="coerce"

    ).to_numpy(
        dtype=float
    )


    positive = (
        y_true
        ==
        1
    )


    negative = (
        y_true
        ==
        0
    )


    tp_warning = (
        pred
        &
        positive
    )


    fp_warning = (
        pred
        &
        negative
    )


    true_events = set(

        episodes[
            positive
        ]
    )


    recalled_events = set(

        episodes[
            tp_warning
        ]
    )


    warned_episodes = set(

        episodes[
            pred
        ]
    )


    event_recall = (

        len(
            recalled_events
        )

        /

        len(
            true_events
        )

        if len(
            true_events
        )

        else np.nan
    )


    event_precision = (

        len(
            recalled_events
        )

        /

        len(
            warned_episodes
        )

        if len(
            warned_episodes
        )

        else np.nan
    )


    leads = []


    for event_id in recalled_events:

        event_warning = (

            tp_warning

            &

            (
                episodes
                ==
                event_id
            )
        )


        event_tto = tto[
            event_warning
        ]


        event_tto = event_tto[
            np.isfinite(
                event_tto
            )
        ]


        if len(
            event_tto
        ):

            # First valid warning is the farthest warning before onset.

            leads.append(
                float(
                    np.max(
                        event_tto
                    )
                )
            )


    negative_hours = (

        negative.sum()

        /
        60.0
    )


    false_alerts_per_hour = (

        float(
            fp_warning.sum()
        )

        /
        negative_hours

        if negative_hours
        >
        0

        else np.nan
    )


    return {

        "Total_True_Events":
            int(
                len(
                    true_events
                )
            ),

        "Recalled_Events":
            int(
                len(
                    recalled_events
                )
            ),

        "Warned_Episodes":
            int(
                len(
                    warned_episodes
                )
            ),

        "Event_Recall":
            float(
                event_recall
            ),

        "Event_Precision":
            float(
                event_precision
            ),

        "Median_First_Warning_Lead_Min":
            (
                float(
                    np.median(
                        leads
                    )
                )

                if len(
                    leads
                )

                else np.nan
            ),

        "Mean_First_Warning_Lead_Min":
            (
                float(
                    np.mean(
                        leads
                    )
                )

                if len(
                    leads
                )

                else np.nan
            ),

        "False_Positive_Warning_Origins":
            int(
                fp_warning.sum()
            ),

        "False_Alerts_Per_Hour":
            float(
                false_alerts_per_hour
            ),
    }


# ==================================================================================================
# TRAIN ONE RUN
# ==================================================================================================

def train_one_run(
    *,
    H,
    config,
    seed,
    X_train,
    y_train,
    X_val,
    y_val,
    pos_weight
):

    set_seed(
        seed
    )


    train_loader, val_loader = make_loaders(

        X_train,
        y_train,
        X_val,
        y_val,
        seed
    )


    model = LSTMClassifier(

        input_size=14,

        hidden_size=config[
            "hidden_size"
        ],

        num_layers=config[
            "num_layers"
        ],

        dropout=config[
            "dropout"
        ]
    ).to(
        DEVICE
    )


    criterion = nn.BCEWithLogitsLoss(

        pos_weight=torch.tensor(

            pos_weight,

            dtype=torch.float32,

            device=DEVICE
        )
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=config[
            "learning_rate"
        ],

        weight_decay=config[
            "weight_decay"
        ]
    )


    best_ap = -np.inf

    best_epoch = None

    best_state = None

    epochs_without_improvement = 0


    history = []


    for epoch in range(
        1,
        MAX_EPOCHS
        +
        1
    ):

        model.train()


        running_loss = 0.0

        observed = 0


        for xb, yb in train_loader:

            xb = xb.to(
                DEVICE,
                non_blocking=True
            )


            yb = yb.to(
                DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits = model(
                xb
            )


            loss = criterion(
                logits,
                yb
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                GRAD_CLIP
            )


            optimizer.step()


            running_loss += float(
                loss.item()
            ) * len(
                xb
            )


            observed += len(
                xb
            )


        train_loss = (

            running_loss

            /
            observed
        )


        val_targets, val_scores = predict_scores(

            model,

            val_loader
        )


        val_ap = float(

            average_precision_score(

                val_targets,

                val_scores
            )
        )


        history.append(
            {
                "Epoch":
                    epoch,

                "Train_Loss":
                    train_loss,

                "Validation_AP":
                    val_ap,
            }
        )


        improved = (

            val_ap

            >

            best_ap
            +
            MIN_DELTA
        )


        if improved:

            best_ap = val_ap

            best_epoch = epoch

            best_state = {

                key:
                    value
                    .detach()
                    .cpu()
                    .clone()

                for key, value
                in model.state_dict().items()
            }


            epochs_without_improvement = 0


        else:

            epochs_without_improvement += 1


        if (
            epoch
            ==
            1

            or

            epoch
            %
            5
            ==
            0

            or

            improved
        ):

            print(
                f"H{H} | "
                f"{config['Config_ID']} | "
                f"seed={seed} | "
                f"epoch={epoch:02d} | "
                f"loss={train_loss:.5f} | "
                f"val_AP={val_ap:.6f} | "
                f"best={best_ap:.6f}"
            )


        if (
            epochs_without_improvement
            >=
            PATIENCE
        ):

            print(
                f"Early stopping at epoch {epoch}; "
                f"best epoch={best_epoch}"
            )

            break


    if best_state is None:

        raise RuntimeError(
            "Training produced no best model state."
        )


    model.load_state_dict(
        best_state
    )


    model = model.to(
        DEVICE
    )


    final_targets, final_scores = predict_scores(

        model,

        val_loader
    )


    final_ap = float(

        average_precision_score(

            final_targets,

            final_scores
        )
    )


    threshold = choose_validation_threshold(

        final_targets,

        final_scores
    )


    metrics = calculate_binary_metrics(

        final_targets,

        final_scores,

        threshold
    )


    history_df = pd.DataFrame(
        history
    )


    return {

        "model":
            model,

        "state_dict":
            best_state,

        "history":
            history_df,

        "val_targets":
            final_targets,

        "val_scores":
            final_scores,

        "metrics":
            metrics,

        "best_epoch":
            int(
                best_epoch
            ),

        "best_ap":
            final_ap,

        "threshold":
            threshold,

        "parameter_count":
            int(
                sum(
                    p.numel()

                    for p
                    in model.parameters()
                )
            ),
    }


print(
    "✓ LSTM architecture ready"
)

print(
    "✓ 4 frozen tuning configurations ready"
)

print(
    "✓ validation AP early stopping ready"
)

print(
    "✓ validation F1 threshold selection ready"
)

print(
    "✓ event metrics ready"
)

print(
    "✓ TEST is still not loaded"
)

In [ ]:
# ==================================================================================================
# LSTM MODEL NOTEBOOK — CELL 3
# TUNE H5/H10 → FINAL 3-SEED RUNS → SAVE VALIDATION RESULTS
# ==================================================================================================

print("=" * 120)
print("LSTM TRAINING START")
print("W10 / H5 + H10")
print("TEST REMAINS EMBARGOED")
print("=" * 120)


# ==================================================================================================
# OUTPUT SAFETY
# ==================================================================================================

if MODEL_ROOT.exists():

    completed_lock = (
        MODEL_ROOT
        / "LSTM_DEVELOPMENT_LOCK.json"
    )


    if completed_lock.is_file():

        print(
            "A completed LSTM development run already exists:"
        )

        print(
            MODEL_ROOT
        )

        raise SystemExit


    attempt = 1


    while (

        MODEL_ROOT.parent
        / f"_incomplete_LSTM_attempt_{attempt}"

    ).exists():

        attempt += 1


    backup = (

        MODEL_ROOT.parent
        / f"_incomplete_LSTM_attempt_{attempt}"
    )


    shutil.move(
        str(
            MODEL_ROOT
        ),
        str(
            backup
        )
    )


    print(
        "Previous incomplete attempt preserved:"
    )

    print(
        backup
    )


MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# SAVE NOTEBOOK CONTRACT
# ==================================================================================================

write_json(
    {
        "Model":
            "LSTM",

        "History_Window_Minutes":
            10,

        "Horizons":
            [
                5,
                10
            ],

        "Input_Features":
            14,

        "Representation":
            "Phase 12.2 log1p + TRAIN-only StandardScaler",

        "Training_Distribution":
            "natural TRAIN",

        "Loss":
            "BCEWithLogitsLoss",

        "Class_Weight":
            "Phase 12.2 frozen BCE_Pos_Weight",

        "Tuning_Trials_Per_Horizon":
            4,

        "Tuning_Seed":
            TUNING_SEED,

        "Final_Replicate_Seeds":
            FINAL_SEEDS,

        "Selection_Metric":
            "Validation Average Precision",

        "Threshold":
            "Validation F1 optimum",

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "Phase_12_3_Protocol_Lock_SHA256":
            lock_hash,
    },

    MODEL_ROOT
    / "LSTM_EXPERIMENT_CONTRACT.json"
)


# ==================================================================================================
# STORAGE
# ==================================================================================================

all_tuning_rows = []

all_final_rows = []

horizon_summaries = []


# ==================================================================================================
# HORIZON LOOP
# ==================================================================================================

for H in [
    5,
    10
]:

    print("\n" + "=" * 120)
    print(
        f"LSTM — H{H}"
    )
    print("=" * 120)


    horizon_dir = (
        MODEL_ROOT
        / f"H{H:02d}"
    )


    tuning_dir = (
        horizon_dir
        / "tuning"
    )


    final_dir = (
        horizon_dir
        / "final_replicates"
    )


    tuning_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    final_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # ==============================================================================================
    # LOAD THIS HORIZON ONLY
    # ==============================================================================================

    data = get_horizon_data(
        H
    )


    X_train = data[
        "X_train"
    ]

    y_train = data[
        "y_train"
    ]

    X_val = data[
        "X_val"
    ]

    y_val = data[
        "y_val"
    ]

    val_index = data[
        "val_index"
    ]

    pos_weight = data[
        "pos_weight"
    ]


    print(
        "TRAIN:",
        X_train.shape,
        "positive=",
        int(
            y_train.sum()
        ),
        "negative=",
        int(
            (
                y_train
                ==
                0
            ).sum()
        )
    )


    print(
        "VALIDATION:",
        X_val.shape,
        "positive=",
        int(
            y_val.sum()
        ),
        "negative=",
        int(
            (
                y_val
                ==
                0
            ).sum()
        )
    )


    print(
        "Frozen pos_weight:",
        pos_weight
    )


    # ==============================================================================================
    # FOUR FROZEN HYPERPARAMETER TRIALS
    # ==============================================================================================

    tuning_results = []


    for trial_number, config in enumerate(
        LSTM_TRIALS,
        start=1
    ):

        print("\n" + "-" * 120)

        print(
            f"H{H} TUNING TRIAL "
            f"{trial_number}/4 — "
            f"{config['Config_ID']}"
        )

        print(
            config
        )

        print("-" * 120)


        result = train_one_run(

            H=H,

            config=config,

            seed=TUNING_SEED,

            X_train=X_train,

            y_train=y_train,

            X_val=X_val,

            y_val=y_val,

            pos_weight=pos_weight
        )


        event_metrics = calculate_event_metrics(

            result[
                "val_targets"
            ],

            result[
                "val_scores"
            ],

            result[
                "threshold"
            ],

            val_index
        )


        trial_dir = (

            tuning_dir
            / config[
                "Config_ID"
            ]
        )


        trial_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        result[
            "history"
        ].to_csv(

            trial_dir
            / "training_history.csv",

            index=False
        )


        metrics_record = {

            "Horizon":
                H,

            "Trial":
                trial_number,

            **config,

            "Seed":
                TUNING_SEED,

            "Best_Epoch":
                result[
                    "best_epoch"
                ],

            "Parameters":
                result[
                    "parameter_count"
                ],

            **result[
                "metrics"
            ],

            **event_metrics,
        }


        write_json(

            metrics_record,

            trial_dir
            / "validation_metrics.json"
        )


        tuning_results.append(
            metrics_record
        )


        all_tuning_rows.append(
            metrics_record
        )


        print(
            f"\nRESULT | H{H} | "
            f"{config['Config_ID']} | "
            f"AP={result['metrics']['Average_Precision']:.6f} | "
            f"F1={result['metrics']['F1']:.6f} | "
            f"EventRecall={event_metrics['Event_Recall']:.6f}"
        )


        del result

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    # ==============================================================================================
    # SELECT CONFIGURATION USING FROZEN 12.3 RULE
    #
    # First:
    #     highest validation AP
    #
    # AP tie tolerance:
    #     0.0001
    #
    # tie:
    #     F1
    #     lower complexity
    #     lexical Config_ID
    # ==============================================================================================

    tuning_df = pd.DataFrame(
        tuning_results
    )


    tuning_df.to_csv(

        horizon_dir
        / "tuning_summary.csv",

        index=False
    )


    best_ap = tuning_df[
        "Average_Precision"
    ].max()


    candidates = tuning_df.loc[
        (
            best_ap

            -

            tuning_df[
                "Average_Precision"
            ]
        )

        <=
        0.0001
    ].copy()


    candidates[
        "Complexity"
    ] = (

        candidates[
            "hidden_size"
        ]

        *

        candidates[
            "num_layers"
        ]
    )


    candidates = candidates.sort_values(

        [
            "F1",

            "Complexity",

            "Config_ID",
        ],

        ascending=[
            False,
            True,
            True,
        ]
    )


    selected_row = candidates.iloc[
        0
    ]


    selected_config_id = selected_row[
        "Config_ID"
    ]


    selected_config = next(

        config

        for config in LSTM_TRIALS

        if config[
            "Config_ID"
        ]
        ==
        selected_config_id
    )


    print("\n" + "=" * 120)

    print(
        f"H{H} SELECTED CONFIGURATION"
    )

    print(
        selected_config
    )

    print(
        "Tuning validation AP:",
        float(
            selected_row[
                "Average_Precision"
            ]
        )
    )

    print("=" * 120)


    write_json(
        {
            "Horizon":
                H,

            "Selection_Rule":
                (
                    "max validation AP; within 0.0001 AP tie "
                    "use higher F1, lower complexity, Config_ID"
                ),

            "Selected_Config":
                selected_config,

            "Tuning_Validation_AP":
                float(
                    selected_row[
                        "Average_Precision"
                    ]
                ),

            "Tuning_Validation_F1":
                float(
                    selected_row[
                        "F1"
                    ]
                ),
        },

        horizon_dir
        / "selected_configuration.json"
    )


    # ==============================================================================================
    # FINAL THREE FROZEN REPLICATE SEEDS
    # ==============================================================================================

    replicate_rows = []


    for replicate_number, seed in enumerate(
        FINAL_SEEDS,
        start=1
    ):

        print("\n" + "#" * 120)

        print(
            f"H{H} FINAL REPLICATE "
            f"{replicate_number}/3 — "
            f"seed={seed}"
        )

        print("#" * 120)


        result = train_one_run(

            H=H,

            config=selected_config,

            seed=seed,

            X_train=X_train,

            y_train=y_train,

            X_val=X_val,

            y_val=y_val,

            pos_weight=pos_weight
        )


        binary_metrics = result[
            "metrics"
        ]


        event_metrics = calculate_event_metrics(

            result[
                "val_targets"
            ],

            result[
                "val_scores"
            ],

            result[
                "threshold"
            ],

            val_index
        )


        replicate_dir = (

            final_dir
            / f"seed_{seed}"
        )


        replicate_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        # ------------------------------------------------------------------------------------------
        # Save model
        # ------------------------------------------------------------------------------------------

        model_payload = {

            "model_type":
                "LSTM",

            "horizon":
                H,

            "history_window":
                10,

            "feature_count":
                14,

            "seed":
                seed,

            "config":
                selected_config,

            "state_dict":
                result[
                    "state_dict"
                ],

            "validation_threshold":
                result[
                    "threshold"
                ],

            "phase_12_3_protocol_lock_sha256":
                lock_hash,
        }


        torch.save(

            model_payload,

            replicate_dir
            / "model.pt"
        )


        # ------------------------------------------------------------------------------------------
        # Save history
        # ------------------------------------------------------------------------------------------

        result[
            "history"
        ].to_csv(

            replicate_dir
            / "training_history.csv",

            index=False
        )


        # ------------------------------------------------------------------------------------------
        # Save validation predictions
        # ------------------------------------------------------------------------------------------

        validation_predictions = val_index[
            [
                "Episode_ID",
                "Minute_Index",
                "Minutes_To_Next_Onset"
            ]
        ].copy()


        validation_predictions[
            "Y_True"
        ] = result[
            "val_targets"
        ]


        validation_predictions[
            "Probability"
        ] = result[
            "val_scores"
        ]


        validation_predictions[
            "Threshold"
        ] = result[
            "threshold"
        ]


        validation_predictions[
            "Prediction"
        ] = (

            result[
                "val_scores"
            ]

            >=

            result[
                "threshold"
            ]
        ).astype(
            np.int8
        )


        validation_predictions.to_parquet(

            replicate_dir
            / "validation_predictions.parquet",

            index=False
        )


        # ------------------------------------------------------------------------------------------
        # Metrics
        # ------------------------------------------------------------------------------------------

        replicate_record = {

            "Model":
                "LSTM",

            "Horizon":
                H,

            "Replicate":
                replicate_number,

            "Seed":
                seed,

            "Config_ID":
                selected_config[
                    "Config_ID"
                ],

            "Best_Epoch":
                result[
                    "best_epoch"
                ],

            "Parameters":
                result[
                    "parameter_count"
                ],

            **binary_metrics,

            **event_metrics,

            "TEST_Used":
                False,
        }


        write_json(

            replicate_record,

            replicate_dir
            / "validation_metrics.json"
        )


        replicate_rows.append(
            replicate_record
        )


        all_final_rows.append(
            replicate_record
        )


        print(
            f"\nFINAL VALIDATION RESULT | "
            f"H{H} | seed={seed}"
        )


        print(
            f"AP          = "
            f"{binary_metrics['Average_Precision']:.6f}"
        )


        print(
            f"ROC-AUC     = "
            f"{binary_metrics['ROC_AUC']:.6f}"
        )


        print(
            f"Precision   = "
            f"{binary_metrics['Precision']:.6f}"
        )


        print(
            f"Recall      = "
            f"{binary_metrics['Recall']:.6f}"
        )


        print(
            f"F1          = "
            f"{binary_metrics['F1']:.6f}"
        )


        print(
            f"Threshold   = "
            f"{binary_metrics['Threshold']:.6f}"
        )


        print(
            f"Event Recall = "
            f"{event_metrics['Event_Recall']:.6f}"
        )


        print(
            f"Median Lead  = "
            f"{event_metrics['Median_First_Warning_Lead_Min']}"
        )


        print(
            f"False alerts/hour = "
            f"{event_metrics['False_Alerts_Per_Hour']:.6f}"
        )


        del result

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    # ==============================================================================================
    # HORIZON REPLICATE SUMMARY
    # ==============================================================================================

    replicate_df = pd.DataFrame(
        replicate_rows
    )


    replicate_df.to_csv(

        horizon_dir
        / "final_replicate_summary.csv",

        index=False
    )


    summary = {

        "Model":
            "LSTM",

        "Horizon":
            H,

        "Config_ID":
            selected_config[
                "Config_ID"
            ],

        "Replicate_Count":
            len(
                replicate_df
            ),

        "Validation_AP_Mean":
            float(
                replicate_df[
                    "Average_Precision"
                ].mean()
            ),

        "Validation_AP_Std":
            float(
                replicate_df[
                    "Average_Precision"
                ].std(
                    ddof=1
                )
            ),

        "Validation_F1_Mean":
            float(
                replicate_df[
                    "F1"
                ].mean()
            ),

        "Validation_F1_Std":
            float(
                replicate_df[
                    "F1"
                ].std(
                    ddof=1
                )
            ),

        "Event_Recall_Mean":
            float(
                replicate_df[
                    "Event_Recall"
                ].mean()
            ),

        "Median_Lead_Mean_Across_Replicates":
            float(
                replicate_df[
                    "Median_First_Warning_Lead_Min"
                ].mean()
            ),

        "False_Alerts_Per_Hour_Mean":
            float(
                replicate_df[
                    "False_Alerts_Per_Hour"
                ].mean()
            ),

        "TEST_Used":
            False,
    }


    write_json(

        summary,

        horizon_dir
        / "final_validation_summary.json"
    )


    horizon_summaries.append(
        summary
    )


    print("\n" + "=" * 120)

    print(
        f"H{H} FINAL VALIDATION SUMMARY"
    )


    print(
        "Mean AP:",
        summary[
            "Validation_AP_Mean"
        ]
    )


    print(
        "AP std:",
        summary[
            "Validation_AP_Std"
        ]
    )


    print(
        "Mean F1:",
        summary[
            "Validation_F1_Mean"
        ]
    )


    print(
        "Mean Event Recall:",
        summary[
            "Event_Recall_Mean"
        ]
    )


    print("=" * 120)


    # ==============================================================================================
    # Release horizon arrays from RAM
    # ==============================================================================================

    del X_train

    del y_train

    del X_val

    del y_val

    del val_index

    del data

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ==================================================================================================
# GLOBAL SAVE
# ==================================================================================================

pd.DataFrame(
    all_tuning_rows
).to_csv(

    MODEL_ROOT
    / "ALL_LSTM_TUNING_RESULTS.csv",

    index=False
)


pd.DataFrame(
    all_final_rows
).to_csv(

    MODEL_ROOT
    / "ALL_LSTM_FINAL_REPLICATES.csv",

    index=False
)


horizon_summary_df = pd.DataFrame(
    horizon_summaries
)


horizon_summary_df.to_csv(

    MODEL_ROOT
    / "LSTM_VALIDATION_SUMMARY.csv",

    index=False
)


print("\n" + "=" * 120)
print("LSTM VALIDATION SUMMARY")
print("=" * 120)


print(
    horizon_summary_df.to_string(
        index=False
    )
)


# ==================================================================================================
# DEVELOPMENT LOCK
# ==================================================================================================

development_lock = {

    "Model":
        "LSTM",

    "Status":
        "TRAIN_VALIDATION_DEVELOPMENT_COMPLETE",

    "History_Window_Minutes":
        10,

    "Horizons":
        [
            5,
            10
        ],

    "Tuning_Trials_Per_Horizon":
        4,

    "Tuning_Seed":
        TUNING_SEED,

    "Final_Replicate_Seeds":
        FINAL_SEEDS,

    "Primary_Selection_Metric":
        "Validation Average Precision",

    "Threshold_Selection":
        "Validation F1",

    "TEST_Loaded":
        False,

    "TEST_Used":
        False,

    "TEST_Evaluation_Performed":
        False,

    "Phase_12_3_Protocol_Lock_SHA256":
        lock_hash,

    "Next":
        "LOGISTIC_REGRESSION_OR_OTHER_PREDECLARED_MODEL_DEVELOPMENT",
}


write_json(

    development_lock,

    MODEL_ROOT
    / "LSTM_DEVELOPMENT_LOCK.json"
)


lock_digest = sha256_file(

    MODEL_ROOT
    / "LSTM_DEVELOPMENT_LOCK.json"
)


(
    MODEL_ROOT
    / "LSTM_DEVELOPMENT_LOCK_SHA256.txt"
).write_text(
    lock_digest
    +
    "\n",
    encoding="utf-8"
)


print("\n" + "=" * 120)
print("✓ LSTM TRAIN + VALIDATION DEVELOPMENT COMPLETE")
print("=" * 120)


print(
    "\nH5 + H10 trained ✓"
)

print(
    "4 frozen tuning trials per horizon ✓"
)

print(
    "3 frozen final replicate seeds per horizon ✓"
)

print(
    "Validation AP used for model selection ✓"
)

print(
    "Validation F1 used for thresholds ✓"
)

print(
    "Event metrics calculated on validation ✓"
)

print(
    "Models/results saved to:"
)

print(
    MODEL_ROOT
)


print(
    "\nTEST:"
)

print(
    "NOT LOADED ✓"
)

print(
    "NOT EVALUATED ✓"
)

print(
    "STILL EMBARGOED ✓"
)


print("\nNEXT:")
print(
    "CURRENT-MINUTE + TEMPORAL LOGISTIC REGRESSION"
)


print("=" * 120)

In [ ]:
# ==================================================================================================
# LSTM MODEL NOTEBOOK — CELL 4
# FULL VALIDATION DIAGNOSTIC REPORT
# ==================================================================================================
#
# PURPOSE
# -------
# Evaluate the already-trained LSTM models much more completely.
#
# NO TRAINING.
# NO HYPERPARAMETER CHANGES.
# NO TEST DATA.
#
# Metrics:
#
#   Accuracy
#   Balanced Accuracy
#   Precision
#   Recall / Sensitivity
#   Specificity
#   F1
#   MCC
#   ROC-AUC
#   Average Precision
#   trapezoidal PR-AUC
#   Brier Score
#
#   TP / FP / TN / FN
#
#   Positive prevalence
#   always-negative accuracy
#   AP lift over prevalence
#
#   Event Recall
#   Event Precision
#   First-warning lead time
#   False-positive warning origins/hour
#
#   Mean ± std across the 3 frozen seeds
#
# Also saves:
#
#   detailed per-seed metrics
#   summary metrics
#   PR curves
#   ROC curves
#   confusion matrices
#
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    auc
)


print("=" * 120)
print("LSTM — FULL VALIDATION DIAGNOSTIC REPORT")
print("NO RETRAINING")
print("NO TEST DATA")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

MODEL_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
)

DEVELOPMENT_LOCK = (
    MODEL_ROOT
    / "LSTM_DEVELOPMENT_LOCK.json"
)

DIAGNOSTIC_ROOT = (
    MODEL_ROOT
    / "validation_diagnostics"
)


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():

        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )


def read_json(path):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


# ==================================================================================================
# VERIFY COMPLETED LSTM RUN
# ==================================================================================================

lock = read_json(
    DEVELOPMENT_LOCK
)


if lock.get(
    "Status"
) != "TRAIN_VALIDATION_DEVELOPMENT_COMPLETE":

    raise RuntimeError(
        "LSTM development run is not marked complete."
    )


if lock.get(
    "TEST_Loaded"
) is not False:

    raise RuntimeError(
        "Development lock indicates TEST was loaded."
    )


if lock.get(
    "TEST_Used"
) is not False:

    raise RuntimeError(
        "Development lock indicates TEST was used."
    )


if lock.get(
    "TEST_Evaluation_Performed"
) is not False:

    raise RuntimeError(
        "Development lock indicates TEST evaluation occurred."
    )


FINAL_SEEDS = lock[
    "Final_Replicate_Seeds"
]


if FINAL_SEEDS != [
    20260930,
    20260931,
    20260932
]:

    raise RuntimeError(
        "Unexpected replicate seeds."
    )


DIAGNOSTIC_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "✓ Completed LSTM run verified"
)

print(
    "✓ TEST was not loaded"
)

print(
    "✓ TEST was not evaluated"
)

print(
    "✓ Frozen seeds:",
    FINAL_SEEDS
)


# ==================================================================================================
# METRIC FUNCTION
# ==================================================================================================

def evaluate_predictions(
    predictions
):

    y_true = predictions[
        "Y_True"
    ].to_numpy(
        dtype=np.int8
    )


    scores = predictions[
        "Probability"
    ].to_numpy(
        dtype=np.float64
    )


    pred = predictions[
        "Prediction"
    ].to_numpy(
        dtype=np.int8
    )


    threshold_values = predictions[
        "Threshold"
    ].to_numpy(
        dtype=np.float64
    )


    unique_thresholds = np.unique(
        threshold_values
    )


    if len(
        unique_thresholds
    ) != 1:

        raise RuntimeError(
            "Multiple thresholds found within one replicate."
        )


    threshold = float(
        unique_thresholds[
            0
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # CONFUSION MATRIX
    # ----------------------------------------------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(

        y_true,

        pred,

        labels=[
            0,
            1
        ]

    ).ravel()


    sensitivity = (

        tp

        /

        (
            tp
            +
            fn
        )

        if (
            tp
            +
            fn
        )

        else np.nan
    )


    specificity = (

        tn

        /

        (
            tn
            +
            fp
        )

        if (
            tn
            +
            fp
        )

        else np.nan
    )


    negative_predictive_value = (

        tn

        /

        (
            tn
            +
            fn
        )

        if (
            tn
            +
            fn
        )

        else np.nan
    )


    false_positive_rate = (

        fp

        /

        (
            fp
            +
            tn
        )

        if (
            fp
            +
            tn
        )

        else np.nan
    )


    false_negative_rate = (

        fn

        /

        (
            fn
            +
            tp
        )

        if (
            fn
            +
            tp
        )

        else np.nan
    )


    # ----------------------------------------------------------------------------------------------
    # THRESHOLD-FREE METRICS
    # ----------------------------------------------------------------------------------------------

    roc_auc = roc_auc_score(

        y_true,

        scores
    )


    average_precision = average_precision_score(

        y_true,

        scores
    )


    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(

        y_true,

        scores
    )


    # sklearn returns recall in descending order.
    # auc() handles monotonic descending x correctly.

    trapezoidal_pr_auc = auc(

        recall_curve,

        precision_curve
    )


    fpr_curve, tpr_curve, roc_thresholds = roc_curve(

        y_true,

        scores
    )


    # ----------------------------------------------------------------------------------------------
    # PREVALENCE / BASELINES
    # ----------------------------------------------------------------------------------------------

    prevalence = float(
        y_true.mean()
    )


    always_negative_accuracy = float(
        (
            y_true
            ==
            0
        ).mean()
    )


    ap_lift = (

        average_precision

        /

        prevalence

        if prevalence
        >
        0

        else np.nan
    )


    # ----------------------------------------------------------------------------------------------
    # EVENT METRICS
    # ----------------------------------------------------------------------------------------------

    episodes = (

        predictions[
            "Episode_ID"
        ]

        .astype(
            str
        )

        .to_numpy()
    )


    tto = pd.to_numeric(

        predictions[
            "Minutes_To_Next_Onset"
        ],

        errors="coerce"

    ).to_numpy(
        dtype=float
    )


    positive = (
        y_true
        ==
        1
    )


    negative = (
        y_true
        ==
        0
    )


    warnings = (
        pred
        ==
        1
    )


    true_positive_warnings = (

        warnings

        &
        positive
    )


    false_positive_warnings = (

        warnings

        &
        negative
    )


    true_event_ids = set(

        episodes[
            positive
        ]
    )


    recalled_event_ids = set(

        episodes[
            true_positive_warnings
        ]
    )


    warned_episode_ids = set(

        episodes[
            warnings
        ]
    )


    event_recall = (

        len(
            recalled_event_ids
        )

        /

        len(
            true_event_ids
        )

        if len(
            true_event_ids
        )

        else np.nan
    )


    event_precision = (

        len(
            recalled_event_ids
        )

        /

        len(
            warned_episode_ids
        )

        if len(
            warned_episode_ids
        )

        else np.nan
    )


    first_warning_leads = []


    for episode_id in recalled_event_ids:

        episode_mask = (

            true_positive_warnings

            &

            (
                episodes
                ==
                episode_id
            )
        )


        leads = tto[
            episode_mask
        ]


        leads = leads[
            np.isfinite(
                leads
            )
        ]


        if len(
            leads
        ):

            # Farthest warning before onset = first warning.

            first_warning_leads.append(

                float(
                    np.max(
                        leads
                    )
                )
            )


    negative_origin_hours = (

        negative.sum()

        /
        60.0
    )


    false_warning_origins_per_hour = (

        false_positive_warnings.sum()

        /

        negative_origin_hours

        if negative_origin_hours
        >
        0

        else np.nan
    )


    metrics = {

        # ------------------------------------------------------------------------------------------
        # COUNTS
        # ------------------------------------------------------------------------------------------

        "Samples":
            int(
                len(
                    y_true
                )
            ),

        "Positive_Samples":
            int(
                positive.sum()
            ),

        "Negative_Samples":
            int(
                negative.sum()
            ),


        # ------------------------------------------------------------------------------------------
        # CONFUSION MATRIX
        # ------------------------------------------------------------------------------------------

        "TP":
            int(
                tp
            ),

        "FP":
            int(
                fp
            ),

        "TN":
            int(
                tn
            ),

        "FN":
            int(
                fn
            ),


        # ------------------------------------------------------------------------------------------
        # THRESHOLD
        # ------------------------------------------------------------------------------------------

        "Threshold":
            threshold,


        # ------------------------------------------------------------------------------------------
        # STANDARD CLASSIFICATION
        # ------------------------------------------------------------------------------------------

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    pred
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    pred
                )
            ),

        "Precision":
            float(
                precision_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "Recall_Sensitivity":
            float(
                sensitivity
            ),

        "Specificity":
            float(
                specificity
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y_true,
                    pred
                )
            ),

        "NPV":
            float(
                negative_predictive_value
            ),

        "False_Positive_Rate":
            float(
                false_positive_rate
            ),

        "False_Negative_Rate":
            float(
                false_negative_rate
            ),


        # ------------------------------------------------------------------------------------------
        # PROBABILITY / RANKING METRICS
        # ------------------------------------------------------------------------------------------

        "ROC_AUC":
            float(
                roc_auc
            ),

        "Average_Precision":
            float(
                average_precision
            ),

        "PR_AUC_Trapezoidal":
            float(
                trapezoidal_pr_auc
            ),

        "Brier_Score":
            float(
                brier_score_loss(
                    y_true,
                    scores
                )
            ),


        # ------------------------------------------------------------------------------------------
        # BASELINE CONTEXT
        # ------------------------------------------------------------------------------------------

        "Positive_Prevalence":
            prevalence,

        "Always_Negative_Accuracy":
            always_negative_accuracy,

        "AP_Lift_Over_Prevalence":
            float(
                ap_lift
            ),


        # ------------------------------------------------------------------------------------------
        # EVENT METRICS
        # ------------------------------------------------------------------------------------------

        "True_Attack_Events":
            int(
                len(
                    true_event_ids
                )
            ),

        "Recalled_Attack_Events":
            int(
                len(
                    recalled_event_ids
                )
            ),

        "Warned_Episodes":
            int(
                len(
                    warned_episode_ids
                )
            ),

        "Event_Recall":
            float(
                event_recall
            ),

        "Event_Precision":
            float(
                event_precision
            ),

        "Median_First_Warning_Lead_Min":
            (
                float(
                    np.median(
                        first_warning_leads
                    )
                )

                if len(
                    first_warning_leads
                )

                else np.nan
            ),

        "Mean_First_Warning_Lead_Min":
            (
                float(
                    np.mean(
                        first_warning_leads
                    )
                )

                if len(
                    first_warning_leads
                )

                else np.nan
            ),

        "False_Positive_Warning_Origins":
            int(
                false_positive_warnings.sum()
            ),

        "False_Positive_Warning_Origins_Per_Hour":
            float(
                false_warning_origins_per_hour
            ),
    }


    curves = {

        "precision":
            precision_curve,

        "recall":
            recall_curve,

        "pr_thresholds":
            pr_thresholds,

        "fpr":
            fpr_curve,

        "tpr":
            tpr_curve,

        "roc_thresholds":
            roc_thresholds,
    }


    return (
        metrics,
        curves
    )


# ==================================================================================================
# RUN DIAGNOSTICS
# ==================================================================================================

all_rows = []

all_curves = {}


for H in [
    5,
    10
]:

    print("\n" + "=" * 120)

    print(
        f"H{H} DIAGNOSTICS"
    )

    print("=" * 120)


    horizon_dir = (
        DIAGNOSTIC_ROOT
        / f"H{H:02d}"
    )


    horizon_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    all_curves[
        H
    ] = {}


    for seed in FINAL_SEEDS:

        prediction_file = (

            MODEL_ROOT
            / f"H{H:02d}"
            / "final_replicates"
            / f"seed_{seed}"
            / "validation_predictions.parquet"
        )


        require_file(
            prediction_file
        )


        predictions = pd.read_parquet(
            prediction_file
        )


        required_columns = {

            "Episode_ID",

            "Minute_Index",

            "Minutes_To_Next_Onset",

            "Y_True",

            "Probability",

            "Threshold",

            "Prediction",
        }


        missing = (

            required_columns

            -
            set(
                predictions.columns
            )
        )


        if missing:

            raise RuntimeError(
                f"H{H} seed={seed}: "
                f"missing columns {sorted(missing)}"
            )


        metrics, curves = evaluate_predictions(
            predictions
        )


        metrics[
            "Horizon"
        ] = H


        metrics[
            "Seed"
        ] = seed


        metrics[
            "TEST_Used"
        ] = False


        all_rows.append(
            metrics
        )


        all_curves[
            H
        ][
            seed
        ] = curves


        write_json(

            metrics,

            horizon_dir
            / f"seed_{seed}_full_metrics.json"
        )


        print(
            f"\nH{H} | seed={seed}"
        )


        print(
            f"Accuracy          : {metrics['Accuracy']:.4f}"
        )


        print(
            f"Balanced Accuracy : {metrics['Balanced_Accuracy']:.4f}"
        )


        print(
            f"Precision         : {metrics['Precision']:.4f}"
        )


        print(
            f"Recall            : {metrics['Recall_Sensitivity']:.4f}"
        )


        print(
            f"Specificity       : {metrics['Specificity']:.4f}"
        )


        print(
            f"F1                : {metrics['F1']:.4f}"
        )


        print(
            f"MCC               : {metrics['MCC']:.4f}"
        )


        print(
            f"ROC-AUC           : {metrics['ROC_AUC']:.4f}"
        )


        print(
            f"Average Precision : {metrics['Average_Precision']:.4f}"
        )


        print(
            f"PR-AUC            : {metrics['PR_AUC_Trapezoidal']:.4f}"
        )


        print(
            f"Brier             : {metrics['Brier_Score']:.4f}"
        )


        print(
            f"Prevalence        : {metrics['Positive_Prevalence']:.4f}"
        )


        print(
            f"AP lift           : {metrics['AP_Lift_Over_Prevalence']:.2f}x"
        )


        print(
            f"TP / FP / TN / FN : "
            f"{metrics['TP']} / "
            f"{metrics['FP']} / "
            f"{metrics['TN']} / "
            f"{metrics['FN']}"
        )


        print(
            f"Event Recall      : {metrics['Event_Recall']:.4f}"
        )


        print(
            f"Event Precision   : {metrics['Event_Precision']:.4f}"
        )


        print(
            f"Median lead       : "
            f"{metrics['Median_First_Warning_Lead_Min']:.2f} min"
        )


        print(
            f"False warning origins/hour: "
            f"{metrics['False_Positive_Warning_Origins_Per_Hour']:.4f}"
        )


        del predictions

        gc.collect()


# ==================================================================================================
# DETAILED TABLE
# ==================================================================================================

detail = pd.DataFrame(
    all_rows
)


detail = detail.sort_values(
    [
        "Horizon",
        "Seed"
    ]
).reset_index(
    drop=True
)


detail.to_csv(

    DIAGNOSTIC_ROOT
    / "LSTM_FULL_VALIDATION_METRICS_BY_SEED.csv",

    index=False
)


# ==================================================================================================
# MEAN ± STD SUMMARY
# ==================================================================================================

summary_metrics = [

    "Accuracy",

    "Balanced_Accuracy",

    "Precision",

    "Recall_Sensitivity",

    "Specificity",

    "F1",

    "MCC",

    "ROC_AUC",

    "Average_Precision",

    "PR_AUC_Trapezoidal",

    "Brier_Score",

    "Positive_Prevalence",

    "AP_Lift_Over_Prevalence",

    "Event_Recall",

    "Event_Precision",

    "Median_First_Warning_Lead_Min",

    "Mean_First_Warning_Lead_Min",

    "False_Positive_Warning_Origins_Per_Hour",
]


summary_rows = []


for H in [
    5,
    10
]:

    subset = detail.loc[
        detail[
            "Horizon"
        ]
        ==
        H
    ]


    row = {

        "Horizon":
            H,

        "Replicates":
            len(
                subset
            ),
    }


    for metric in summary_metrics:

        values = subset[
            metric
        ].to_numpy(
            dtype=float
        )


        row[
            f"{metric}_Mean"
        ] = float(
            np.nanmean(
                values
            )
        )


        row[
            f"{metric}_Std"
        ] = float(
            np.nanstd(
                values,
                ddof=1
            )
        )


    # Counts are deterministic enough to show mean for convenience.

    for metric in [

        "TP",

        "FP",

        "TN",

        "FN",

    ]:

        row[
            f"{metric}_Mean"
        ] = float(
            subset[
                metric
            ].mean()
        )


    summary_rows.append(
        row
    )


summary = pd.DataFrame(
    summary_rows
)


summary.to_csv(

    DIAGNOSTIC_ROOT
    / "LSTM_FULL_VALIDATION_SUMMARY.csv",

    index=False
)


# ==================================================================================================
# HUMAN-READABLE SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("LSTM — THREE-SEED VALIDATION SUMMARY")
print("=" * 120)


display_columns = [

    "Horizon",

    "Accuracy_Mean",

    "Balanced_Accuracy_Mean",

    "Precision_Mean",

    "Recall_Sensitivity_Mean",

    "Specificity_Mean",

    "F1_Mean",

    "MCC_Mean",

    "ROC_AUC_Mean",

    "Average_Precision_Mean",

    "PR_AUC_Trapezoidal_Mean",

    "Brier_Score_Mean",

    "AP_Lift_Over_Prevalence_Mean",

    "Event_Recall_Mean",

    "Event_Precision_Mean",

    "Median_First_Warning_Lead_Min_Mean",

    "False_Positive_Warning_Origins_Per_Hour_Mean",
]


print(

    summary[
        display_columns
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# BASELINE CONTEXT
# ==================================================================================================

print("\n" + "=" * 120)
print("BASELINE CONTEXT")
print("=" * 120)


for H in [
    5,
    10
]:

    subset = detail.loc[
        detail[
            "Horizon"
        ]
        ==
        H
    ]


    prevalence = subset[
        "Positive_Prevalence"
    ].mean()


    always_negative_accuracy = subset[
        "Always_Negative_Accuracy"
    ].mean()


    model_accuracy = subset[
        "Accuracy"
    ].mean()


    model_ap = subset[
        "Average_Precision"
    ].mean()


    print(
        f"\nH{H}"
    )


    print(
        f"Positive prevalence            : "
        f"{prevalence:.4f} "
        f"({prevalence*100:.2f}%)"
    )


    print(
        f"Random-ranking AP baseline     : "
        f"{prevalence:.4f}"
    )


    print(
        f"LSTM mean AP                   : "
        f"{model_ap:.4f}"
    )


    print(
        f"LSTM AP lift                   : "
        f"{model_ap/prevalence:.2f}x"
    )


    print(
        f"Always-negative accuracy       : "
        f"{always_negative_accuracy:.4f}"
    )


    print(
        f"LSTM accuracy                  : "
        f"{model_accuracy:.4f}"
    )


# ==================================================================================================
# CONFUSION MATRICES
# ==================================================================================================

for H in [
    5,
    10
]:

    subset = detail.loc[
        detail[
            "Horizon"
        ]
        ==
        H
    ]


    cm = np.array(
        [
            [
                subset[
                    "TN"
                ].mean(),

                subset[
                    "FP"
                ].mean(),
            ],

            [
                subset[
                    "FN"
                ].mean(),

                subset[
                    "TP"
                ].mean(),
            ]
        ]
    )


    fig, ax = plt.subplots(
        figsize=(
            5,
            4
        )
    )


    image = ax.imshow(
        cm
    )


    ax.set_xticks(
        [
            0,
            1
        ]
    )

    ax.set_yticks(
        [
            0,
            1
        ]
    )


    ax.set_xticklabels(
        [
            "Pred Negative",
            "Pred Positive"
        ]
    )


    ax.set_yticklabels(
        [
            "True Negative",
            "True Positive"
        ]
    )


    ax.set_title(
        f"LSTM W10/H{H} — Mean Confusion Matrix"
    )


    for i in range(
        2
    ):

        for j in range(
            2
        ):

            ax.text(
                j,
                i,
                f"{cm[i, j]:.1f}",
                ha="center",
                va="center"
            )


    fig.colorbar(
        image,
        ax=ax
    )


    plt.tight_layout()


    plt.savefig(

        DIAGNOSTIC_ROOT
        / f"H{H:02d}_mean_confusion_matrix.png",

        dpi=160,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ==================================================================================================
# PRECISION-RECALL CURVES
# ==================================================================================================

for H in [
    5,
    10
]:

    plt.figure(
        figsize=(
            7,
            5
        )
    )


    for seed in FINAL_SEEDS:

        curves = all_curves[
            H
        ][
            seed
        ]


        metric_row = detail.loc[
            (
                detail[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                detail[
                    "Seed"
                ]
                ==
                seed
            )
        ].iloc[
            0
        ]


        plt.plot(

            curves[
                "recall"
            ],

            curves[
                "precision"
            ],

            label=(
                f"seed {seed} "
                f"(AP={metric_row['Average_Precision']:.3f})"
            )
        )


    prevalence = detail.loc[
        detail[
            "Horizon"
        ]
        ==
        H,
        "Positive_Prevalence"
    ].mean()


    plt.axhline(

        prevalence,

        linestyle="--",

        label=(
            f"prevalence={prevalence:.3f}"
        )
    )


    plt.xlabel(
        "Recall"
    )

    plt.ylabel(
        "Precision"
    )

    plt.title(
        f"LSTM W10/H{H} — Validation Precision-Recall Curve"
    )

    plt.legend()

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()


    plt.savefig(

        DIAGNOSTIC_ROOT
        / f"H{H:02d}_precision_recall_curve.png",

        dpi=160,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ==================================================================================================
# ROC CURVES
# ==================================================================================================

for H in [
    5,
    10
]:

    plt.figure(
        figsize=(
            7,
            5
        )
    )


    for seed in FINAL_SEEDS:

        curves = all_curves[
            H
        ][
            seed
        ]


        metric_row = detail.loc[
            (
                detail[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                detail[
                    "Seed"
                ]
                ==
                seed
            )
        ].iloc[
            0
        ]


        plt.plot(

            curves[
                "fpr"
            ],

            curves[
                "tpr"
            ],

            label=(
                f"seed {seed} "
                f"(AUC={metric_row['ROC_AUC']:.3f})"
            )
        )


    plt.plot(
        [
            0,
            1
        ],
        [
            0,
            1
        ],
        linestyle="--",
        label="random"
    )


    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.title(
        f"LSTM W10/H{H} — Validation ROC Curve"
    )

    plt.legend()

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()


    plt.savefig(

        DIAGNOSTIC_ROOT
        / f"H{H:02d}_roc_curve.png",

        dpi=160,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ==================================================================================================
# DIAGNOSTIC LOCK
# ==================================================================================================

diagnostic_lock = {

    "Model":
        "LSTM",

    "History_Window":
        10,

    "Horizons":
        [
            5,
            10
        ],

    "Replicate_Seeds":
        FINAL_SEEDS,

    "Evaluation_Split":
        "VALIDATION",

    "Retraining_Performed":
        False,

    "Hyperparameters_Changed":
        False,

    "Thresholds_Changed":
        False,

    "TEST_Loaded":
        False,

    "TEST_Used":
        False,

    "Metrics": [

        "Accuracy",

        "Balanced Accuracy",

        "Precision",

        "Recall/Sensitivity",

        "Specificity",

        "F1",

        "MCC",

        "ROC-AUC",

        "Average Precision",

        "Trapezoidal PR-AUC",

        "Brier Score",

        "TP/FP/TN/FN",

        "Event Recall",

        "Event Precision",

        "First-warning lead time",

        "False-positive warning origins/hour",
    ],

    "Next":
        "VALIDATION_ONLY_LSTM_OPTIMIZATION_BEFORE_TEST",
}


write_json(

    diagnostic_lock,

    DIAGNOSTIC_ROOT
    / "LSTM_VALIDATION_DIAGNOSTIC_LOCK.json"
)


print("\n" + "=" * 120)
print("✓ LSTM FULL VALIDATION DIAGNOSTIC COMPLETE")
print("=" * 120)


print(
    "\nSaved to:"
)

print(
    DIAGNOSTIC_ROOT
)


print(
    "\nNO RETRAINING ✓"
)

print(
    "NO HYPERPARAMETER CHANGES ✓"
)

print(
    "NO TEST DATA ✓"
)

print(
    "\nNEXT:"
)

print(
    "LSTM VALIDATION-ONLY OPTIMIZATION ANALYSIS"
)

print("=" * 120)

In [ ]:
# ==================================================================================================
# LSTM MODEL NOTEBOOK — OPTIMIZATION V1
#
# BOUNDED TRAIN-ONLY INTERNAL OPTIMIZATION
# THEN ONE OFFICIAL VALIDATION CONFIRMATION
#
# ABSOLUTELY NO TEST DATA
# ==================================================================================================
#
# WHY THIS EXISTS
# ---------------
#
# Baseline LSTM diagnostics showed:
#
# H5:
#     AP      ~0.457
#     Recall  ~0.632
#     Precision ~0.463
#
# H10:
#     AP      ~0.581
#     Recall  ~0.508
#     Precision ~0.725
#
# Threshold changes cannot improve AP because AP is threshold-free.
#
# Therefore this bounded optimization tests:
#
#   1. Original full positive weighting
#   2. Reduced sqrt positive weighting
#   3. No positive weighting
#   4. Temporal mean/max/last pooling
#   5. Learnable temporal attention pooling
#   6. Episode-balanced TRAIN sampling
#
#
# SCIENTIFIC SAFETY
# -----------------
#
# 1. Original LSTM baseline is NEVER overwritten.
#
# 2. TEST IS NEVER LOADED.
#
# 3. Optimization candidates are selected using an INTERNAL TRAIN split
#    created at Episode_ID level.
#
# 4. Official VALIDATION is used only after the optimized variant has
#    already been selected.
#
# 5. No change to:
#
#       - benchmark
#       - W10
#       - H5/H10
#       - predictor set
#       - TEST
#
#
# OUTPUT
# ------
#
# /models/LSTM_W10_OPT_V1/
#
# ==================================================================================================


from pathlib import Path
import hashlib
import json
import shutil
import gc
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler
)

from sklearn.model_selection import StratifiedShuffleSplit

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    confusion_matrix,
    auc
)


print("=" * 120)
print("LSTM OPTIMIZATION V1")
print("INTERNAL TRAIN-EPISODE OPTIMIZATION")
print("TEST IS NOT LOADED")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)


P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)


P12_2 = (
    ROOT
    / "model_ready"
    / "12_2"
)


P12_3 = (
    ROOT
    / "model_ready"
    / "12_3"
)


BASELINE_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
)


BASELINE_DIAGNOSTIC = (
    BASELINE_ROOT
    / "validation_diagnostics"
    / "LSTM_FULL_VALIDATION_SUMMARY.csv"
)


OPT_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_OPT_V1"
)


# ==================================================================================================
# PHASE 12.3
# ==================================================================================================

READY_12_3 = (
    P12_3
    / "12_3_READY_FOR_MODELING.json"
)


LOCK_12_3 = (
    P12_3
    / "12_3_PROTOCOL_LOCK.json"
)


LOCK_HASH_12_3 = (
    P12_3
    / "12_3_PROTOCOL_LOCK_SHA256.txt"
)


# ==================================================================================================
# TRAIN FILES
#
# IMPORTANT:
#
# There are intentionally NO TEST file paths in this code.
# ==================================================================================================

X_TRAIN_FILE = (
    P12_2
    / "transformed"
    / "W10"
    / "train"
    / "X_log1p_standardized.npy"
)


Y_TRAIN_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)


ELIGIBLE_TRAIN_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)


TRAIN_INDEX_FILE = (
    P12_0
    / "W10"
    / "train"
    / "index.parquet"
)


# ==================================================================================================
# VALIDATION FILES
# ==================================================================================================

X_VAL_FILE = (
    P12_2
    / "transformed"
    / "W10"
    / "validation"
    / "X_log1p_standardized.npy"
)


Y_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "y_binary.npy"
)


ELIGIBLE_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "eligible.npy"
)


VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

HORIZON_TO_COLUMN = {

    1:
        0,

    5:
        1,

    10:
        2,

    15:
        3,

    30:
        4,
}


HORIZONS = [
    5,
    10
]


INTERNAL_SPLIT_SEED = 20261022


DEVELOPMENT_SEED = 20260930


FINAL_SEEDS = [

    20260930,

    20260931,

    20260932,
]


MAX_EPOCHS = 60


PATIENCE = 8


MIN_DELTA = 0.0001


BATCH_SIZE = 256


GRAD_CLIP = 1.0


AP_VARIANT_TIE_TOLERANCE = 0.002


DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():

        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def read_json(path):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def set_seed(seed):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


# ==================================================================================================
# VERIFY PHASE 12.3
# ==================================================================================================

for path in [

    READY_12_3,

    LOCK_12_3,

    LOCK_HASH_12_3,

    BASELINE_DIAGNOSTIC,

    X_TRAIN_FILE,

    Y_TRAIN_FILE,

    ELIGIBLE_TRAIN_FILE,

    TRAIN_INDEX_FILE,

    X_VAL_FILE,

    Y_VAL_FILE,

    ELIGIBLE_VAL_FILE,

    VAL_INDEX_FILE,

]:

    require_file(
        path
    )


ready = read_json(
    READY_12_3
)


if ready.get(
    "phase_12_3_status"
) != "PASS":

    raise RuntimeError(
        "Phase 12.3 is not PASS."
    )


if ready.get(
    "training_authorized"
) is not True:

    raise RuntimeError(
        "Training is not authorized."
    )


if ready.get(
    "test_evaluation_authorized"
) is not False:

    raise RuntimeError(
        "Unexpected TEST authorization."
    )


protocol_hash = sha256_file(
    LOCK_12_3
)


recorded_protocol_hash = (

    LOCK_HASH_12_3

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if protocol_hash != recorded_protocol_hash:

    raise RuntimeError(
        "Phase 12.3 lock verification failed."
    )


print(
    "✓ Phase 12.3 verified"
)

print(
    "✓ TRAIN/VALIDATION development authorized"
)

print(
    "✓ TEST still embargoed"
)

print(
    "Device:",
    DEVICE
)


if DEVICE.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        )
    )


# ==================================================================================================
# OUTPUT SAFETY
# ==================================================================================================

if OPT_ROOT.exists():

    completion_lock = (
        OPT_ROOT
        / "LSTM_OPTIMIZATION_V1_LOCK.json"
    )


    if completion_lock.is_file():

        print(
            "\n✓ LSTM optimization V1 already completed:"
        )

        print(
            OPT_ROOT
        )

        raise SystemExit


    attempt = 1


    while (

        OPT_ROOT.parent
        / f"_incomplete_LSTM_OPT_V1_attempt_{attempt}"

    ).exists():

        attempt += 1


    backup = (

        OPT_ROOT.parent
        / f"_incomplete_LSTM_OPT_V1_attempt_{attempt}"
    )


    shutil.move(
        str(
            OPT_ROOT
        ),
        str(
            backup
        )
    )


    print(
        "Previous incomplete optimization preserved:"
    )

    print(
        backup
    )


OPT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# OPTIMIZATION PROTOCOL AMENDMENT
#
# This is written BEFORE any optimization training.
# ==================================================================================================

OPTIMIZATION_VARIANTS = [

    {
        "Variant_ID":
            "OPT_A_LAST_FULL",

        "Head":
            "LAST",

        "Loss_Weight":
            "FULL",

        "Episode_Balanced_Sampler":
            False,
    },


    {
        "Variant_ID":
            "OPT_B_LAST_SQRT",

        "Head":
            "LAST",

        "Loss_Weight":
            "SQRT",

        "Episode_Balanced_Sampler":
            False,
    },


    {
        "Variant_ID":
            "OPT_C_LAST_NONE",

        "Head":
            "LAST",

        "Loss_Weight":
            "NONE",

        "Episode_Balanced_Sampler":
            False,
    },


    {
        "Variant_ID":
            "OPT_D_POOL_SQRT",

        "Head":
            "POOL",

        "Loss_Weight":
            "SQRT",

        "Episode_Balanced_Sampler":
            False,
    },


    {
        "Variant_ID":
            "OPT_E_ATTN_SQRT",

        "Head":
            "ATTENTION",

        "Loss_Weight":
            "SQRT",

        "Episode_Balanced_Sampler":
            False,
    },


    {
        "Variant_ID":
            "OPT_F_POOL_EPISODE",

        "Head":
            "POOL",

        "Loss_Weight":
            "NONE",

        "Episode_Balanced_Sampler":
            True,
    },
]


optimization_protocol = {

    "Optimization_Version":
        "LSTM_OPT_V1",

    "Relationship_To_Original":
        "POST_BASELINE_PRE_TEST_OPTIMIZATION_AMENDMENT",

    "Original_LSTM_Overwritten":
        False,

    "History_Window":
        10,

    "Horizons":
        [
            5,
            10
        ],

    "Variants":
        OPTIMIZATION_VARIANTS,

    "Fixed_LSTM_Backbone": {

        "hidden_size":
            64,

        "num_layers":
            2,

        "dropout":
            0.3,

        "learning_rate":
            0.001,

        "weight_decay":
            0.001,
    },

    "Variant_Selection":
        "INTERNAL_TRAIN_EPISODE_DEV_AVERAGE_PRECISION",

    "Internal_Train_Fraction":
        0.80,

    "Internal_Dev_Fraction":
        0.20,

    "Internal_Split_Unit":
        "Episode_ID",

    "Internal_Split_Seed":
        INTERNAL_SPLIT_SEED,

    "Official_Validation_Used_For_Variant_Selection":
        False,

    "Official_Validation_Used_After_Selection":
        True,

    "TEST_Loaded":
        False,

    "TEST_Used":
        False,

    "Selection_Primary_Metric":
        "Average Precision",

    "Tie_Tolerance":
        AP_VARIANT_TIE_TOLERANCE,

    "Final_Replicate_Seeds":
        FINAL_SEEDS,

    "Phase_12_3_Lock_SHA256":
        protocol_hash,
}


write_json(

    optimization_protocol,

    OPT_ROOT
    / "LSTM_OPTIMIZATION_V1_PROTOCOL.json"
)


print(
    "\n✓ Optimization protocol written BEFORE training"
)

print(
    "✓ Six variants frozen"
)


# ==================================================================================================
# LOAD TRAIN DATA
# ==================================================================================================

X_TRAIN_ALL = np.load(

    X_TRAIN_FILE,

    mmap_mode="r"
)


Y_TRAIN_ALL = np.load(

    Y_TRAIN_FILE,

    allow_pickle=False
)


ELIGIBLE_TRAIN_ALL = np.load(

    ELIGIBLE_TRAIN_FILE,

    allow_pickle=False
)


TRAIN_INDEX = pd.read_parquet(
    TRAIN_INDEX_FILE
)


if X_TRAIN_ALL.shape != (
    59837,
    10,
    14
):

    raise RuntimeError(
        f"Unexpected TRAIN shape: {X_TRAIN_ALL.shape}"
    )


if len(
    TRAIN_INDEX
) != X_TRAIN_ALL.shape[
    0
]:

    raise RuntimeError(
        "TRAIN index alignment failure."
    )


TRAIN_EPISODE_IDS = (

    TRAIN_INDEX[
        "Episode_ID"
    ]

    .astype(
        str
    )

    .to_numpy()
)


# ==================================================================================================
# BUILD EPISODE-LEVEL STRATIFICATION TABLE
# ==================================================================================================

episode_columns = [

    "Episode_ID",
]


for candidate in [

    "Episode_Type",

    "Episode_Attack_Family",

    "Calibration_Source",

]:

    if candidate in TRAIN_INDEX.columns:

        episode_columns.append(
            candidate
        )


episode_table = (

    TRAIN_INDEX[
        episode_columns
    ]

    .copy()
)


episode_table[
    "Episode_ID"
] = episode_table[
    "Episode_ID"
].astype(
    str
)


episode_table = (

    episode_table

    .groupby(
        "Episode_ID",
        as_index=False
    )

    .first()
)


for column in [

    "Episode_Type",

    "Episode_Attack_Family",

    "Calibration_Source",

]:

    if column not in episode_table.columns:

        episode_table[
            column
        ] = "UNKNOWN"


    episode_table[
        column
    ] = (

        episode_table[
            column
        ]

        .astype(
            "string"
        )

        .fillna(
            "NONE"
        )

        .astype(
            str
        )
    )


episode_table[
    "Stratum"
] = (

    episode_table[
        "Episode_Type"
    ]

    +

    " | "

    +

    episode_table[
        "Episode_Attack_Family"
    ]

    +

    " | "

    +

    episode_table[
        "Calibration_Source"
    ]
)


stratum_counts = episode_table[
    "Stratum"
].value_counts()


if (
    stratum_counts
    <
    2
).any():

    print(
        "Rare strata detected; falling back to Episode_Type + Family."
    )


    episode_table[
        "Stratum"
    ] = (

        episode_table[
            "Episode_Type"
        ]

        +

        " | "

        +

        episode_table[
            "Episode_Attack_Family"
        ]
    )


# ==================================================================================================
# INTERNAL EPISODE SPLIT
# ==================================================================================================

sss = StratifiedShuffleSplit(

    n_splits=1,

    test_size=0.20,

    random_state=INTERNAL_SPLIT_SEED
)


core_idx, dev_idx = next(

    sss.split(

        episode_table[
            "Episode_ID"
        ],

        episode_table[
            "Stratum"
        ]
    )
)


CORE_EPISODES = set(

    episode_table.iloc[
        core_idx
    ][
        "Episode_ID"
    ]
)


DEV_EPISODES = set(

    episode_table.iloc[
        dev_idx
    ][
        "Episode_ID"
    ]
)


if CORE_EPISODES.intersection(
    DEV_EPISODES
):

    raise RuntimeError(
        "Internal episode leakage detected."
    )


episode_table[
    "Optimization_Split"
] = np.where(

    episode_table[
        "Episode_ID"
    ].isin(
        CORE_EPISODES
    ),

    "CORE_TRAIN",

    "INTERNAL_DEV"
)


episode_table.to_csv(

    OPT_ROOT
    / "INTERNAL_EPISODE_SPLIT.csv",

    index=False
)


print("\n" + "=" * 120)

print(
    "INTERNAL OPTIMIZATION SPLIT"
)

print("=" * 120)


print(
    episode_table[
        "Optimization_Split"
    ].value_counts()
)


print(
    "\nEpisode overlap:",
    len(
        CORE_EPISODES.intersection(
            DEV_EPISODES
        )
    )
)


print(
    "✓ Internal split performed by whole Episode_ID"
)


# ==================================================================================================
# MODEL
# ==================================================================================================

class OptimizedLSTM(
    nn.Module
):

    def __init__(
        self,
        head_type
    ):

        super().__init__()


        hidden_size = 64


        self.head_type = head_type


        self.lstm = nn.LSTM(

            input_size=14,

            hidden_size=hidden_size,

            num_layers=2,

            batch_first=True,

            dropout=0.3
        )


        self.dropout = nn.Dropout(
            0.3
        )


        if head_type == "LAST":

            self.head = nn.Linear(

                hidden_size,

                1
            )


        elif head_type == "POOL":

            pooled_size = (

                hidden_size

                *
                3
            )


            self.layer_norm = nn.LayerNorm(
                pooled_size
            )


            self.head = nn.Sequential(

                nn.Linear(
                    pooled_size,
                    hidden_size
                ),

                nn.GELU(),

                nn.Dropout(
                    0.3
                ),

                nn.Linear(
                    hidden_size,
                    1
                )
            )


        elif head_type == "ATTENTION":

            self.attention = nn.Linear(

                hidden_size,

                1
            )


            combined_size = (

                hidden_size

                *
                2
            )


            self.layer_norm = nn.LayerNorm(
                combined_size
            )


            self.head = nn.Sequential(

                nn.Linear(
                    combined_size,
                    hidden_size
                ),

                nn.GELU(),

                nn.Dropout(
                    0.3
                ),

                nn.Linear(
                    hidden_size,
                    1
                )
            )


        else:

            raise ValueError(
                f"Unknown head type: {head_type}"
            )


    def forward(
        self,
        x
    ):

        sequence_output, (
            hidden,
            _
        ) = self.lstm(
            x
        )


        last_hidden = hidden[
            -1
        ]


        if self.head_type == "LAST":

            features = self.dropout(
                last_hidden
            )


        elif self.head_type == "POOL":

            mean_pool = sequence_output.mean(
                dim=1
            )


            max_pool = sequence_output.max(
                dim=1
            ).values


            features = torch.cat(

                [
                    last_hidden,

                    mean_pool,

                    max_pool,
                ],

                dim=1
            )


            features = self.layer_norm(
                features
            )


        elif self.head_type == "ATTENTION":

            attention_logits = (

                self.attention(
                    sequence_output
                )

                .squeeze(
                    -1
                )
            )


            attention_weights = torch.softmax(

                attention_logits,

                dim=1
            )


            context = torch.sum(

                sequence_output

                *

                attention_weights[
                    :,
                    :,
                    None
                ],

                dim=1
            )


            features = torch.cat(

                [
                    last_hidden,

                    context,
                ],

                dim=1
            )


            features = self.layer_norm(
                features
            )


        logits = self.head(
            features
        ).squeeze(
            1
        )


        return logits


# ==================================================================================================
# EPISODE-BALANCED SAMPLING
# ==================================================================================================

def build_episode_balanced_weights(
    y,
    episodes
):

    y = np.asarray(
        y,
        dtype=np.int8
    )


    episodes = np.asarray(
        episodes
    )


    weights = np.zeros(
        len(
            y
        ),
        dtype=np.float64
    )


    for class_value in [
        0,
        1
    ]:

        class_mask = (
            y
            ==
            class_value
        )


        class_episodes = episodes[
            class_mask
        ]


        counts = (

            pd.Series(
                class_episodes
            )

            .value_counts()
        )


        n_episodes = len(
            counts
        )


        counts_per_sample = (

            pd.Series(
                class_episodes
            )

            .map(
                counts
            )

            .to_numpy(
                dtype=np.float64
            )
        )


        weights[
            class_mask
        ] = (

            0.5

            /

            (
                n_episodes

                *

                counts_per_sample
            )
        )


    if not np.isclose(

        weights.sum(),

        1.0,

        atol=1e-10

    ):

        raise RuntimeError(
            "Episode-balanced sampling weights do not sum to 1."
        )


    return weights


# ==================================================================================================
# LOADERS
# ==================================================================================================

def make_loaders(
    X_train,
    y_train,
    episode_train,
    X_eval,
    y_eval,
    seed,
    episode_sampler
):

    train_dataset = TensorDataset(

        torch.from_numpy(
            X_train
        ),

        torch.from_numpy(
            y_train
        )
    )


    eval_dataset = TensorDataset(

        torch.from_numpy(
            X_eval
        ),

        torch.from_numpy(
            y_eval
        )
    )


    generator = torch.Generator()

    generator.manual_seed(
        seed
    )


    if episode_sampler:

        sampling_weights = build_episode_balanced_weights(

            y_train,

            episode_train
        )


        sampler = WeightedRandomSampler(

            weights=torch.as_tensor(

                sampling_weights,

                dtype=torch.double
            ),

            num_samples=len(
                sampling_weights
            ),

            replacement=True,

            generator=generator
        )


        train_loader = DataLoader(

            train_dataset,

            batch_size=BATCH_SIZE,

            sampler=sampler,

            shuffle=False,

            num_workers=0,

            pin_memory=(
                DEVICE.type
                ==
                "cuda"
            )
        )


    else:

        train_loader = DataLoader(

            train_dataset,

            batch_size=BATCH_SIZE,

            shuffle=True,

            generator=generator,

            num_workers=0,

            pin_memory=(
                DEVICE.type
                ==
                "cuda"
            )
        )


    eval_loader = DataLoader(

        eval_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            ==
            "cuda"
        )
    )


    return (
        train_loader,
        eval_loader
    )


# ==================================================================================================
# PREDICTIONS
# ==================================================================================================

@torch.no_grad()
def predict_scores(
    model,
    loader
):

    model.eval()


    targets = []

    scores = []


    for xb, yb in loader:

        xb = xb.to(
            DEVICE,
            non_blocking=True
        )


        logits = model(
            xb
        )


        probs = torch.sigmoid(
            logits
        )


        targets.append(

            yb

            .cpu()

            .numpy()
        )


        scores.append(

            probs

            .cpu()

            .numpy()
        )


    return (

        np.concatenate(
            targets
        ).astype(
            np.int8
        ),

        np.concatenate(
            scores
        ).astype(
            np.float64
        )
    )


# ==================================================================================================
# VALIDATION THRESHOLD
# ==================================================================================================

def choose_f1_threshold(
    y_true,
    scores
):

    precision, recall, thresholds = precision_recall_curve(

        y_true,

        scores
    )


    if len(
        thresholds
    ) == 0:

        return 0.5


    precision = precision[
        :-1
    ]

    recall = recall[
        :-1
    ]


    denominator = (

        precision

        +
        recall
    )


    f1 = np.divide(

        2.0
        *
        precision
        *
        recall,

        denominator,

        out=np.zeros_like(
            denominator
        ),

        where=(
            denominator
            >
            0
        )
    )


    table = pd.DataFrame(
        {
            "threshold":
                thresholds,

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1,
        }
    )


    best_f1 = table[
        "f1"
    ].max()


    candidates = table.loc[
        np.isclose(

            table[
                "f1"
            ],

            best_f1,

            atol=1e-12,

            rtol=0.0
        )
    ].copy()


    candidates = candidates.sort_values(

        [
            "precision",

            "threshold",
        ],

        ascending=[
            False,
            False,
        ]
    )


    return float(

        candidates.iloc[
            0
        ][
            "threshold"
        ]
    )


# ==================================================================================================
# FULL METRICS
# ==================================================================================================

def calculate_metrics(
    y_true,
    scores,
    threshold
):

    pred = (
        scores
        >=
        threshold
    ).astype(
        np.int8
    )


    tn, fp, fn, tp = confusion_matrix(

        y_true,

        pred,

        labels=[
            0,
            1
        ]

    ).ravel()


    specificity = (

        tn

        /

        (
            tn
            +
            fp
        )
    )


    precision_curve, recall_curve, _ = precision_recall_curve(

        y_true,

        scores
    )


    pr_auc = auc(

        recall_curve,

        precision_curve
    )


    return {

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    pred
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    pred
                )
            ),

        "Precision":
            float(
                precision_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "Recall":
            float(
                recall_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "Specificity":
            float(
                specificity
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y_true,
                    pred
                )
            ),

        "ROC_AUC":
            float(
                roc_auc_score(
                    y_true,
                    scores
                )
            ),

        "Average_Precision":
            float(
                average_precision_score(
                    y_true,
                    scores
                )
            ),

        "PR_AUC":
            float(
                pr_auc
            ),

        "Brier":
            float(
                brier_score_loss(
                    y_true,
                    scores
                )
            ),

        "Threshold":
            float(
                threshold
            ),

        "TP":
            int(
                tp
            ),

        "FP":
            int(
                fp
            ),

        "TN":
            int(
                tn
            ),

        "FN":
            int(
                fn
            ),
    }


# ==================================================================================================
# EVENT METRICS
# ==================================================================================================

def calculate_event_metrics(
    y_true,
    scores,
    threshold,
    index_df
):

    pred = (
        scores
        >=
        threshold
    )


    episodes = (

        index_df[
            "Episode_ID"
        ]

        .astype(
            str
        )

        .to_numpy()
    )


    tto = pd.to_numeric(

        index_df[
            "Minutes_To_Next_Onset"
        ],

        errors="coerce"

    ).to_numpy(
        dtype=float
    )


    positive = (
        y_true
        ==
        1
    )


    negative = (
        y_true
        ==
        0
    )


    tp_warning = (

        pred

        &
        positive
    )


    fp_warning = (

        pred

        &
        negative
    )


    true_events = set(

        episodes[
            positive
        ]
    )


    recalled_events = set(

        episodes[
            tp_warning
        ]
    )


    warned_episodes = set(

        episodes[
            pred
        ]
    )


    event_recall = (

        len(
            recalled_events
        )

        /

        len(
            true_events
        )
    )


    event_precision = (

        len(
            recalled_events
        )

        /

        len(
            warned_episodes
        )

        if len(
            warned_episodes
        )

        else np.nan
    )


    leads = []


    for episode_id in recalled_events:

        mask = (

            tp_warning

            &

            (
                episodes
                ==
                episode_id
            )
        )


        event_tto = tto[
            mask
        ]


        event_tto = event_tto[
            np.isfinite(
                event_tto
            )
        ]


        if len(
            event_tto
        ):

            leads.append(

                float(
                    np.max(
                        event_tto
                    )
                )
            )


    negative_hours = (

        negative.sum()

        /
        60.0
    )


    false_warning_rate = (

        fp_warning.sum()

        /

        negative_hours

        if negative_hours
        >
        0

        else np.nan
    )


    return {

        "Event_Recall":
            float(
                event_recall
            ),

        "Event_Precision":
            float(
                event_precision
            ),

        "Median_Lead_Min":
            (
                float(
                    np.median(
                        leads
                    )
                )

                if len(
                    leads
                )

                else np.nan
            ),

        "Mean_Lead_Min":
            (
                float(
                    np.mean(
                        leads
                    )
                )

                if len(
                    leads
                )

                else np.nan
            ),

        "False_Warning_Origins_Per_Hour":
            float(
                false_warning_rate
            ),
    }


# ==================================================================================================
# TRAINING FUNCTION
# ==================================================================================================

def train_candidate(
    *,
    H,
    variant,
    seed,
    X_train,
    y_train,
    episode_train,
    X_eval,
    y_eval
):

    set_seed(
        seed
    )


    train_loader, eval_loader = make_loaders(

        X_train,

        y_train,

        episode_train,

        X_eval,

        y_eval,

        seed,

        variant[
            "Episode_Balanced_Sampler"
        ]
    )


    model = OptimizedLSTM(

        head_type=variant[
            "Head"
        ]

    ).to(
        DEVICE
    )


    n_positive = float(
        (
            y_train
            ==
            1
        ).sum()
    )


    n_negative = float(
        (
            y_train
            ==
            0
        ).sum()
    )


    raw_pos_weight = (

        n_negative

        /

        n_positive
    )


    weight_mode = variant[
        "Loss_Weight"
    ]


    if weight_mode == "FULL":

        applied_pos_weight = raw_pos_weight


    elif weight_mode == "SQRT":

        applied_pos_weight = np.sqrt(
            raw_pos_weight
        )


    elif weight_mode == "NONE":

        applied_pos_weight = 1.0


    else:

        raise ValueError(
            weight_mode
        )


    criterion = nn.BCEWithLogitsLoss(

        pos_weight=torch.tensor(

            applied_pos_weight,

            dtype=torch.float32,

            device=DEVICE
        )
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=0.001,

        weight_decay=0.001
    )


    best_ap = -np.inf

    best_epoch = None

    best_state = None

    patience_counter = 0


    history = []


    for epoch in range(
        1,
        MAX_EPOCHS
        +
        1
    ):

        model.train()


        total_loss = 0.0

        total_examples = 0


        for xb, yb in train_loader:

            xb = xb.to(
                DEVICE,
                non_blocking=True
            )


            yb = yb.to(
                DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits = model(
                xb
            )


            loss = criterion(
                logits,
                yb
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                GRAD_CLIP
            )


            optimizer.step()


            total_loss += float(
                loss.item()
            ) * len(
                xb
            )


            total_examples += len(
                xb
            )


        train_loss = (

            total_loss

            /
            total_examples
        )


        eval_targets, eval_scores = predict_scores(

            model,

            eval_loader
        )


        eval_ap = float(

            average_precision_score(

                eval_targets,

                eval_scores
            )
        )


        history.append(
            {
                "Epoch":
                    epoch,

                "Train_Loss":
                    train_loss,

                "Eval_AP":
                    eval_ap,
            }
        )


        if (
            eval_ap

            >

            best_ap

            +
            MIN_DELTA
        ):

            best_ap = eval_ap

            best_epoch = epoch

            best_state = {

                key:
                    value
                    .detach()
                    .cpu()
                    .clone()

                for key, value
                in model.state_dict().items()
            }


            patience_counter = 0


        else:

            patience_counter += 1


        if (
            epoch
            ==
            1

            or

            epoch
            %
            5
            ==
            0
        ):

            print(
                f"H{H} | "
                f"{variant['Variant_ID']} | "
                f"epoch={epoch:02d} | "
                f"loss={train_loss:.5f} | "
                f"AP={eval_ap:.6f} | "
                f"best={best_ap:.6f}"
            )


        if patience_counter >= PATIENCE:

            break


    if best_state is None:

        raise RuntimeError(
            "No best model state."
        )


    model.load_state_dict(
        best_state
    )


    model = model.to(
        DEVICE
    )


    eval_targets, eval_scores = predict_scores(

        model,

        eval_loader
    )


    threshold = choose_f1_threshold(

        eval_targets,

        eval_scores
    )


    metrics = calculate_metrics(

        eval_targets,

        eval_scores,

        threshold
    )


    return {

        "model":
            model,

        "state_dict":
            best_state,

        "best_epoch":
            int(
                best_epoch
            ),

        "applied_pos_weight":
            float(
                applied_pos_weight
            ),

        "metrics":
            metrics,

        "scores":
            eval_scores,

        "targets":
            eval_targets,

        "history":
            pd.DataFrame(
                history
            ),

        "parameter_count":
            int(
                sum(

                    p.numel()

                    for p
                    in model.parameters()
                )
            ),
    }


# ==================================================================================================
# GET HORIZON INTERNAL DATA
# ==================================================================================================

def get_internal_horizon_data(
    H
):

    j = HORIZON_TO_COLUMN[
        H
    ]


    eligible = (
        ELIGIBLE_TRAIN_ALL[
            :,
            j
        ]
        ==
        1
    )


    core_episode_mask = np.isin(

        TRAIN_EPISODE_IDS,

        list(
            CORE_EPISODES
        )
    )


    dev_episode_mask = np.isin(

        TRAIN_EPISODE_IDS,

        list(
            DEV_EPISODES
        )
    )


    core_mask = (

        eligible

        &
        core_episode_mask
    )


    dev_mask = (

        eligible

        &
        dev_episode_mask
    )


    if np.any(

        core_mask

        &
        dev_mask
    ):

        raise RuntimeError(
            "Internal sample overlap detected."
        )


    X_core = np.asarray(

        X_TRAIN_ALL[
            core_mask
        ],

        dtype=np.float32
    )


    y_core = np.asarray(

        Y_TRAIN_ALL[
            core_mask,
            j
        ],

        dtype=np.float32
    )


    episode_core = TRAIN_EPISODE_IDS[
        core_mask
    ]


    X_dev = np.asarray(

        X_TRAIN_ALL[
            dev_mask
        ],

        dtype=np.float32
    )


    y_dev = np.asarray(

        Y_TRAIN_ALL[
            dev_mask,
            j
        ],

        dtype=np.float32
    )


    episode_dev = TRAIN_EPISODE_IDS[
        dev_mask
    ]


    return {

        "X_core":
            X_core,

        "y_core":
            y_core,

        "episode_core":
            episode_core,

        "X_dev":
            X_dev,

        "y_dev":
            y_dev,

        "episode_dev":
            episode_dev,
    }


# ==================================================================================================
# STAGE 1
#
# INTERNAL TRAIN-ONLY VARIANT SCREEN
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "STAGE 1 — INTERNAL TRAIN-ONLY LSTM VARIANT SCREEN"
)

print("=" * 120)


screen_rows = []

selected_variants = {}


for H in HORIZONS:

    print("\n" + "#" * 120)

    print(
        f"INTERNAL OPTIMIZATION — H{H}"
    )

    print("#" * 120)


    data = get_internal_horizon_data(
        H
    )


    print(
        "CORE TRAIN:",
        data[
            "X_core"
        ].shape,
        "pos=",
        int(
            data[
                "y_core"
            ].sum()
        ),
        "neg=",
        int(
            (
                data[
                    "y_core"
                ]
                ==
                0
            ).sum()
        )
    )


    print(
        "INTERNAL DEV:",
        data[
            "X_dev"
        ].shape,
        "pos=",
        int(
            data[
                "y_dev"
            ].sum()
        ),
        "neg=",
        int(
            (
                data[
                    "y_dev"
                ]
                ==
                0
            ).sum()
        )
    )


    horizon_screen_dir = (

        OPT_ROOT
        / "internal_screen"
        / f"H{H:02d}"
    )


    horizon_screen_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    horizon_results = []


    for variant in OPTIMIZATION_VARIANTS:

        print("\n" + "-" * 120)

        print(
            H,
            variant
        )

        print("-" * 120)


        result = train_candidate(

            H=H,

            variant=variant,

            seed=DEVELOPMENT_SEED,

            X_train=data[
                "X_core"
            ],

            y_train=data[
                "y_core"
            ],

            episode_train=data[
                "episode_core"
            ],

            X_eval=data[
                "X_dev"
            ],

            y_eval=data[
                "y_dev"
            ]
        )


        record = {

            "Horizon":
                H,

            **variant,

            "Seed":
                DEVELOPMENT_SEED,

            "Best_Epoch":
                result[
                    "best_epoch"
                ],

            "Applied_Pos_Weight":
                result[
                    "applied_pos_weight"
                ],

            "Parameters":
                result[
                    "parameter_count"
                ],

            **result[
                "metrics"
            ],
        }


        horizon_results.append(
            record
        )


        screen_rows.append(
            record
        )


        variant_dir = (

            horizon_screen_dir
            / variant[
                "Variant_ID"
            ]
        )


        variant_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        result[
            "history"
        ].to_csv(

            variant_dir
            / "history.csv",

            index=False
        )


        write_json(

            record,

            variant_dir
            / "internal_dev_metrics.json"
        )


        print(
            f"INTERNAL RESULT | H{H} | "
            f"{variant['Variant_ID']} | "
            f"AP={record['Average_Precision']:.6f} | "
            f"F1={record['F1']:.6f} | "
            f"MCC={record['MCC']:.6f}"
        )


        del result

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    horizon_df = pd.DataFrame(
        horizon_results
    )


    horizon_df.to_csv(

        horizon_screen_dir
        / "variant_screen.csv",

        index=False
    )


    best_ap = horizon_df[
        "Average_Precision"
    ].max()


    candidate_df = horizon_df.loc[

        (
            best_ap

            -

            horizon_df[
                "Average_Precision"
            ]
        )

        <=

        AP_VARIANT_TIE_TOLERANCE

    ].copy()


    head_complexity = {

        "LAST":
            0,

        "ATTENTION":
            1,

        "POOL":
            2,
    }


    candidate_df[
        "Head_Complexity"
    ] = candidate_df[
        "Head"
    ].map(
        head_complexity
    )


    candidate_df = candidate_df.sort_values(

        [
            "F1",

            "Head_Complexity",

            "Variant_ID",
        ],

        ascending=[
            False,
            True,
            True,
        ]
    )


    selected_id = candidate_df.iloc[
        0
    ][
        "Variant_ID"
    ]


    selected_variant = next(

        variant

        for variant
        in OPTIMIZATION_VARIANTS

        if variant[
            "Variant_ID"
        ]
        ==
        selected_id
    )


    selected_variants[
        H
    ] = selected_variant


    print("\n" + "=" * 120)

    print(
        f"H{H} SELECTED INTERNAL VARIANT"
    )

    print(
        selected_variant
    )

    print(
        "Internal dev AP:",
        float(

            horizon_df.loc[

                horizon_df[
                    "Variant_ID"
                ]
                ==
                selected_id,

                "Average_Precision"

            ].iloc[
                0
            ]
        )
    )

    print("=" * 120)


    write_json(
        {
            "Horizon":
                H,

            "Selection_Source":
                "INTERNAL_TRAIN_DEV",

            "Official_Validation_Used":
                False,

            "Selected_Variant":
                selected_variant,
        },

        horizon_screen_dir
        / "selected_variant.json"
    )


    del data

    gc.collect()


screen_df = pd.DataFrame(
    screen_rows
)


screen_df.to_csv(

    OPT_ROOT
    / "INTERNAL_VARIANT_SCREEN_ALL.csv",

    index=False
)


# ==================================================================================================
# RELEASE INTERNAL SCREEN DATA FROM MEMORY
# ==================================================================================================

gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ==================================================================================================
# STAGE 2
#
# FULL TRAIN → OFFICIAL VALIDATION
#
# The optimized architecture is ALREADY selected.
# Official validation did not select the architecture.
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "STAGE 2 — SELECTED OPTIMIZED LSTM → OFFICIAL VALIDATION"
)

print("=" * 120)


X_VAL_ALL = np.load(

    X_VAL_FILE,

    mmap_mode="r"
)


Y_VAL_ALL = np.load(

    Y_VAL_FILE,

    allow_pickle=False
)


ELIGIBLE_VAL_ALL = np.load(

    ELIGIBLE_VAL_FILE,

    allow_pickle=False
)


VAL_INDEX_ALL = pd.read_parquet(
    VAL_INDEX_FILE
)


VAL_EPISODE_IDS = (

    VAL_INDEX_ALL[
        "Episode_ID"
    ]

    .astype(
        str
    )

    .to_numpy()
)


final_rows = []

final_horizon_summaries = []


for H in HORIZONS:

    print("\n" + "#" * 120)

    print(
        f"OPTIMIZED FINAL DEVELOPMENT — H{H}"
    )

    print("#" * 120)


    j = HORIZON_TO_COLUMN[
        H
    ]


    train_mask = (
        ELIGIBLE_TRAIN_ALL[
            :,
            j
        ]
        ==
        1
    )


    val_mask = (
        ELIGIBLE_VAL_ALL[
            :,
            j
        ]
        ==
        1
    )


    X_train = np.asarray(

        X_TRAIN_ALL[
            train_mask
        ],

        dtype=np.float32
    )


    y_train = np.asarray(

        Y_TRAIN_ALL[
            train_mask,
            j
        ],

        dtype=np.float32
    )


    episode_train = TRAIN_EPISODE_IDS[
        train_mask
    ]


    X_val = np.asarray(

        X_VAL_ALL[
            val_mask
        ],

        dtype=np.float32
    )


    y_val = np.asarray(

        Y_VAL_ALL[
            val_mask,
            j
        ],

        dtype=np.float32
    )


    val_index = (

        VAL_INDEX_ALL

        .loc[
            val_mask
        ]

        .reset_index(
            drop=True
        )
    )


    selected_variant = selected_variants[
        H
    ]


    print(
        "Selected variant:",
        selected_variant
    )


    print(
        "TRAIN:",
        X_train.shape
    )


    print(
        "VALIDATION:",
        X_val.shape
    )


    horizon_dir = (

        OPT_ROOT
        / f"H{H:02d}"
        / "final_replicates"
    )


    horizon_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    horizon_rows = []


    for replicate_number, seed in enumerate(

        FINAL_SEEDS,

        start=1
    ):

        print("\n" + "-" * 120)

        print(
            f"H{H} OPTIMIZED REPLICATE "
            f"{replicate_number}/3 "
            f"seed={seed}"
        )

        print("-" * 120)


        result = train_candidate(

            H=H,

            variant=selected_variant,

            seed=seed,

            X_train=X_train,

            y_train=y_train,

            episode_train=episode_train,

            X_eval=X_val,

            y_eval=y_val
        )


        metrics = result[
            "metrics"
        ]


        event_metrics = calculate_event_metrics(

            result[
                "targets"
            ],

            result[
                "scores"
            ],

            metrics[
                "Threshold"
            ],

            val_index
        )


        record = {

            "Model":
                "LSTM_OPT_V1",

            "Horizon":
                H,

            "Replicate":
                replicate_number,

            "Seed":
                seed,

            **selected_variant,

            "Best_Epoch":
                result[
                    "best_epoch"
                ],

            "Applied_Pos_Weight":
                result[
                    "applied_pos_weight"
                ],

            "Parameters":
                result[
                    "parameter_count"
                ],

            **metrics,

            **event_metrics,

            "TEST_Used":
                False,
        }


        horizon_rows.append(
            record
        )


        final_rows.append(
            record
        )


        replicate_dir = (

            horizon_dir
            / f"seed_{seed}"
        )


        replicate_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        torch.save(
            {
                "model_type":
                    "LSTM_OPT_V1",

                "horizon":
                    H,

                "variant":
                    selected_variant,

                "seed":
                    seed,

                "state_dict":
                    result[
                        "state_dict"
                    ],

                "validation_threshold":
                    metrics[
                        "Threshold"
                    ],

                "phase_12_3_lock_sha256":
                    protocol_hash,

                "TEST_Used":
                    False,
            },

            replicate_dir
            / "model.pt"
        )


        result[
            "history"
        ].to_csv(

            replicate_dir
            / "training_history.csv",

            index=False
        )


        prediction_df = val_index[
            [
                "Episode_ID",

                "Minute_Index",

                "Minutes_To_Next_Onset",
            ]
        ].copy()


        prediction_df[
            "Y_True"
        ] = result[
            "targets"
        ]


        prediction_df[
            "Probability"
        ] = result[
            "scores"
        ]


        prediction_df[
            "Threshold"
        ] = metrics[
            "Threshold"
        ]


        prediction_df[
            "Prediction"
        ] = (

            result[
                "scores"
            ]

            >=

            metrics[
                "Threshold"
            ]
        ).astype(
            np.int8
        )


        prediction_df.to_parquet(

            replicate_dir
            / "validation_predictions.parquet",

            index=False
        )


        write_json(

            record,

            replicate_dir
            / "validation_metrics.json"
        )


        print(
            f"AP={metrics['Average_Precision']:.6f}"
        )


        print(
            f"PR-AUC={metrics['PR_AUC']:.6f}"
        )


        print(
            f"ROC-AUC={metrics['ROC_AUC']:.6f}"
        )


        print(
            f"Precision={metrics['Precision']:.6f}"
        )


        print(
            f"Recall={metrics['Recall']:.6f}"
        )


        print(
            f"F1={metrics['F1']:.6f}"
        )


        print(
            f"MCC={metrics['MCC']:.6f}"
        )


        print(
            f"Event Recall={event_metrics['Event_Recall']:.6f}"
        )


        print(
            f"Event Precision={event_metrics['Event_Precision']:.6f}"
        )


        print(
            f"Median Lead={event_metrics['Median_Lead_Min']}"
        )


        print(
            f"False warning origins/hour="
            f"{event_metrics['False_Warning_Origins_Per_Hour']:.6f}"
        )


        del result

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    horizon_df = pd.DataFrame(
        horizon_rows
    )


    horizon_df.to_csv(

        OPT_ROOT
        / f"H{H:02d}"
        / "optimized_validation_replicates.csv",

        index=False
    )


    summary = {

        "Horizon":
            H,

        "Selected_Variant":
            selected_variant[
                "Variant_ID"
            ],

        "AP_Mean":
            float(
                horizon_df[
                    "Average_Precision"
                ].mean()
            ),

        "AP_Std":
            float(
                horizon_df[
                    "Average_Precision"
                ].std(
                    ddof=1
                )
            ),

        "PR_AUC_Mean":
            float(
                horizon_df[
                    "PR_AUC"
                ].mean()
            ),

        "ROC_AUC_Mean":
            float(
                horizon_df[
                    "ROC_AUC"
                ].mean()
            ),

        "Accuracy_Mean":
            float(
                horizon_df[
                    "Accuracy"
                ].mean()
            ),

        "Balanced_Accuracy_Mean":
            float(
                horizon_df[
                    "Balanced_Accuracy"
                ].mean()
            ),

        "Precision_Mean":
            float(
                horizon_df[
                    "Precision"
                ].mean()
            ),

        "Recall_Mean":
            float(
                horizon_df[
                    "Recall"
                ].mean()
            ),

        "F1_Mean":
            float(
                horizon_df[
                    "F1"
                ].mean()
            ),

        "MCC_Mean":
            float(
                horizon_df[
                    "MCC"
                ].mean()
            ),

        "Brier_Mean":
            float(
                horizon_df[
                    "Brier"
                ].mean()
            ),

        "Event_Recall_Mean":
            float(
                horizon_df[
                    "Event_Recall"
                ].mean()
            ),

        "Event_Precision_Mean":
            float(
                horizon_df[
                    "Event_Precision"
                ].mean()
            ),

        "Median_Lead_Mean":
            float(
                horizon_df[
                    "Median_Lead_Min"
                ].mean()
            ),

        "False_Warning_Origins_Per_Hour_Mean":
            float(
                horizon_df[
                    "False_Warning_Origins_Per_Hour"
                ].mean()
            ),

        "TEST_Used":
            False,
    }


    final_horizon_summaries.append(
        summary
    )


    del X_train

    del y_train

    del episode_train

    del X_val

    del y_val

    del val_index

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ==================================================================================================
# SAVE OPTIMIZED SUMMARY
# ==================================================================================================

final_df = pd.DataFrame(
    final_rows
)


final_df.to_csv(

    OPT_ROOT
    / "LSTM_OPT_V1_ALL_FINAL_REPLICATES.csv",

    index=False
)


optimized_summary = pd.DataFrame(
    final_horizon_summaries
)


optimized_summary.to_csv(

    OPT_ROOT
    / "LSTM_OPT_V1_VALIDATION_SUMMARY.csv",

    index=False
)


# ==================================================================================================
# COMPARE AGAINST ORIGINAL BASELINE
# ==================================================================================================

baseline = pd.read_csv(
    BASELINE_DIAGNOSTIC
)


comparison_rows = []


for H in HORIZONS:

    base_row = baseline.loc[
        baseline[
            "Horizon"
        ]
        ==
        H
    ].iloc[
        0
    ]


    opt_row = optimized_summary.loc[
        optimized_summary[
            "Horizon"
        ]
        ==
        H
    ].iloc[
        0
    ]


    baseline_ap = float(
        base_row[
            "Average_Precision_Mean"
        ]
    )


    optimized_ap = float(
        opt_row[
            "AP_Mean"
        ]
    )


    delta_ap = (

        optimized_ap

        -
        baseline_ap
    )


    baseline_f1 = float(
        base_row[
            "F1_Mean"
        ]
    )


    optimized_f1 = float(
        opt_row[
            "F1_Mean"
        ]
    )


    baseline_mcc = float(
        base_row[
            "MCC_Mean"
        ]
    )


    optimized_mcc = float(
        opt_row[
            "MCC_Mean"
        ]
    )


    # Primary model selection metric remains AP.
    #
    # Require at least +0.002 absolute AP to replace baseline.
    # Otherwise preserve simpler already-frozen baseline.

    if delta_ap > 0.002:

        recommendation = "OPTIMIZED_LSTM"

    else:

        recommendation = "ORIGINAL_BASELINE_LSTM"


    comparison_rows.append(
        {
            "Horizon":
                H,

            "Optimized_Variant":
                opt_row[
                    "Selected_Variant"
                ],

            "Baseline_AP":
                baseline_ap,

            "Optimized_AP":
                optimized_ap,

            "AP_Absolute_Change":
                delta_ap,

            "AP_Relative_Change_Percent":
                (
                    100.0

                    *
                    delta_ap

                    /
                    baseline_ap
                ),

            "Baseline_F1":
                baseline_f1,

            "Optimized_F1":
                optimized_f1,

            "F1_Change":
                optimized_f1
                -
                baseline_f1,

            "Baseline_MCC":
                baseline_mcc,

            "Optimized_MCC":
                optimized_mcc,

            "MCC_Change":
                optimized_mcc
                -
                baseline_mcc,

            "Recommended_Candidate":
                recommendation,
        }
    )


comparison = pd.DataFrame(
    comparison_rows
)


comparison.to_csv(

    OPT_ROOT
    / "BASELINE_VS_OPTIMIZED_LSTM.csv",

    index=False
)


# ==================================================================================================
# PRINT RESULTS
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "OPTIMIZED LSTM VALIDATION SUMMARY"
)

print("=" * 120)


print(
    optimized_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 120)

print(
    "ORIGINAL VS OPTIMIZED"
)

print("=" * 120)


print(
    comparison.to_string(
        index=False
    )
)


# ==================================================================================================
# FINAL OPTIMIZATION LOCK
# ==================================================================================================

write_json(
    {
        "Model":
            "LSTM",

        "Optimization_Version":
            "V1",

        "Status":
            "OPTIMIZATION_COMPLETE_PRE_TEST",

        "Original_Baseline_Preserved":
            True,

        "Internal_Optimization_Split":
            "Episode_ID",

        "Official_Validation_Used_For_Variant_Selection":
            False,

        "Official_Validation_Used_For_Final_Confirmation":
            True,

        "Horizons":
            [
                5,
                10
            ],

        "Selected_Variants":
            {

                str(
                    H
                ):
                    selected_variants[
                        H
                    ]

                for H in HORIZONS
            },

        "Final_Seeds":
            FINAL_SEEDS,

        "Primary_Comparison_Metric":
            "Average Precision",

        "Minimum_AP_Gain_To_Replace_Baseline":
            0.002,

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "TEST_Evaluation_Performed":
            False,

        "Phase_12_3_Protocol_Lock_SHA256":
            protocol_hash,
    },

    OPT_ROOT
    / "LSTM_OPTIMIZATION_V1_LOCK.json"
)


print("\n" + "=" * 120)

print(
    "✓ LSTM OPTIMIZATION V1 COMPLETE"
)

print("=" * 120)


print(
    "\nOriginal LSTM preserved ✓"
)

print(
    "Internal optimization performed by Episode_ID ✓"
)

print(
    "Official validation did NOT select optimization variant ✓"
)

print(
    "Three frozen seeds used for final validation ✓"
)

print(
    "TEST loaded: NO ✓"
)

print(
    "TEST evaluated: NO ✓"
)


print(
    "\nSaved to:"
)

print(
    OPT_ROOT
)


print(
    "\nDO NOT RUN TEST YET."
)

print("=" * 120)


gc.collect()

In [ ]:
# ==================================================================================================
# LSTM MODEL NOTEBOOK — CELL 5
#
# FAILURE-SLICE / FORECASTABILITY ANALYSIS
#
# NO TRAINING
# NO HYPERPARAMETER TUNING
# NO THRESHOLD RETUNING
# NO TEST DATA
#
# QUESTIONS ANSWERED
# ------------------
#
# 1. Which precursor regimes are easiest / hardest?
#
#       NO_PRECURSOR
#       SHORT_WEAK
#       SHORT_STRONG
#       GRADUAL_WEAK
#       GRADUAL_STRONG
#       BURSTY
#
# 2. Which attack families are easiest / hardest?
#
# 3. Is performance substantially different for CIC-derived vs UNSW-derived
#    synthetic episodes?
#
# 4. How does probability / recall change as true onset approaches?
#
# 5. Are false warnings concentrated in benign-only episodes?
#
#
# IMPORTANT SLICE SEMANTICS
# -------------------------
#
# PRECURSOR REGIME:
#
#     evaluate positive + negative origins WITHIN attack episodes
#     belonging to that regime.
#
# FAMILY:
#
#     evaluate positive + negative origins WITHIN attack episodes
#     belonging to that family.
#
# SOURCE:
#
#     evaluate all eligible origins belonging to that calibration source.
#
#
# The model's ORIGINAL validation threshold is used everywhere.
# We DO NOT optimize thresholds independently by slice.
#
# ==================================================================================================


from pathlib import Path
import json
import shutil
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    confusion_matrix,
    auc
)


print("=" * 120)
print("CELL 5 — LSTM FAILURE-SLICE ANALYSIS")
print("VALIDATION ONLY")
print("NO TRAINING / NO TEST")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

MODEL_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
)

DEVELOPMENT_LOCK_FILE = (
    MODEL_ROOT
    / "LSTM_DEVELOPMENT_LOCK.json"
)

VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)

OUTPUT_ROOT = (
    MODEL_ROOT
    / "validation_diagnostics"
    / "failure_slices"
)

COMPLETE_FILE = (
    OUTPUT_ROOT
    / "CELL5_FAILURE_SLICE_COMPLETE.json"
)


FINAL_SEEDS = [
    20260930,
    20260931,
    20260932,
]

HORIZONS = [
    5,
    10,
]


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():

        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )


def read_json(path):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def normalize_text(series):

    return (

        series

        .astype(
            "string"
        )

        .fillna(
            "NONE"
        )

        .astype(
            str
        )

        .str.strip()
    )


def normalize_source_value(value):

    text = str(
        value
    ).upper()


    if "CIC" in text:

        return "CIC"


    if "UNSW" in text:

        return "UNSW"


    return text


def safe_auc(
    y_true,
    scores
):

    if len(
        np.unique(
            y_true
        )
    ) < 2:

        return np.nan


    return float(
        roc_auc_score(
            y_true,
            scores
        )
    )


def safe_ap(
    y_true,
    scores
):

    if (
        y_true
        ==
        1
    ).sum() == 0:

        return np.nan


    return float(
        average_precision_score(
            y_true,
            scores
        )
    )


def safe_pr_auc(
    y_true,
    scores
):

    if (
        y_true
        ==
        1
    ).sum() == 0:

        return np.nan


    precision_curve, recall_curve, _ = precision_recall_curve(

        y_true,

        scores
    )


    return float(
        auc(
            recall_curve,
            precision_curve
        )
    )


# ==================================================================================================
# VERIFY ORIGINAL LSTM
# ==================================================================================================

require_file(
    DEVELOPMENT_LOCK_FILE
)

require_file(
    VAL_INDEX_FILE
)


development_lock = read_json(
    DEVELOPMENT_LOCK_FILE
)


if development_lock.get(
    "Status"
) != "TRAIN_VALIDATION_DEVELOPMENT_COMPLETE":

    raise RuntimeError(
        "Original LSTM development is not complete."
    )


if development_lock.get(
    "TEST_Loaded"
) is not False:

    raise RuntimeError(
        "Development lock says TEST was loaded."
    )


if development_lock.get(
    "TEST_Used"
) is not False:

    raise RuntimeError(
        "Development lock says TEST was used."
    )


if development_lock.get(
    "TEST_Evaluation_Performed"
) is not False:

    raise RuntimeError(
        "Development lock says TEST was evaluated."
    )


if development_lock.get(
    "Final_Replicate_Seeds"
) != FINAL_SEEDS:

    raise RuntimeError(
        "Frozen final replicate seeds changed."
    )


print(
    "✓ Original LSTM development verified"
)

print(
    "✓ TEST was never loaded"
)

print(
    "✓ TEST was never evaluated"
)


# ==================================================================================================
# OUTPUT SAFETY
# ==================================================================================================

if COMPLETE_FILE.is_file():

    print(
        "\n✓ Cell 5 failure-slice analysis already completed."
    )

    print(
        OUTPUT_ROOT
    )

    raise SystemExit


if OUTPUT_ROOT.exists():

    attempt = 1


    while (

        OUTPUT_ROOT.parent
        / f"_incomplete_failure_slices_attempt_{attempt}"

    ).exists():

        attempt += 1


    backup = (

        OUTPUT_ROOT.parent
        / f"_incomplete_failure_slices_attempt_{attempt}"
    )


    shutil.move(
        str(
            OUTPUT_ROOT
        ),
        str(
            backup
        )
    )


    print(
        "Previous incomplete failure analysis preserved:"
    )

    print(
        backup
    )


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# LOAD AUTHORITATIVE VALIDATION INDEX
# ==================================================================================================

val_index_all = pd.read_parquet(
    VAL_INDEX_FILE
)


required_metadata_columns = [

    "Episode_ID",

    "Minute_Index",

    "Episode_Attack_Family",

    "Calibration_Source",

    "Precursor_Regime",

    "Episode_Type",

    "Minutes_To_Next_Onset",
]


missing = [

    column

    for column
    in required_metadata_columns

    if column not in val_index_all.columns
]


if missing:

    raise RuntimeError(
        "Validation index is missing metadata columns:\n"
        +
        str(
            missing
        )
    )


val_index_all = val_index_all.copy()


val_index_all[
    "Episode_ID"
] = val_index_all[
    "Episode_ID"
].astype(
    str
)


val_index_all[
    "Minute_Index"
] = pd.to_numeric(

    val_index_all[
        "Minute_Index"
    ],

    errors="raise"

).astype(
    np.int64
)


val_index_all[
    "Episode_Attack_Family"
] = normalize_text(

    val_index_all[
        "Episode_Attack_Family"
    ]
)


val_index_all[
    "Precursor_Regime"
] = normalize_text(

    val_index_all[
        "Precursor_Regime"
    ]
)


val_index_all[
    "Episode_Type"
] = normalize_text(

    val_index_all[
        "Episode_Type"
    ]
)


val_index_all[
    "Calibration_Source_Normalized"
] = (

    val_index_all[
        "Calibration_Source"
    ]

    .map(
        normalize_source_value
    )
)


if val_index_all.duplicated(
    subset=[
        "Episode_ID",
        "Minute_Index"
    ]
).any():

    raise RuntimeError(
        "Validation index contains duplicate Episode_ID/Minute_Index."
    )


print("\nMETADATA VALUES FOUND")
print("-" * 120)


print(
    "Episode types:",
    sorted(
        val_index_all[
            "Episode_Type"
        ].unique()
    )
)


print(
    "Families:",
    sorted(
        val_index_all[
            "Episode_Attack_Family"
        ].unique()
    )
)


print(
    "Precursor regimes:",
    sorted(
        val_index_all[
            "Precursor_Regime"
        ].unique()
    )
)


print(
    "Calibration sources:",
    sorted(
        val_index_all[
            "Calibration_Source_Normalized"
        ].unique()
    )
)


# ==================================================================================================
# METRIC CALCULATOR
# ==================================================================================================

def compute_slice_metrics(
    df
):

    y_true = df[
        "Y_True"
    ].to_numpy(
        dtype=np.int8
    )


    scores = df[
        "Probability"
    ].to_numpy(
        dtype=np.float64
    )


    pred = df[
        "Prediction"
    ].to_numpy(
        dtype=np.int8
    )


    n = len(
        df
    )


    positives = int(
        (
            y_true
            ==
            1
        ).sum()
    )


    negatives = int(
        (
            y_true
            ==
            0
        ).sum()
    )


    prevalence = (

        positives
        /
        n

        if n

        else np.nan
    )


    # ----------------------------------------------------------------------------------------------
    # One-class slices
    # ----------------------------------------------------------------------------------------------

    if positives == 0:

        accuracy = float(
            accuracy_score(
                y_true,
                pred
            )
        )


        specificity = float(
            (
                pred
                ==
                0
            ).mean()
        )


        return {

            "Samples":
                n,

            "Positives":
                positives,

            "Negatives":
                negatives,

            "Prevalence":
                prevalence,

            "Accuracy":
                accuracy,

            "Balanced_Accuracy":
                np.nan,

            "Precision":
                np.nan,

            "Recall":
                np.nan,

            "Specificity":
                specificity,

            "F1":
                np.nan,

            "MCC":
                np.nan,

            "ROC_AUC":
                np.nan,

            "Average_Precision":
                np.nan,

            "PR_AUC":
                np.nan,

            "AP_Lift":
                np.nan,

            "Brier":
                float(
                    brier_score_loss(
                        y_true,
                        scores
                    )
                ),

            "TP":
                0,

            "FP":
                int(
                    (
                        pred
                        ==
                        1
                    ).sum()
                ),

            "TN":
                int(
                    (
                        pred
                        ==
                        0
                    ).sum()
                ),

            "FN":
                0,

            "Predicted_Positive_Rate":
                float(
                    pred.mean()
                ),

            "Mean_Probability":
                float(
                    scores.mean()
                ),

            "Median_Probability":
                float(
                    np.median(
                        scores
                    )
                ),

            "P95_Probability":
                float(
                    np.percentile(
                        scores,
                        95
                    )
                ),
        }


    # ----------------------------------------------------------------------------------------------
    # Standard two-class slices
    # ----------------------------------------------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(

        y_true,

        pred,

        labels=[
            0,
            1
        ]

    ).ravel()


    specificity = (

        tn

        /

        (
            tn
            +
            fp
        )

        if (
            tn
            +
            fp
        )
        >
        0

        else np.nan
    )


    ap = safe_ap(
        y_true,
        scores
    )


    return {

        "Samples":
            n,

        "Positives":
            positives,

        "Negatives":
            negatives,

        "Prevalence":
            prevalence,

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    pred
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    pred
                )
            ),

        "Precision":
            float(
                precision_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "Recall":
            float(
                recall_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "Specificity":
            float(
                specificity
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    pred,
                    zero_division=0
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y_true,
                    pred
                )
            ),

        "ROC_AUC":
            safe_auc(
                y_true,
                scores
            ),

        "Average_Precision":
            ap,

        "PR_AUC":
            safe_pr_auc(
                y_true,
                scores
            ),

        "AP_Lift":
            (
                ap
                /
                prevalence

                if (
                    prevalence
                    >
                    0
                )

                else np.nan
            ),

        "Brier":
            float(
                brier_score_loss(
                    y_true,
                    scores
                )
            ),

        "TP":
            int(
                tp
            ),

        "FP":
            int(
                fp
            ),

        "TN":
            int(
                tn
            ),

        "FN":
            int(
                fn
            ),

        "Predicted_Positive_Rate":
            float(
                pred.mean()
            ),

        "Mean_Probability":
            float(
                scores.mean()
            ),

        "Median_Probability":
            float(
                np.median(
                    scores
                )
            ),

        "P95_Probability":
            float(
                np.percentile(
                    scores,
                    95
                )
            ),
    }


# ==================================================================================================
# EVENT METRICS FOR A SLICE
# ==================================================================================================

def compute_event_metrics(
    df
):

    y_true = df[
        "Y_True"
    ].to_numpy(
        dtype=np.int8
    )


    pred = df[
        "Prediction"
    ].to_numpy(
        dtype=np.int8
    )


    episodes = df[
        "Episode_ID"
    ].astype(
        str
    ).to_numpy()


    tto = pd.to_numeric(

        df[
            "Minutes_To_Next_Onset"
        ],

        errors="coerce"

    ).to_numpy(
        dtype=float
    )


    positive = (
        y_true
        ==
        1
    )


    warned = (
        pred
        ==
        1
    )


    tp_warning = (

        positive

        &
        warned
    )


    true_events = set(

        episodes[
            positive
        ]
    )


    recalled_events = set(

        episodes[
            tp_warning
        ]
    )


    warned_episodes = set(

        episodes[
            warned
        ]
    )


    if len(
        true_events
    ) == 0:

        return {

            "Unique_True_Events":
                0,

            "Recalled_Events":
                0,

            "Event_Recall":
                np.nan,

            "Event_Precision":
                np.nan,

            "Median_First_Warning_Lead_Min":
                np.nan,

            "Mean_First_Warning_Lead_Min":
                np.nan,
        }


    event_recall = (

        len(
            recalled_events
        )

        /

        len(
            true_events
        )
    )


    event_precision = (

        len(
            recalled_events
        )

        /

        len(
            warned_episodes
        )

        if len(
            warned_episodes
        )

        else np.nan
    )


    leads = []


    for episode_id in recalled_events:

        event_mask = (

            tp_warning

            &

            (
                episodes
                ==
                episode_id
            )
        )


        event_tto = tto[
            event_mask
        ]


        event_tto = event_tto[
            np.isfinite(
                event_tto
            )
        ]


        if len(
            event_tto
        ):

            leads.append(

                float(
                    np.max(
                        event_tto
                    )
                )
            )


    return {

        "Unique_True_Events":
            int(
                len(
                    true_events
                )
            ),

        "Recalled_Events":
            int(
                len(
                    recalled_events
                )
            ),

        "Event_Recall":
            float(
                event_recall
            ),

        "Event_Precision":
            float(
                event_precision
            ),

        "Median_First_Warning_Lead_Min":
            (
                float(
                    np.median(
                        leads
                    )
                )

                if len(
                    leads
                )

                else np.nan
            ),

        "Mean_First_Warning_Lead_Min":
            (
                float(
                    np.mean(
                        leads
                    )
                )

                if len(
                    leads
                )

                else np.nan
            ),
    }


# ==================================================================================================
# LOAD EACH SEED/HORIZON AND JOIN AUTHORITATIVE METADATA
# ==================================================================================================

all_predictions = []


for H in HORIZONS:

    for seed in FINAL_SEEDS:

        prediction_file = (

            MODEL_ROOT
            / f"H{H:02d}"
            / "final_replicates"
            / f"seed_{seed}"
            / "validation_predictions.parquet"
        )


        require_file(
            prediction_file
        )


        pred = pd.read_parquet(
            prediction_file
        )


        pred[
            "Episode_ID"
        ] = pred[
            "Episode_ID"
        ].astype(
            str
        )


        pred[
            "Minute_Index"
        ] = pd.to_numeric(

            pred[
                "Minute_Index"
            ],

            errors="raise"

        ).astype(
            np.int64
        )


        if pred.duplicated(
            subset=[
                "Episode_ID",
                "Minute_Index"
            ]
        ).any():

            raise RuntimeError(
                f"H{H} seed={seed}: duplicate predictions."
            )


        # ------------------------------------------------------------------------------------------
        # Avoid using duplicate TTO columns during merge.
        # Authoritative metadata comes from the Phase 12.0 index.
        # ------------------------------------------------------------------------------------------

        pred_for_merge = pred.drop(
            columns=[
                "Minutes_To_Next_Onset"
            ],
            errors="ignore"
        )


        joined = pred_for_merge.merge(

            val_index_all[
                required_metadata_columns
                +
                [
                    "Calibration_Source_Normalized"
                ]
            ],

            on=[
                "Episode_ID",
                "Minute_Index"
            ],

            how="left",

            validate="one_to_one"
        )


        if joined[
            "Calibration_Source"
        ].isna().any():

            raise RuntimeError(
                f"H{H} seed={seed}: metadata merge failed."
            )


        joined[
            "Horizon"
        ] = H


        joined[
            "Seed"
        ] = seed


        all_predictions.append(
            joined
        )


        print(
            f"✓ joined H{H} seed={seed}: "
            f"{len(joined):,} validation origins"
        )


predictions = pd.concat(

    all_predictions,

    ignore_index=True
)


del all_predictions

gc.collect()


# ==================================================================================================
# VERIFY FROZEN THRESHOLDS
# ==================================================================================================

threshold_table = (

    predictions

    .groupby(
        [
            "Horizon",
            "Seed"
        ]
    )[
        "Threshold"
    ]

    .nunique()

    .reset_index(
        name="Unique_Threshold_Count"
    )
)


if not (

    threshold_table[
        "Unique_Threshold_Count"
    ]
    ==
    1

).all():

    raise RuntimeError(
        "A replicate contains multiple operating thresholds."
    )


print(
    "\n✓ original validation thresholds preserved"
)

print(
    "✓ no slice-specific threshold tuning"
)


# ==================================================================================================
# PRECURSOR REGIME ANALYSIS
#
# IMPORTANT:
#
# We evaluate within episodes carrying the regime.
#
# Therefore the model must distinguish earlier negative origins from
# positive near-onset origins within attack episodes of that regime.
# ==================================================================================================

print("\n" + "=" * 120)
print("PRECURSOR REGIME FAILURE ANALYSIS")
print("=" * 120)


regime_rows = []


REGIME_ORDER = [

    "NO_PRECURSOR",

    "SHORT_WEAK",

    "SHORT_STRONG",

    "GRADUAL_WEAK",

    "GRADUAL_STRONG",

    "BURSTY",
]


for H in HORIZONS:

    for seed in FINAL_SEEDS:

        base = predictions.loc[
            (
                predictions[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                predictions[
                    "Seed"
                ]
                ==
                seed
            )
        ].copy()


        for regime in REGIME_ORDER:

            subset = base.loc[
                base[
                    "Precursor_Regime"
                ]
                ==
                regime
            ].copy()


            if len(
                subset
            ) == 0:

                continue


            metrics = compute_slice_metrics(
                subset
            )


            event_metrics = compute_event_metrics(
                subset
            )


            regime_rows.append(
                {
                    "Horizon":
                        H,

                    "Seed":
                        seed,

                    "Precursor_Regime":
                        regime,

                    **metrics,

                    **event_metrics,
                }
            )


regime_detail = pd.DataFrame(
    regime_rows
)


regime_detail.to_csv(

    OUTPUT_ROOT
    / "PRECURSOR_REGIME_BY_SEED.csv",

    index=False
)


# ==================================================================================================
# FAMILY ANALYSIS
# ==================================================================================================

print("\n" + "=" * 120)
print("ATTACK FAMILY FAILURE ANALYSIS")
print("=" * 120)


family_rows = []


FAMILY_ORDER = [

    "DOS_DDOS",

    "RECON_SCAN",

    "EXPLOIT_APP",

    "BRUTE_AUTH",

    "FUZZING",
]


for H in HORIZONS:

    for seed in FINAL_SEEDS:

        base = predictions.loc[
            (
                predictions[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                predictions[
                    "Seed"
                ]
                ==
                seed
            )
        ].copy()


        for family in FAMILY_ORDER:

            subset = base.loc[
                base[
                    "Episode_Attack_Family"
                ]
                ==
                family
            ].copy()


            if len(
                subset
            ) == 0:

                continue


            metrics = compute_slice_metrics(
                subset
            )


            event_metrics = compute_event_metrics(
                subset
            )


            family_rows.append(
                {
                    "Horizon":
                        H,

                    "Seed":
                        seed,

                    "Attack_Family":
                        family,

                    **metrics,

                    **event_metrics,
                }
            )


family_detail = pd.DataFrame(
    family_rows
)


family_detail.to_csv(

    OUTPUT_ROOT
    / "ATTACK_FAMILY_BY_SEED.csv",

    index=False
)


# ==================================================================================================
# SOURCE ANALYSIS
# ==================================================================================================

print("\n" + "=" * 120)
print("CALIBRATION SOURCE FAILURE ANALYSIS")
print("=" * 120)


source_rows = []


sources = sorted(

    predictions[
        "Calibration_Source_Normalized"
    ].dropna().unique()
)


for H in HORIZONS:

    for seed in FINAL_SEEDS:

        base = predictions.loc[
            (
                predictions[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                predictions[
                    "Seed"
                ]
                ==
                seed
            )
        ].copy()


        for source in sources:

            subset = base.loc[
                base[
                    "Calibration_Source_Normalized"
                ]
                ==
                source
            ].copy()


            if len(
                subset
            ) == 0:

                continue


            metrics = compute_slice_metrics(
                subset
            )


            event_metrics = compute_event_metrics(
                subset
            )


            source_rows.append(
                {
                    "Horizon":
                        H,

                    "Seed":
                        seed,

                    "Calibration_Source":
                        source,

                    **metrics,

                    **event_metrics,
                }
            )


source_detail = pd.DataFrame(
    source_rows
)


source_detail.to_csv(

    OUTPUT_ROOT
    / "CALIBRATION_SOURCE_BY_SEED.csv",

    index=False
)


# ==================================================================================================
# BENIGN-ONLY FALSE WARNING ANALYSIS
# ==================================================================================================

print("\n" + "=" * 120)
print("BENIGN-ONLY FALSE-WARNING ANALYSIS")
print("=" * 120)


benign_rows = []


for H in HORIZONS:

    for seed in FINAL_SEEDS:

        subset = predictions.loc[
            (
                predictions[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                predictions[
                    "Seed"
                ]
                ==
                seed
            )

            &

            (
                predictions[
                    "Episode_Type"
                ]
                .str.upper()
                .str.contains(
                    "BENIGN"
                )
            )
        ].copy()


        if len(
            subset
        ) == 0:

            continue


        metrics = compute_slice_metrics(
            subset
        )


        benign_rows.append(
            {
                "Horizon":
                    H,

                "Seed":
                    seed,

                **metrics,
            }
        )


benign_detail = pd.DataFrame(
    benign_rows
)


benign_detail.to_csv(

    OUTPUT_ROOT
    / "BENIGN_ONLY_FALSE_WARNINGS_BY_SEED.csv",

    index=False
)


# ==================================================================================================
# TIME-TO-ONSET ANALYSIS
#
# Positive origins only.
#
# This tells us whether the model mainly succeeds 1–2 minutes before onset
# and fails farther out.
# ==================================================================================================

print("\n" + "=" * 120)
print("TIME-TO-ONSET SCORE PROFILE")
print("=" * 120)


tto_rows = []


positive_predictions = predictions.loc[
    predictions[
        "Y_True"
    ]
    ==
    1
].copy()


positive_predictions[
    "TTO_Minute"
] = pd.to_numeric(

    positive_predictions[
        "Minutes_To_Next_Onset"
    ],

    errors="coerce"

).round().astype(
    "Int64"
)


for (
    H,
    seed,
    tto_minute
), group in positive_predictions.groupby(
    [
        "Horizon",
        "Seed",
        "TTO_Minute"
    ],
    dropna=True
):

    scores = group[
        "Probability"
    ].to_numpy(
        dtype=float
    )


    pred = group[
        "Prediction"
    ].to_numpy(
        dtype=np.int8
    )


    tto_rows.append(
        {
            "Horizon":
                int(
                    H
                ),

            "Seed":
                int(
                    seed
                ),

            "Minutes_To_Onset":
                int(
                    tto_minute
                ),

            "Positive_Origins":
                int(
                    len(
                        group
                    )
                ),

            "Detected_Origins":
                int(
                    pred.sum()
                ),

            "Recall_At_Frozen_Threshold":
                float(
                    pred.mean()
                ),

            "Mean_Probability":
                float(
                    scores.mean()
                ),

            "Median_Probability":
                float(
                    np.median(
                        scores
                    )
                ),

            "P25_Probability":
                float(
                    np.percentile(
                        scores,
                        25
                    )
                ),

            "P75_Probability":
                float(
                    np.percentile(
                        scores,
                        75
                    )
                ),
        }
    )


tto_detail = pd.DataFrame(
    tto_rows
)


tto_detail.to_csv(

    OUTPUT_ROOT
    / "TIME_TO_ONSET_BY_SEED.csv",

    index=False
)


# ==================================================================================================
# SUMMARIZE ACROSS THREE SEEDS
# ==================================================================================================

SUMMARY_METRICS = [

    "Samples",

    "Positives",

    "Negatives",

    "Prevalence",

    "Accuracy",

    "Balanced_Accuracy",

    "Precision",

    "Recall",

    "Specificity",

    "F1",

    "MCC",

    "ROC_AUC",

    "Average_Precision",

    "PR_AUC",

    "AP_Lift",

    "Brier",

    "TP",

    "FP",

    "TN",

    "FN",

    "Predicted_Positive_Rate",

    "Mean_Probability",

    "Median_Probability",

    "P95_Probability",

    "Unique_True_Events",

    "Recalled_Events",

    "Event_Recall",

    "Event_Precision",

    "Median_First_Warning_Lead_Min",

    "Mean_First_Warning_Lead_Min",
]


def summarize_across_seeds(
    df,
    group_columns
):

    rows = []


    for group_values, group in df.groupby(
        group_columns,
        dropna=False
    ):

        if not isinstance(
            group_values,
            tuple
        ):

            group_values = (
                group_values,
            )


        row = {

            column:
                value

            for column, value
            in zip(
                group_columns,
                group_values
            )
        }


        row[
            "Replicates"
        ] = len(
            group
        )


        for metric in SUMMARY_METRICS:

            if metric not in group.columns:

                continue


            values = pd.to_numeric(

                group[
                    metric
                ],

                errors="coerce"

            ).to_numpy(
                dtype=float
            )


            if np.isfinite(
                values
            ).any():

                row[
                    f"{metric}_Mean"
                ] = float(
                    np.nanmean(
                        values
                    )
                )


                row[
                    f"{metric}_Std"
                ] = (

                    float(
                        np.nanstd(
                            values,
                            ddof=1
                        )
                    )

                    if np.isfinite(
                        values
                    ).sum()
                    >
                    1

                    else np.nan
                )


            else:

                row[
                    f"{metric}_Mean"
                ] = np.nan


                row[
                    f"{metric}_Std"
                ] = np.nan


        rows.append(
            row
        )


    return pd.DataFrame(
        rows
    )


regime_summary = summarize_across_seeds(

    regime_detail,

    [
        "Horizon",
        "Precursor_Regime"
    ]
)


family_summary = summarize_across_seeds(

    family_detail,

    [
        "Horizon",
        "Attack_Family"
    ]
)


source_summary = summarize_across_seeds(

    source_detail,

    [
        "Horizon",
        "Calibration_Source"
    ]
)


benign_summary = summarize_across_seeds(

    benign_detail,

    [
        "Horizon"
    ]
)


regime_summary.to_csv(

    OUTPUT_ROOT
    / "PRECURSOR_REGIME_SUMMARY.csv",

    index=False
)


family_summary.to_csv(

    OUTPUT_ROOT
    / "ATTACK_FAMILY_SUMMARY.csv",

    index=False
)


source_summary.to_csv(

    OUTPUT_ROOT
    / "CALIBRATION_SOURCE_SUMMARY.csv",

    index=False
)


benign_summary.to_csv(

    OUTPUT_ROOT
    / "BENIGN_ONLY_FALSE_WARNING_SUMMARY.csv",

    index=False
)


# ==================================================================================================
# TTO SUMMARY
# ==================================================================================================

tto_summary = (

    tto_detail

    .groupby(
        [
            "Horizon",
            "Minutes_To_Onset"
        ],
        as_index=False
    )

    .agg(
        Replicates=(
            "Seed",
            "count"
        ),

        Positive_Origins_Mean=(
            "Positive_Origins",
            "mean"
        ),

        Recall_Mean=(
            "Recall_At_Frozen_Threshold",
            "mean"
        ),

        Recall_Std=(
            "Recall_At_Frozen_Threshold",
            "std"
        ),

        Mean_Probability=(
            "Mean_Probability",
            "mean"
        ),

        Mean_Probability_Std=(
            "Mean_Probability",
            "std"
        ),

        Median_Probability=(
            "Median_Probability",
            "mean"
        )
    )
)


tto_summary.to_csv(

    OUTPUT_ROOT
    / "TIME_TO_ONSET_SUMMARY.csv",

    index=False
)


# ==================================================================================================
# PRINT REGIME SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("PRECURSOR REGIME — THREE-SEED MEAN")
print("=" * 120)


regime_display = [

    "Horizon",

    "Precursor_Regime",

    "Positives_Mean",

    "Prevalence_Mean",

    "Average_Precision_Mean",

    "AP_Lift_Mean",

    "ROC_AUC_Mean",

    "Precision_Mean",

    "Recall_Mean",

    "F1_Mean",

    "MCC_Mean",

    "Event_Recall_Mean",

    "Event_Precision_Mean",

    "Median_First_Warning_Lead_Min_Mean",
]


print(

    regime_summary[
        [
            column

            for column
            in regime_display

            if column
            in regime_summary.columns
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# PRINT FAMILY SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("ATTACK FAMILY — THREE-SEED MEAN")
print("=" * 120)


family_display = [

    "Horizon",

    "Attack_Family",

    "Positives_Mean",

    "Average_Precision_Mean",

    "AP_Lift_Mean",

    "ROC_AUC_Mean",

    "Precision_Mean",

    "Recall_Mean",

    "F1_Mean",

    "MCC_Mean",

    "Event_Recall_Mean",

    "Event_Precision_Mean",
]


print(

    family_summary[
        [
            column

            for column
            in family_display

            if column
            in family_summary.columns
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# PRINT SOURCE SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("CALIBRATION SOURCE — THREE-SEED MEAN")
print("=" * 120)


source_display = [

    "Horizon",

    "Calibration_Source",

    "Samples_Mean",

    "Positives_Mean",

    "Prevalence_Mean",

    "Average_Precision_Mean",

    "AP_Lift_Mean",

    "ROC_AUC_Mean",

    "Precision_Mean",

    "Recall_Mean",

    "F1_Mean",

    "MCC_Mean",

    "Event_Recall_Mean",

    "Event_Precision_Mean",
]


print(

    source_summary[
        [
            column

            for column
            in source_display

            if column
            in source_summary.columns
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# PRINT BENIGN-ONLY FALSE WARNING SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("BENIGN-ONLY EPISODES — FALSE WARNING PROFILE")
print("=" * 120)


benign_display = [

    "Horizon",

    "Samples_Mean",

    "FP_Mean",

    "Specificity_Mean",

    "Predicted_Positive_Rate_Mean",

    "Mean_Probability_Mean",

    "Median_Probability_Mean",

    "P95_Probability_Mean",
]


print(

    benign_summary[
        [
            column

            for column
            in benign_display

            if column
            in benign_summary.columns
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# PRINT TIME-TO-ONSET SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("POSITIVE ORIGINS — PERFORMANCE BY MINUTES TO ONSET")
print("=" * 120)


print(

    tto_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# AUTOMATIC FAILURE FINDINGS
# ==================================================================================================

print("\n" + "=" * 120)
print("AUTOMATIC FAILURE-SLICE FINDINGS")
print("=" * 120)


findings = {}


for H in HORIZONS:

    findings[
        str(
            H
        )
    ] = {}


    # ----------------------------------------------------------------------------------------------
    # REGIME
    # ------------------------------------------------------------------------------------------

    r = regime_summary.loc[
        regime_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        r
    ):

        r_valid = r.dropna(
            subset=[
                "Average_Precision_Mean"
            ]
        )


        if len(
            r_valid
        ):

            best_regime = r_valid.loc[

                r_valid[
                    "Average_Precision_Mean"
                ].idxmax()
            ]


            worst_regime = r_valid.loc[

                r_valid[
                    "Average_Precision_Mean"
                ].idxmin()
            ]


            findings[
                str(
                    H
                )
            ][
                "Best_Regime"
            ] = {

                "Name":
                    str(
                        best_regime[
                            "Precursor_Regime"
                        ]
                    ),

                "AP":
                    float(
                        best_regime[
                            "Average_Precision_Mean"
                        ]
                    ),
            }


            findings[
                str(
                    H
                )
            ][
                "Worst_Regime"
            ] = {

                "Name":
                    str(
                        worst_regime[
                            "Precursor_Regime"
                        ]
                    ),

                "AP":
                    float(
                        worst_regime[
                            "Average_Precision_Mean"
                        ]
                    ),
            }


            print(
                f"\nH{H} precursor regime:"
            )


            print(
                "  BEST :",
                best_regime[
                    "Precursor_Regime"
                ],
                "| AP =",
                round(
                    float(
                        best_regime[
                            "Average_Precision_Mean"
                        ]
                    ),
                    4
                )
            )


            print(
                "  WORST:",
                worst_regime[
                    "Precursor_Regime"
                ],
                "| AP =",
                round(
                    float(
                        worst_regime[
                            "Average_Precision_Mean"
                        ]
                    ),
                    4
                )
            )


    # ----------------------------------------------------------------------------------------------
    # FAMILY
    # ------------------------------------------------------------------------------------------

    f = family_summary.loc[
        family_summary[
            "Horizon"
        ]
        ==
        H
    ].dropna(
        subset=[
            "Average_Precision_Mean"
        ]
    )


    if len(
        f
    ):

        best_family = f.loc[

            f[
                "Average_Precision_Mean"
            ].idxmax()
        ]


        worst_family = f.loc[

            f[
                "Average_Precision_Mean"
            ].idxmin()
        ]


        findings[
            str(
                H
            )
        ][
            "Best_Family"
        ] = {

            "Name":
                str(
                    best_family[
                        "Attack_Family"
                    ]
                ),

            "AP":
                float(
                    best_family[
                        "Average_Precision_Mean"
                    ]
                ),
        }


        findings[
            str(
                H
            )
        ][
            "Worst_Family"
        ] = {

            "Name":
                str(
                    worst_family[
                        "Attack_Family"
                    ]
                ),

            "AP":
                float(
                    worst_family[
                        "Average_Precision_Mean"
                    ]
                ),
        }


        print(
            f"H{H} family:"
        )


        print(
            "  BEST :",
            best_family[
                "Attack_Family"
            ],
            "| AP =",
            round(
                float(
                    best_family[
                        "Average_Precision_Mean"
                    ]
                ),
                4
            )
        )


        print(
            "  WORST:",
            worst_family[
                "Attack_Family"
            ],
            "| AP =",
            round(
                float(
                    worst_family[
                        "Average_Precision_Mean"
                    ]
                ),
                4
            )
        )


    # ----------------------------------------------------------------------------------------------
    # SOURCE GAP
    # ------------------------------------------------------------------------------------------

    s = source_summary.loc[
        source_summary[
            "Horizon"
        ]
        ==
        H
    ].dropna(
        subset=[
            "Average_Precision_Mean"
        ]
    )


    if len(
        s
    ) >= 2:

        source_ap = {

            str(
                row[
                    "Calibration_Source"
                ]
            ):
                float(
                    row[
                        "Average_Precision_Mean"
                    ]
                )

            for _, row
            in s.iterrows()
        }


        findings[
            str(
                H
            )
        ][
            "Source_AP"
        ] = source_ap


        ap_values = list(
            source_ap.values()
        )


        source_gap = (

            max(
                ap_values
            )

            -
            min(
                ap_values
            )
        )


        findings[
            str(
                H
            )
        ][
            "Absolute_Source_AP_Gap"
        ] = float(
            source_gap
        )


        print(
            f"H{H} source AP:"
        )


        for source, ap_value in source_ap.items():

            print(
                f"  {source}: {ap_value:.4f}"
            )


        print(
            f"  absolute source gap = {source_gap:.4f}"
        )


write_json(

    findings,

    OUTPUT_ROOT
    / "AUTOMATIC_FAILURE_FINDINGS.json"
)


# ==================================================================================================
# PLOTS
# ==================================================================================================

# --------------------------------------------------------------------------------------------------
# REGIME AP
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = regime_summary.loc[
        regime_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        plot_df
    ) == 0:

        continue


    plot_df[
        "Order"
    ] = plot_df[
        "Precursor_Regime"
    ].map(
        {
            name:
                i

            for i, name
            in enumerate(
                REGIME_ORDER
            )
        }
    )


    plot_df = plot_df.sort_values(
        "Order"
    )


    plt.figure(
        figsize=(
            10,
            5
        )
    )


    plt.bar(

        plot_df[
            "Precursor_Regime"
        ],

        plot_df[
            "Average_Precision_Mean"
        ]
    )


    plt.ylabel(
        "Mean Average Precision"
    )


    plt.xlabel(
        "Precursor Regime"
    )


    plt.title(
        f"LSTM W10/H{H} — AP by Precursor Regime"
    )


    plt.xticks(
        rotation=30,
        ha="right"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_AP_by_precursor_regime.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# --------------------------------------------------------------------------------------------------
# REGIME EVENT RECALL
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = regime_summary.loc[
        regime_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        plot_df
    ) == 0:

        continue


    plot_df[
        "Order"
    ] = plot_df[
        "Precursor_Regime"
    ].map(
        {
            name:
                i

            for i, name
            in enumerate(
                REGIME_ORDER
            )
        }
    )


    plot_df = plot_df.sort_values(
        "Order"
    )


    plt.figure(
        figsize=(
            10,
            5
        )
    )


    plt.bar(

        plot_df[
            "Precursor_Regime"
        ],

        plot_df[
            "Event_Recall_Mean"
        ]
    )


    plt.ylim(
        0,
        1
    )


    plt.ylabel(
        "Mean Event Recall"
    )


    plt.xlabel(
        "Precursor Regime"
    )


    plt.title(
        f"LSTM W10/H{H} — Event Recall by Precursor Regime"
    )


    plt.xticks(
        rotation=30,
        ha="right"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_event_recall_by_precursor_regime.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# --------------------------------------------------------------------------------------------------
# FAMILY AP
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = family_summary.loc[
        family_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        plot_df
    ) == 0:

        continue


    plot_df[
        "Order"
    ] = plot_df[
        "Attack_Family"
    ].map(
        {
            name:
                i

            for i, name
            in enumerate(
                FAMILY_ORDER
            )
        }
    )


    plot_df = plot_df.sort_values(
        "Order"
    )


    plt.figure(
        figsize=(
            9,
            5
        )
    )


    plt.bar(

        plot_df[
            "Attack_Family"
        ],

        plot_df[
            "Average_Precision_Mean"
        ]
    )


    plt.ylabel(
        "Mean Average Precision"
    )


    plt.xlabel(
        "Attack Family"
    )


    plt.title(
        f"LSTM W10/H{H} — AP by Attack Family"
    )


    plt.xticks(
        rotation=25,
        ha="right"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_AP_by_attack_family.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# --------------------------------------------------------------------------------------------------
# SOURCE AP
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = source_summary.loc[
        source_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        plot_df
    ) == 0:

        continue


    plt.figure(
        figsize=(
            7,
            5
        )
    )


    plt.bar(

        plot_df[
            "Calibration_Source"
        ],

        plot_df[
            "Average_Precision_Mean"
        ]
    )


    plt.ylabel(
        "Mean Average Precision"
    )


    plt.xlabel(
        "Calibration Source"
    )


    plt.title(
        f"LSTM W10/H{H} — AP by Calibration Source"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_AP_by_calibration_source.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# --------------------------------------------------------------------------------------------------
# TTO PROBABILITY PROFILE
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = tto_summary.loc[
        tto_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        plot_df
    ) == 0:

        continue


    plot_df = plot_df.sort_values(
        "Minutes_To_Onset"
    )


    plt.figure(
        figsize=(
            8,
            5
        )
    )


    plt.plot(

        plot_df[
            "Minutes_To_Onset"
        ],

        plot_df[
            "Mean_Probability"
        ],

        marker="o"
    )


    plt.xlabel(
        "Minutes to True Attack Onset"
    )


    plt.ylabel(
        "Mean Predicted Attack Probability"
    )


    plt.title(
        f"LSTM W10/H{H} — Probability vs Time to Onset"
    )


    plt.grid(
        alpha=0.25
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_probability_vs_time_to_onset.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# --------------------------------------------------------------------------------------------------
# TTO RECALL PROFILE
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = tto_summary.loc[
        tto_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        plot_df
    ) == 0:

        continue


    plot_df = plot_df.sort_values(
        "Minutes_To_Onset"
    )


    plt.figure(
        figsize=(
            8,
            5
        )
    )


    plt.plot(

        plot_df[
            "Minutes_To_Onset"
        ],

        plot_df[
            "Recall_Mean"
        ],

        marker="o"
    )


    plt.ylim(
        0,
        1
    )


    plt.xlabel(
        "Minutes to True Attack Onset"
    )


    plt.ylabel(
        "Recall at Original Frozen Threshold"
    )


    plt.title(
        f"LSTM W10/H{H} — Detection Recall vs Time to Onset"
    )


    plt.grid(
        alpha=0.25
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_recall_vs_time_to_onset.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ==================================================================================================
# COMPLETE MARKER
# ==================================================================================================

write_json(
    {
        "Cell":
            5,

        "Analysis":
            "LSTM_FAILURE_SLICE_ANALYSIS",

        "Evaluation_Split":
            "VALIDATION",

        "Horizons":
            HORIZONS,

        "Seeds":
            FINAL_SEEDS,

        "Training_Performed":
            False,

        "Hyperparameter_Tuning_Performed":
            False,

        "Threshold_Retuning_Performed":
            False,

        "Slice_Specific_Thresholds":
            False,

        "Original_Thresholds_Preserved":
            True,

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "Outputs": [

            "PRECURSOR_REGIME_SUMMARY.csv",

            "ATTACK_FAMILY_SUMMARY.csv",

            "CALIBRATION_SOURCE_SUMMARY.csv",

            "BENIGN_ONLY_FALSE_WARNING_SUMMARY.csv",

            "TIME_TO_ONSET_SUMMARY.csv",

            "AUTOMATIC_FAILURE_FINDINGS.json",
        ],

        "Next":
            (
                "REVIEW FAILURE SLICES BEFORE "
                "CLEAN EPISODE-CV LSTM OPTIMIZATION"
            ),
    },

    COMPLETE_FILE
)


print("\n" + "=" * 120)
print("✓ CELL 5 — FAILURE-SLICE ANALYSIS COMPLETE")
print("=" * 120)


print(
    "\nSaved to:"
)

print(
    OUTPUT_ROOT
)


print(
    "\nNO TRAINING ✓"
)

print(
    "NO HYPERPARAMETER CHANGES ✓"
)

print(
    "NO THRESHOLD RETUNING ✓"
)

print(
    "NO TEST DATA ✓"
)


print(
    "\nNEXT:"
)

print(
    "Review precursor-regime / family / source / time-to-onset results."
)

print(
    "Then build the CLEAN episode-CV optimization pipeline."
)


print("=" * 120)


gc.collect()

In [ ]:
# ==================================================================================================
# LSTM MODEL NOTEBOOK — CELL 5B
#
# TARGETED FALSE-POSITIVE BOUNDARY + SHORT-PRECURSOR SIGNAL AUDIT
#
# NO TRAINING
# NO TUNING
# NO NEW THRESHOLDS
# NO TEST
#
# ==================================================================================================

from pathlib import Path
import json
import shutil
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


print("=" * 120)
print("CELL 5B — FALSE-POSITIVE BOUNDARY + SHORT-PRECURSOR SIGNAL AUDIT")
print("VALIDATION ONLY")
print("NO TRAINING / NO TEST")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

P12_2 = (
    ROOT
    / "model_ready"
    / "12_2"
)

MODEL_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
)

DEVELOPMENT_LOCK_FILE = (
    MODEL_ROOT
    / "LSTM_DEVELOPMENT_LOCK.json"
)

VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)

X_VAL_FILE = (
    P12_2
    / "transformed"
    / "W10"
    / "validation"
    / "X_log1p_standardized.npy"
)

Y_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "y_binary.npy"
)

ELIGIBLE_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "eligible.npy"
)

OUTPUT_ROOT = (
    MODEL_ROOT
    / "validation_diagnostics"
    / "failure_slices"
    / "cell5b"
)

COMPLETE_FILE = (
    OUTPUT_ROOT
    / "CELL5B_COMPLETE.json"
)


FINAL_SEEDS = [
    20260930,
    20260931,
    20260932,
]

HORIZONS = [
    5,
    10,
]

HORIZON_TO_COLUMN = {
    1: 0,
    5: 1,
    10: 2,
    15: 3,
    30: 4,
}

FEATURES = [
    "Flow_Count",
    "Flow_Duration_Mean",
    "Flow_Duration_Std",
    "Flow_Duration_Max",
    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Total_Packets",
    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum",
    "Total_Bytes",
    "Packets_Per_Second",
    "Bytes_Per_Second",
    "Forward_Backward_Packet_Ratio",
    "Forward_Backward_Byte_Ratio",
]

SHORT_REGIMES = [
    "SHORT_WEAK",
    "SHORT_STRONG",
]


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )


def read_json(path):

    require_file(path)

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def normalize_text(series):

    return (
        series
        .astype("string")
        .fillna("NONE")
        .astype(str)
        .str.strip()
    )


def pooled_cohens_d(
    group_a,
    group_b
):

    a = np.asarray(
        group_a,
        dtype=np.float64
    )

    b = np.asarray(
        group_b,
        dtype=np.float64
    )


    if (
        len(a) < 2
        or
        len(b) < 2
    ):

        return np.nan


    var_a = np.var(
        a,
        ddof=1
    )

    var_b = np.var(
        b,
        ddof=1
    )


    pooled_var = (
        (
            (len(a) - 1) * var_a
            +
            (len(b) - 1) * var_b
        )
        /
        (
            len(a)
            +
            len(b)
            -
            2
        )
    )


    if pooled_var <= 0:

        return 0.0


    return float(
        (
            np.mean(a)
            -
            np.mean(b)
        )
        /
        np.sqrt(
            pooled_var
        )
    )


# ==================================================================================================
# VERIFY ORIGINAL EXPERIMENT
# ==================================================================================================

for path in [
    DEVELOPMENT_LOCK_FILE,
    VAL_INDEX_FILE,
    X_VAL_FILE,
    Y_VAL_FILE,
    ELIGIBLE_VAL_FILE,
]:

    require_file(
        path
    )


development_lock = read_json(
    DEVELOPMENT_LOCK_FILE
)


if development_lock.get(
    "Status"
) != "TRAIN_VALIDATION_DEVELOPMENT_COMPLETE":

    raise RuntimeError(
        "Original LSTM run is not complete."
    )


if development_lock.get(
    "TEST_Loaded"
) is not False:

    raise RuntimeError(
        "TEST was reportedly loaded."
    )


if development_lock.get(
    "TEST_Used"
) is not False:

    raise RuntimeError(
        "TEST was reportedly used."
    )


if development_lock.get(
    "TEST_Evaluation_Performed"
) is not False:

    raise RuntimeError(
        "TEST was reportedly evaluated."
    )


print(
    "✓ Original LSTM baseline verified"
)

print(
    "✓ TEST was never loaded"
)

print(
    "✓ TEST was never evaluated"
)


# ==================================================================================================
# OUTPUT SAFETY
# ==================================================================================================

if COMPLETE_FILE.is_file():

    print(
        "\n✓ Cell 5B already completed:"
    )

    print(
        OUTPUT_ROOT
    )

    raise SystemExit


if OUTPUT_ROOT.exists():

    attempt = 1


    while (
        OUTPUT_ROOT.parent
        / f"_incomplete_cell5b_attempt_{attempt}"
    ).exists():

        attempt += 1


    backup = (
        OUTPUT_ROOT.parent
        / f"_incomplete_cell5b_attempt_{attempt}"
    )


    shutil.move(
        str(
            OUTPUT_ROOT
        ),
        str(
            backup
        )
    )


    print(
        "Previous incomplete Cell 5B preserved:"
    )

    print(
        backup
    )


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# LOAD VALIDATION MODEL-READY DATA
# ==================================================================================================

val_index = pd.read_parquet(
    VAL_INDEX_FILE
)


X_val = np.load(
    X_VAL_FILE,
    mmap_mode="r"
)


Y_val = np.load(
    Y_VAL_FILE,
    allow_pickle=False
)


eligible_val = np.load(
    ELIGIBLE_VAL_FILE,
    allow_pickle=False
)


if len(
    val_index
) != X_val.shape[0]:

    raise RuntimeError(
        "Validation index/X alignment mismatch."
    )


if Y_val.shape != eligible_val.shape:

    raise RuntimeError(
        "Validation label/eligibility shape mismatch."
    )


if X_val.shape[1:] != (
    10,
    14
):

    raise RuntimeError(
        f"Unexpected validation tensor shape: {X_val.shape}"
    )


val_index = val_index.copy()


val_index[
    "Episode_ID"
] = val_index[
    "Episode_ID"
].astype(str)


val_index[
    "Precursor_Regime"
] = normalize_text(
    val_index[
        "Precursor_Regime"
    ]
)


val_index[
    "Episode_Type"
] = normalize_text(
    val_index[
        "Episode_Type"
    ]
)


val_index[
    "Episode_Attack_Family"
] = normalize_text(
    val_index[
        "Episode_Attack_Family"
    ]
)


val_index[
    "Minutes_To_Next_Onset"
] = pd.to_numeric(

    val_index[
        "Minutes_To_Next_Onset"
    ],

    errors="coerce"
)


# ==================================================================================================
# LOAD ORIGINAL LSTM PREDICTIONS
# ==================================================================================================

prediction_sets = []


for H in HORIZONS:

    for seed in FINAL_SEEDS:

        prediction_file = (
            MODEL_ROOT
            / f"H{H:02d}"
            / "final_replicates"
            / f"seed_{seed}"
            / "validation_predictions.parquet"
        )


        require_file(
            prediction_file
        )


        pred = pd.read_parquet(
            prediction_file
        )


        pred[
            "Episode_ID"
        ] = pred[
            "Episode_ID"
        ].astype(str)


        pred[
            "Minute_Index"
        ] = pd.to_numeric(
            pred[
                "Minute_Index"
            ],
            errors="raise"
        ).astype(np.int64)


        merge_metadata = val_index[
            [
                "Episode_ID",
                "Minute_Index",
                "Episode_Type",
                "Episode_Attack_Family",
                "Precursor_Regime",
                "Minutes_To_Next_Onset",
            ]
        ]


        pred = pred.drop(
            columns=[
                "Minutes_To_Next_Onset"
            ],
            errors="ignore"
        )


        joined = pred.merge(

            merge_metadata,

            on=[
                "Episode_ID",
                "Minute_Index"
            ],

            how="left",

            validate="one_to_one"
        )


        if joined[
            "Episode_Type"
        ].isna().any():

            raise RuntimeError(
                f"H{H} seed={seed}: metadata merge failed."
            )


        joined[
            "Horizon"
        ] = H


        joined[
            "Seed"
        ] = seed


        prediction_sets.append(
            joined
        )


predictions = pd.concat(
    prediction_sets,
    ignore_index=True
)


del prediction_sets

gc.collect()


# ==================================================================================================
# PART A — WHERE DO FALSE POSITIVES ACTUALLY OCCUR?
# ==================================================================================================

print("\n" + "=" * 120)
print("PART A — FALSE-POSITIVE BOUNDARY ANALYSIS")
print("=" * 120)


def assign_negative_bucket(
    row
):

    H = int(
        row[
            "Horizon"
        ]
    )


    if (
        "BENIGN"
        in
        str(
            row[
                "Episode_Type"
            ]
        ).upper()
    ):

        return "BENIGN_ONLY"


    tto = row[
        "Minutes_To_Next_Onset"
    ]


    if not np.isfinite(
        tto
    ):

        return "ATTACK_EPISODE_NO_TTO"


    if tto <= H:

        return "ERROR_NEGATIVE_INSIDE_HORIZON"


    extra = (
        tto
        -
        H
    )


    if extra <= 2:

        return "EARLY_BY_1_2_MIN"


    elif extra <= 5:

        return "EARLY_BY_3_5_MIN"


    elif extra <= 10:

        return "EARLY_BY_6_10_MIN"


    else:

        return "EARLY_BY_GT10_MIN"


negative_predictions = predictions.loc[
    predictions[
        "Y_True"
    ]
    ==
    0
].copy()


negative_predictions[
    "Negative_Bucket"
] = negative_predictions.apply(
    assign_negative_bucket,
    axis=1
)


bad_negative_semantics = negative_predictions.loc[
    negative_predictions[
        "Negative_Bucket"
    ]
    ==
    "ERROR_NEGATIVE_INSIDE_HORIZON"
]


if len(
    bad_negative_semantics
):

    raise RuntimeError(
        "Found negative labels whose onset is actually inside H."
    )


bucket_rows = []


for (
    H,
    seed,
    bucket
), group in negative_predictions.groupby(
    [
        "Horizon",
        "Seed",
        "Negative_Bucket"
    ]
):

    warnings = (
        group[
            "Prediction"
        ].to_numpy(
            dtype=np.int8
        )
        ==
        1
    )


    probs = group[
        "Probability"
    ].to_numpy(
        dtype=float
    )


    total_fp_this_seed = int(

        (
            (
                negative_predictions[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                negative_predictions[
                    "Seed"
                ]
                ==
                seed
            )

            &

            (
                negative_predictions[
                    "Prediction"
                ]
                ==
                1
            )
        ).sum()
    )


    fp_count = int(
        warnings.sum()
    )


    bucket_rows.append(
        {
            "Horizon":
                int(H),

            "Seed":
                int(seed),

            "Negative_Bucket":
                bucket,

            "Negative_Origins":
                int(
                    len(
                        group
                    )
                ),

            "False_Positive_Warnings":
                fp_count,

            "False_Positive_Rate":
                float(
                    warnings.mean()
                ),

            "Fraction_Of_All_False_Positives":
                (
                    fp_count
                    /
                    total_fp_this_seed

                    if total_fp_this_seed
                    >
                    0

                    else np.nan
                ),

            "Mean_Probability":
                float(
                    probs.mean()
                ),

            "Median_Probability":
                float(
                    np.median(
                        probs
                    )
                ),

            "P95_Probability":
                float(
                    np.percentile(
                        probs,
                        95
                    )
                ),
        }
    )


bucket_detail = pd.DataFrame(
    bucket_rows
)


bucket_detail.to_csv(

    OUTPUT_ROOT
    / "FALSE_POSITIVE_BOUNDARY_BY_SEED.csv",

    index=False
)


bucket_summary = (

    bucket_detail

    .groupby(
        [
            "Horizon",
            "Negative_Bucket"
        ],
        as_index=False
    )

    .agg(
        Replicates=(
            "Seed",
            "count"
        ),

        Negative_Origins_Mean=(
            "Negative_Origins",
            "mean"
        ),

        False_Positive_Warnings_Mean=(
            "False_Positive_Warnings",
            "mean"
        ),

        False_Positive_Rate_Mean=(
            "False_Positive_Rate",
            "mean"
        ),

        Fraction_Of_All_False_Positives_Mean=(
            "Fraction_Of_All_False_Positives",
            "mean"
        ),

        Mean_Probability_Mean=(
            "Mean_Probability",
            "mean"
        ),

        Median_Probability_Mean=(
            "Median_Probability",
            "mean"
        ),

        P95_Probability_Mean=(
            "P95_Probability",
            "mean"
        ),
    )
)


bucket_summary.to_csv(

    OUTPUT_ROOT
    / "FALSE_POSITIVE_BOUNDARY_SUMMARY.csv",

    index=False
)


print(
    bucket_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# PART B — HOW MANY ATTACK EPISODES RECEIVE AN EARLY WARNING BEFORE H?
# ==================================================================================================

print("\n" + "=" * 120)
print("PART B — PRE-HORIZON EARLY WARNING EPISODES")
print("=" * 120)


early_episode_rows = []


for H in HORIZONS:

    for seed in FINAL_SEEDS:

        subset = predictions.loc[
            (
                predictions[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                predictions[
                    "Seed"
                ]
                ==
                seed
            )
        ].copy()


        attack_subset = subset.loc[
            ~subset[
                "Episode_Type"
            ]
            .str.upper()
            .str.contains(
                "BENIGN"
            )
        ].copy()


        positive_event_ids = set(

            attack_subset.loc[
                attack_subset[
                    "Y_True"
                ]
                ==
                1,
                "Episode_ID"
            ].astype(str)
        )


        early_negative = attack_subset.loc[
            (
                attack_subset[
                    "Y_True"
                ]
                ==
                0
            )

            &

            np.isfinite(
                attack_subset[
                    "Minutes_To_Next_Onset"
                ]
            )

            &

            (
                attack_subset[
                    "Minutes_To_Next_Onset"
                ]
                >
                H
            )

            &

            (
                attack_subset[
                    "Prediction"
                ]
                ==
                1
            )
        ].copy()


        early_episode_ids = set(

            early_negative[
                "Episode_ID"
            ].astype(str)
        )


        early_episode_ids = (
            early_episode_ids

            &
            positive_event_ids
        )


        earliest_leads = []


        for episode_id in early_episode_ids:

            tto_values = pd.to_numeric(

                early_negative.loc[
                    early_negative[
                        "Episode_ID"
                    ].astype(str)
                    ==
                    episode_id,
                    "Minutes_To_Next_Onset"
                ],

                errors="coerce"
            ).dropna()


            if len(
                tto_values
            ):

                earliest_leads.append(
                    float(
                        tto_values.max()
                    )
                )


        early_episode_rows.append(
            {
                "Horizon":
                    H,

                "Seed":
                    seed,

                "Attack_Events":
                    len(
                        positive_event_ids
                    ),

                "Events_With_At_Least_One_PreHorizon_Warning":
                    len(
                        early_episode_ids
                    ),

                "PreHorizon_Event_Warning_Rate":
                    (
                        len(
                            early_episode_ids
                        )
                        /
                        len(
                            positive_event_ids
                        )

                        if len(
                            positive_event_ids
                        )

                        else np.nan
                    ),

                "Median_Earliest_Warning_TTO":
                    (
                        float(
                            np.median(
                                earliest_leads
                            )
                        )

                        if len(
                            earliest_leads
                        )

                        else np.nan
                    ),

                "Mean_Earliest_Warning_TTO":
                    (
                        float(
                            np.mean(
                                earliest_leads
                            )
                        )

                        if len(
                            earliest_leads
                        )

                        else np.nan
                    ),
            }
        )


early_episode_detail = pd.DataFrame(
    early_episode_rows
)


early_episode_summary = (

    early_episode_detail

    .groupby(
        "Horizon",
        as_index=False
    )

    .agg(
        Attack_Events_Mean=(
            "Attack_Events",
            "mean"
        ),

        Events_With_PreHorizon_Warning_Mean=(
            "Events_With_At_Least_One_PreHorizon_Warning",
            "mean"
        ),

        PreHorizon_Event_Warning_Rate_Mean=(
            "PreHorizon_Event_Warning_Rate",
            "mean"
        ),

        PreHorizon_Event_Warning_Rate_Std=(
            "PreHorizon_Event_Warning_Rate",
            "std"
        ),

        Median_Earliest_Warning_TTO_Mean=(
            "Median_Earliest_Warning_TTO",
            "mean"
        ),

        Mean_Earliest_Warning_TTO_Mean=(
            "Mean_Earliest_Warning_TTO",
            "mean"
        ),
    )
)


early_episode_detail.to_csv(

    OUTPUT_ROOT
    / "PREHORIZON_WARNING_EVENTS_BY_SEED.csv",

    index=False
)


early_episode_summary.to_csv(

    OUTPUT_ROOT
    / "PREHORIZON_WARNING_EVENTS_SUMMARY.csv",

    index=False
)


print(
    early_episode_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# PART C — SHORT_WEAK vs SHORT_STRONG FEATURE SEPARABILITY
#
# We do NOT fit a model.
#
# We compare:
#
#    NEAR ONSET:
#        Minutes_To_Onset <= 3
#
# against:
#
#    FAR PRE-ONSET REFERENCE:
#        Minutes_To_Onset > H + 5
#
#
# Representation:
#
#    current standardized feature values
#    +
#    change over W10 = last - first
#
# This gives 28 descriptive dimensions.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("PART C — SHORT_WEAK vs SHORT_STRONG SIGNAL SEPARATION")
print("=" * 120)


separation_rows = []

feature_effect_rows = []


X_val_array = np.asarray(
    X_val,
    dtype=np.float32
)


current_features = X_val_array[
    :,
    -1,
    :
]


delta_features = (

    X_val_array[
        :,
        -1,
        :
    ]

    -

    X_val_array[
        :,
        0,
        :
    ]
)


combined_features = np.concatenate(

    [
        current_features,
        delta_features,
    ],

    axis=1
)


combined_feature_names = (

    [
        f"CURRENT__{feature}"
        for feature
        in FEATURES
    ]

    +

    [
        f"DELTA_W10__{feature}"
        for feature
        in FEATURES
    ]
)


for H in HORIZONS:

    j = HORIZON_TO_COLUMN[
        H
    ]


    eligible_mask = (
        eligible_val[
            :,
            j
        ]
        ==
        1
    )


    for regime in SHORT_REGIMES:

        regime_mask = (
            val_index[
                "Precursor_Regime"
            ].to_numpy()
            ==
            regime
        )


        near_mask = (

            eligible_mask

            &

            regime_mask

            &

            np.isfinite(
                val_index[
                    "Minutes_To_Next_Onset"
                ].to_numpy(
                    dtype=float
                )
            )

            &

            (
                val_index[
                    "Minutes_To_Next_Onset"
                ].to_numpy(
                    dtype=float
                )
                >
                0
            )

            &

            (
                val_index[
                    "Minutes_To_Next_Onset"
                ].to_numpy(
                    dtype=float
                )
                <=
                3
            )
        )


        far_mask = (

            eligible_mask

            &

            regime_mask

            &

            np.isfinite(
                val_index[
                    "Minutes_To_Next_Onset"
                ].to_numpy(
                    dtype=float
                )
            )

            &

            (
                val_index[
                    "Minutes_To_Next_Onset"
                ].to_numpy(
                    dtype=float
                )
                >
                (
                    H
                    +
                    5
                )
            )
        )


        near_X = combined_features[
            near_mask
        ]


        far_X = combined_features[
            far_mask
        ]


        if (
            len(
                near_X
            )
            ==
            0

            or

            len(
                far_X
            )
            ==
            0
        ):

            raise RuntimeError(
                f"H{H} {regime}: insufficient near/far samples."
            )


        # ------------------------------------------------------------------------------------------
        # Pooled mean-vector separation
        # ------------------------------------------------------------------------------------------

        mean_difference = (

            near_X.mean(
                axis=0
            )

            -

            far_X.mean(
                axis=0
            )
        )


        l2_mean_separation = float(
            np.linalg.norm(
                mean_difference
            )
        )


        mean_abs_shift = float(
            np.abs(
                mean_difference
            ).mean()
        )


        max_abs_shift = float(
            np.abs(
                mean_difference
            ).max()
        )


        # ------------------------------------------------------------------------------------------
        # Episode-level separation
        # ------------------------------------------------------------------------------------------

        episode_ids = val_index[
            "Episode_ID"
        ].astype(str).to_numpy()


        candidate_episodes = sorted(

            set(
                episode_ids[
                    near_mask
                ]
            )

            &
            set(
                episode_ids[
                    far_mask
                ]
            )
        )


        episode_distances = []


        for episode_id in candidate_episodes:

            episode_near = (

                near_mask

                &

                (
                    episode_ids
                    ==
                    episode_id
                )
            )


            episode_far = (

                far_mask

                &

                (
                    episode_ids
                    ==
                    episode_id
                )
            )


            near_mean = combined_features[
                episode_near
            ].mean(
                axis=0
            )


            far_mean = combined_features[
                episode_far
            ].mean(
                axis=0
            )


            episode_distances.append(

                float(
                    np.linalg.norm(
                        near_mean
                        -
                        far_mean
                    )
                )
            )


        separation_rows.append(
            {
                "Horizon":
                    H,

                "Precursor_Regime":
                    regime,

                "Near_Onset_Origins":
                    int(
                        near_mask.sum()
                    ),

                "Far_Reference_Origins":
                    int(
                        far_mask.sum()
                    ),

                "Episodes_With_Both_Regions":
                    len(
                        candidate_episodes
                    ),

                "L2_Mean_Vector_Separation":
                    l2_mean_separation,

                "Mean_Absolute_Feature_Shift":
                    mean_abs_shift,

                "Max_Absolute_Feature_Shift":
                    max_abs_shift,

                "Episode_Separation_Mean":
                    float(
                        np.mean(
                            episode_distances
                        )
                    ),

                "Episode_Separation_Median":
                    float(
                        np.median(
                            episode_distances
                        )
                    ),

                "Episode_Separation_Std":
                    float(
                        np.std(
                            episode_distances,
                            ddof=1
                        )
                    ),
            }
        )


        # ------------------------------------------------------------------------------------------
        # Feature-level effect sizes
        # ------------------------------------------------------------------------------------------

        for feature_idx, feature_name in enumerate(
            combined_feature_names
        ):

            effect = pooled_cohens_d(

                near_X[
                    :,
                    feature_idx
                ],

                far_X[
                    :,
                    feature_idx
                ]
            )


            feature_effect_rows.append(
                {
                    "Horizon":
                        H,

                    "Precursor_Regime":
                        regime,

                    "Feature":
                        feature_name,

                    "Cohens_D":
                        effect,

                    "Abs_Cohens_D":
                        abs(
                            effect
                        ),

                    "Near_Mean":
                        float(
                            near_X[
                                :,
                                feature_idx
                            ].mean()
                        ),

                    "Far_Mean":
                        float(
                            far_X[
                                :,
                                feature_idx
                            ].mean()
                        ),
                }
            )


separation_summary = pd.DataFrame(
    separation_rows
)


feature_effects = pd.DataFrame(
    feature_effect_rows
)


separation_summary.to_csv(

    OUTPUT_ROOT
    / "SHORT_REGIME_SIGNAL_SEPARATION.csv",

    index=False
)


feature_effects.to_csv(

    OUTPUT_ROOT
    / "SHORT_REGIME_FEATURE_EFFECTS.csv",

    index=False
)


print(
    separation_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# PART D — TOP FEATURES DISTINGUISHING NEAR ONSET FROM FAR PRE-ONSET
# ==================================================================================================

print("\n" + "=" * 120)
print("PART D — STRONGEST SHORT-PRECURSOR FEATURE EFFECTS")
print("=" * 120)


top_feature_rows = []


for H in HORIZONS:

    for regime in SHORT_REGIMES:

        subset = feature_effects.loc[
            (
                feature_effects[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                feature_effects[
                    "Precursor_Regime"
                ]
                ==
                regime
            )
        ].copy()


        subset = subset.sort_values(

            "Abs_Cohens_D",

            ascending=False
        ).head(
            10
        )


        subset[
            "Rank"
        ] = np.arange(
            1,
            len(
                subset
            )
            +
            1
        )


        top_feature_rows.append(
            subset
        )


        print(
            f"\nH{H} | {regime}"
        )


        print(
            subset[
                [
                    "Rank",
                    "Feature",
                    "Cohens_D",
                    "Abs_Cohens_D",
                ]
            ].to_string(
                index=False
            )
        )


top_features = pd.concat(
    top_feature_rows,
    ignore_index=True
)


top_features.to_csv(

    OUTPUT_ROOT
    / "SHORT_REGIME_TOP_FEATURE_EFFECTS.csv",

    index=False
)


# ==================================================================================================
# PART E — MODEL SCORE BY SHORT REGIME AND EXACT TTO
# ==================================================================================================

print("\n" + "=" * 120)
print("PART E — SHORT REGIME MODEL SCORE BY TIME TO ONSET")
print("=" * 120)


short_score_rows = []


short_predictions = predictions.loc[
    predictions[
        "Precursor_Regime"
    ].isin(
        SHORT_REGIMES
    )
].copy()


short_predictions = short_predictions.loc[
    short_predictions[
        "Y_True"
    ]
    ==
    1
].copy()


short_predictions[
    "TTO_Minute"
] = pd.to_numeric(

    short_predictions[
        "Minutes_To_Next_Onset"
    ],

    errors="coerce"

).round().astype(
    "Int64"
)


for (
    H,
    seed,
    regime,
    tto
), group in short_predictions.groupby(
    [
        "Horizon",
        "Seed",
        "Precursor_Regime",
        "TTO_Minute"
    ],
    dropna=True
):

    short_score_rows.append(
        {
            "Horizon":
                int(
                    H
                ),

            "Seed":
                int(
                    seed
                ),

            "Precursor_Regime":
                regime,

            "Minutes_To_Onset":
                int(
                    tto
                ),

            "Positive_Origins":
                len(
                    group
                ),

            "Mean_Probability":
                float(
                    group[
                        "Probability"
                    ].mean()
                ),

            "Median_Probability":
                float(
                    group[
                        "Probability"
                    ].median()
                ),

            "Recall_At_Frozen_Threshold":
                float(
                    group[
                        "Prediction"
                    ].mean()
                ),
        }
    )


short_score_detail = pd.DataFrame(
    short_score_rows
)


short_score_summary = (

    short_score_detail

    .groupby(
        [
            "Horizon",
            "Precursor_Regime",
            "Minutes_To_Onset"
        ],
        as_index=False
    )

    .agg(
        Replicates=(
            "Seed",
            "count"
        ),

        Positive_Origins_Mean=(
            "Positive_Origins",
            "mean"
        ),

        Mean_Probability=(
            "Mean_Probability",
            "mean"
        ),

        Mean_Probability_Std=(
            "Mean_Probability",
            "std"
        ),

        Recall_Mean=(
            "Recall_At_Frozen_Threshold",
            "mean"
        ),

        Recall_Std=(
            "Recall_At_Frozen_Threshold",
            "std"
        ),
    )
)


short_score_detail.to_csv(

    OUTPUT_ROOT
    / "SHORT_REGIME_TTO_BY_SEED.csv",

    index=False
)


short_score_summary.to_csv(

    OUTPUT_ROOT
    / "SHORT_REGIME_TTO_SUMMARY.csv",

    index=False
)


print(
    short_score_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# PART F — AUTOMATIC INTERPRETATION
# ==================================================================================================

print("\n" + "=" * 120)
print("PART F — AUTOMATIC CELL 5B FINDINGS")
print("=" * 120)


findings = {}


for H in HORIZONS:

    findings[
        str(
            H
        )
    ] = {}


    bucket_H = bucket_summary.loc[
        bucket_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    benign_row = bucket_H.loc[
        bucket_H[
            "Negative_Bucket"
        ]
        ==
        "BENIGN_ONLY"
    ]


    attack_early_rows = bucket_H.loc[
        bucket_H[
            "Negative_Bucket"
        ]
        .str.startswith(
            "EARLY_"
        )
    ]


    benign_fp_fraction = (

        float(
            benign_row[
                "Fraction_Of_All_False_Positives_Mean"
            ].iloc[
                0
            ]
        )

        if len(
            benign_row
        )

        else np.nan
    )


    early_fp_fraction = float(

        attack_early_rows[
            "Fraction_Of_All_False_Positives_Mean"
        ].sum()
    )


    findings[
        str(
            H
        )
    ][
        "Benign_Only_FP_Fraction"
    ] = benign_fp_fraction


    findings[
        str(
            H
        )
    ][
        "Attack_Episode_Early_FP_Fraction"
    ] = early_fp_fraction


    print(
        f"\nH{H}:"
    )


    print(
        f"  fraction of false positives on benign-only origins:"
        f" {benign_fp_fraction:.3f}"
    )


    print(
        f"  fraction of false positives that are early warnings "
        f"inside attack episodes: {early_fp_fraction:.3f}"
    )


    sep_H = separation_summary.loc[
        separation_summary[
            "Horizon"
        ]
        ==
        H
    ]


    weak = sep_H.loc[
        sep_H[
            "Precursor_Regime"
        ]
        ==
        "SHORT_WEAK"
    ].iloc[
        0
    ]


    strong = sep_H.loc[
        sep_H[
            "Precursor_Regime"
        ]
        ==
        "SHORT_STRONG"
    ].iloc[
        0
    ]


    weak_distance = float(
        weak[
            "Episode_Separation_Median"
        ]
    )


    strong_distance = float(
        strong[
            "Episode_Separation_Median"
        ]
    )


    findings[
        str(
            H
        )
    ][
        "SHORT_WEAK_Episode_Separation_Median"
    ] = weak_distance


    findings[
        str(
            H
        )
    ][
        "SHORT_STRONG_Episode_Separation_Median"
    ] = strong_distance


    findings[
        str(
            H
        )
    ][
        "SHORT_STRONG_Greater_Raw_Separation"
    ] = bool(
        strong_distance
        >
        weak_distance
    )


    print(
        f"  SHORT_WEAK median episode separation  : "
        f"{weak_distance:.4f}"
    )


    print(
        f"  SHORT_STRONG median episode separation: "
        f"{strong_distance:.4f}"
    )


    if strong_distance > weak_distance:

        print(
            "  ✓ SHORT_STRONG has stronger underlying feature separation."
        )

        print(
            "    Its lower AP is therefore likely due to composition/"
            "temporal/model interactions, not weaker generated signal."
        )

    else:

        print(
            "  ⚠ SHORT_STRONG does NOT show stronger measured separation."
        )

        print(
            "    We should inspect calibration/source/family composition "
            "before optimization."
        )


write_json(

    findings,

    OUTPUT_ROOT
    / "CELL5B_AUTOMATIC_FINDINGS.json"
)


# ==================================================================================================
# PLOTS — FALSE POSITIVE ORIGIN
# ==================================================================================================

bucket_order = [
    "BENIGN_ONLY",
    "EARLY_BY_1_2_MIN",
    "EARLY_BY_3_5_MIN",
    "EARLY_BY_6_10_MIN",
    "EARLY_BY_GT10_MIN",
    "ATTACK_EPISODE_NO_TTO",
]


for H in HORIZONS:

    plot_df = bucket_summary.loc[
        bucket_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    plot_df[
        "Order"
    ] = plot_df[
        "Negative_Bucket"
    ].map(
        {
            value: i
            for i, value
            in enumerate(
                bucket_order
            )
        }
    )


    plot_df = plot_df.sort_values(
        "Order"
    )


    plt.figure(
        figsize=(
            10,
            5
        )
    )


    plt.bar(

        plot_df[
            "Negative_Bucket"
        ],

        plot_df[
            "False_Positive_Warnings_Mean"
        ]
    )


    plt.ylabel(
        "Mean False-Positive Warning Origins"
    )


    plt.xlabel(
        "Negative-Origin Type"
    )


    plt.title(
        f"LSTM W10/H{H} — Where False Positives Occur"
    )


    plt.xticks(
        rotation=30,
        ha="right"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_false_positive_origin.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ==================================================================================================
# PLOTS — SHORT REGIME FEATURE SEPARATION
# ==================================================================================================

for H in HORIZONS:

    plot_df = separation_summary.loc[
        separation_summary[
            "Horizon"
        ]
        ==
        H
    ].copy()


    plt.figure(
        figsize=(
            7,
            5
        )
    )


    plt.bar(

        plot_df[
            "Precursor_Regime"
        ],

        plot_df[
            "Episode_Separation_Median"
        ]
    )


    plt.ylabel(
        "Median Episode-Level Feature Separation"
    )


    plt.xlabel(
        "Short Precursor Regime"
    )


    plt.title(
        f"W10/H{H} — SHORT_WEAK vs SHORT_STRONG Signal Separation"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_short_regime_signal_separation.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ==================================================================================================
# PLOTS — SHORT REGIME TTO PROBABILITIES
# ==================================================================================================

for H in HORIZONS:

    plt.figure(
        figsize=(
            8,
            5
        )
    )


    for regime in SHORT_REGIMES:

        plot_df = short_score_summary.loc[
            (
                short_score_summary[
                    "Horizon"
                ]
                ==
                H
            )

            &

            (
                short_score_summary[
                    "Precursor_Regime"
                ]
                ==
                regime
            )
        ].copy()


        plot_df = plot_df.sort_values(
            "Minutes_To_Onset"
        )


        plt.plot(

            plot_df[
                "Minutes_To_Onset"
            ],

            plot_df[
                "Mean_Probability"
            ],

            marker="o",

            label=regime
        )


    plt.xlabel(
        "Minutes to True Attack Onset"
    )


    plt.ylabel(
        "Mean Predicted Probability"
    )


    plt.title(
        f"LSTM W10/H{H} — Short-Regime Probability vs TTO"
    )


    plt.legend()

    plt.grid(
        alpha=0.25
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_short_regime_probability_vs_tto.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ==================================================================================================
# COMPLETE
# ==================================================================================================

write_json(
    {
        "Cell":
            "5B",

        "Analysis":
            (
                "FALSE_POSITIVE_BOUNDARY_AND_SHORT_PRECURSOR_SIGNAL_AUDIT"
            ),

        "Evaluation_Split":
            "VALIDATION",

        "Training_Performed":
            False,

        "Hyperparameter_Tuning_Performed":
            False,

        "Threshold_Retuning_Performed":
            False,

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "Outputs": [
            "FALSE_POSITIVE_BOUNDARY_SUMMARY.csv",
            "PREHORIZON_WARNING_EVENTS_SUMMARY.csv",
            "SHORT_REGIME_SIGNAL_SEPARATION.csv",
            "SHORT_REGIME_FEATURE_EFFECTS.csv",
            "SHORT_REGIME_TTO_SUMMARY.csv",
            "CELL5B_AUTOMATIC_FINDINGS.json",
        ],

        "Next":
            "CLEAN_EPISODE_CV_LSTM_OPTIMIZATION",
    },

    COMPLETE_FILE
)


print("\n" + "=" * 120)
print("✓ CELL 5B COMPLETE")
print("=" * 120)


print(
    "\nSaved to:"
)

print(
    OUTPUT_ROOT
)


print(
    "\nNO TRAINING ✓"
)

print(
    "NO THRESHOLD RETUNING ✓"
)

print(
    "NO TEST DATA ✓"
)


print(
    "\nNEXT:"
)

print(
    "Clean episode-level CV + fold-specific preprocessing + serious LSTM optimization."
)


print("=" * 120)


gc.collect()

In [ ]:
# ==================================================================================================
# LSTM MODEL NOTEBOOK — CELL 5C
#
# SHORT_WEAK vs SHORT_STRONG
# COMPOSITION + MATCHED SIGNAL + MATCHED MODEL PERFORMANCE AUDIT
#
# NO TRAINING
# NO HYPERPARAMETER TUNING
# NO THRESHOLD CHANGES
# NO TEST DATA
#
# PURPOSE
# -------
#
# Cell 5B showed:
#
#   SHORT_WEAK observed separation > SHORT_STRONG
#
# even though SHORT_STRONG was generated with a larger nominal precursor alpha.
#
# This cell determines WHY.
#
#
# WE CHECK:
#
#   1. Episode counts
#   2. Attack-family composition
#   3. CIC / UNSW composition
#   4. Precursor-length composition
#   5. Precursor alpha values
#   6. Optional calibration-profile metadata if present
#   7. Episode-level signal separation
#   8. Weak-vs-strong comparisons within SAME family/source
#   9. Original LSTM performance within SAME family/source
#  10. Composition-adjusted weak-vs-strong differences
#
#
# IMPORTANT
# ---------
#
# This cell does NOT attempt to "fix" the dataset.
#
# It is purely explanatory.
#
# ==================================================================================================


from pathlib import Path
import json
import shutil
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)


print("=" * 120)
print("CELL 5C — SHORT_WEAK vs SHORT_STRONG MATCHED COMPOSITION AUDIT")
print("VALIDATION ONLY")
print("NO TRAINING / NO TEST")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

P12_2 = (
    ROOT
    / "model_ready"
    / "12_2"
)

MODEL_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
)

DEVELOPMENT_LOCK_FILE = (
    MODEL_ROOT
    / "LSTM_DEVELOPMENT_LOCK.json"
)

VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)

X_VAL_FILE = (
    P12_2
    / "transformed"
    / "W10"
    / "validation"
    / "X_log1p_standardized.npy"
)

ELIGIBLE_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "eligible.npy"
)

OUTPUT_ROOT = (
    MODEL_ROOT
    / "validation_diagnostics"
    / "failure_slices"
    / "cell5c"
)

COMPLETE_FILE = (
    OUTPUT_ROOT
    / "CELL5C_COMPLETE.json"
)


# ==================================================================================================
# FROZEN CONSTANTS
# ==================================================================================================

FINAL_SEEDS = [
    20260930,
    20260931,
    20260932,
]

HORIZONS = [
    5,
    10,
]

HORIZON_TO_COLUMN = {
    1: 0,
    5: 1,
    10: 2,
    15: 3,
    30: 4,
}

SHORT_REGIMES = [
    "SHORT_WEAK",
    "SHORT_STRONG",
]

FEATURES = [
    "Flow_Count",
    "Flow_Duration_Mean",
    "Flow_Duration_Std",
    "Flow_Duration_Max",
    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Total_Packets",
    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum",
    "Total_Bytes",
    "Packets_Per_Second",
    "Bytes_Per_Second",
    "Forward_Backward_Packet_Ratio",
    "Forward_Backward_Byte_Ratio",
]


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )


def read_json(path):

    require_file(path)

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def normalize_text(series):

    return (
        series
        .astype("string")
        .fillna("NONE")
        .astype(str)
        .str.strip()
    )


def normalize_source(value):

    text = str(
        value
    ).upper()


    if "CIC" in text:

        return "CIC"


    if "UNSW" in text:

        return "UNSW"


    return text


def safe_ap(
    y,
    scores
):

    y = np.asarray(
        y
    )


    if (
        y
        ==
        1
    ).sum() == 0:

        return np.nan


    return float(
        average_precision_score(
            y,
            scores
        )
    )


def safe_roc(
    y,
    scores
):

    y = np.asarray(
        y
    )


    if len(
        np.unique(
            y
        )
    ) < 2:

        return np.nan


    return float(
        roc_auc_score(
            y,
            scores
        )
    )


def total_variation_distance(
    a,
    b
):

    categories = sorted(
        set(
            a.index
        )
        |
        set(
            b.index
        )
    )


    a = a.reindex(
        categories,
        fill_value=0.0
    )


    b = b.reindex(
        categories,
        fill_value=0.0
    )


    return float(
        0.5
        *
        np.abs(
            a
            -
            b
        ).sum()
    )


def weighted_mean(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )


    weights = np.asarray(
        weights,
        dtype=float
    )


    valid = (
        np.isfinite(
            values
        )

        &

        np.isfinite(
            weights
        )

        &

        (
            weights
            >
            0
        )
    )


    if not valid.any():

        return np.nan


    return float(
        np.average(
            values[
                valid
            ],
            weights=weights[
                valid
            ]
        )
    )


# ==================================================================================================
# VERIFY ORIGINAL LSTM
# ==================================================================================================

for path in [
    DEVELOPMENT_LOCK_FILE,
    VAL_INDEX_FILE,
    X_VAL_FILE,
    ELIGIBLE_VAL_FILE,
]:

    require_file(
        path
    )


development_lock = read_json(
    DEVELOPMENT_LOCK_FILE
)


if development_lock.get(
    "Status"
) != "TRAIN_VALIDATION_DEVELOPMENT_COMPLETE":

    raise RuntimeError(
        "Original LSTM development is not complete."
    )


if development_lock.get(
    "TEST_Loaded"
) is not False:

    raise RuntimeError(
        "Development lock indicates TEST was loaded."
    )


if development_lock.get(
    "TEST_Used"
) is not False:

    raise RuntimeError(
        "Development lock indicates TEST was used."
    )


if development_lock.get(
    "TEST_Evaluation_Performed"
) is not False:

    raise RuntimeError(
        "Development lock indicates TEST evaluation occurred."
    )


print(
    "✓ Original LSTM baseline verified"
)

print(
    "✓ TEST was never loaded"
)

print(
    "✓ TEST was never evaluated"
)


# ==================================================================================================
# OUTPUT SAFETY
# ==================================================================================================

if COMPLETE_FILE.is_file():

    print(
        "\n✓ Cell 5C already completed:"
    )

    print(
        OUTPUT_ROOT
    )

    raise SystemExit


if OUTPUT_ROOT.exists():

    attempt = 1


    while (
        OUTPUT_ROOT.parent
        / f"_incomplete_cell5c_attempt_{attempt}"
    ).exists():

        attempt += 1


    backup = (
        OUTPUT_ROOT.parent
        / f"_incomplete_cell5c_attempt_{attempt}"
    )


    shutil.move(
        str(
            OUTPUT_ROOT
        ),
        str(
            backup
        )
    )


    print(
        "Previous incomplete Cell 5C preserved:"
    )

    print(
        backup
    )


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# LOAD VALIDATION INDEX / X
# ==================================================================================================

index_df = pd.read_parquet(
    VAL_INDEX_FILE
)


X_val = np.load(
    X_VAL_FILE,
    mmap_mode="r"
)


eligible_val = np.load(
    ELIGIBLE_VAL_FILE,
    allow_pickle=False
)


if len(
    index_df
) != X_val.shape[
    0
]:

    raise RuntimeError(
        "Validation index/X alignment failure."
    )


if X_val.shape[
    1:
] != (
    10,
    14
):

    raise RuntimeError(
        f"Unexpected W10 validation X shape: {X_val.shape}"
    )


required_columns = [
    "Episode_ID",
    "Minute_Index",
    "Episode_Attack_Family",
    "Calibration_Source",
    "Precursor_Regime",
    "Minutes_To_Next_Onset",
]


missing = [
    c

    for c
    in required_columns

    if c not in index_df.columns
]


if missing:

    raise RuntimeError(
        f"Required metadata missing: {missing}"
    )


index_df = index_df.copy()


index_df[
    "Episode_ID"
] = index_df[
    "Episode_ID"
].astype(
    str
)


index_df[
    "Episode_Attack_Family"
] = normalize_text(

    index_df[
        "Episode_Attack_Family"
    ]
)


index_df[
    "Precursor_Regime"
] = normalize_text(

    index_df[
        "Precursor_Regime"
    ]
)


index_df[
    "Calibration_Source_Normalized"
] = (

    index_df[
        "Calibration_Source"
    ]

    .map(
        normalize_source
    )
)


index_df[
    "Minutes_To_Next_Onset"
] = pd.to_numeric(

    index_df[
        "Minutes_To_Next_Onset"
    ],

    errors="coerce"
)


# ==================================================================================================
# OPTIONAL METADATA
# ==================================================================================================

OPTIONAL_METADATA = []


for candidate in [
    "Precursor_Length_Minutes",
    "Precursor_Max_Alpha",
    "Calibration_Profile",
    "Calibration_Profile_ID",
    "Calibration_Profile_Name",
]:

    if candidate in index_df.columns:

        OPTIONAL_METADATA.append(
            candidate
        )


print("\nOptional metadata found:")

if OPTIONAL_METADATA:

    for c in OPTIONAL_METADATA:

        print(
            "  ✓",
            c
        )

else:

    print(
        "  none"
    )


PROFILE_COLUMN = None


for candidate in [
    "Calibration_Profile",
    "Calibration_Profile_ID",
    "Calibration_Profile_Name",
]:

    if candidate in index_df.columns:

        PROFILE_COLUMN = candidate

        break


# ==================================================================================================
# PART A — EPISODE-LEVEL COMPOSITION
# ==================================================================================================

print("\n" + "=" * 120)
print("PART A — SHORT REGIME EPISODE COMPOSITION")
print("=" * 120)


episode_columns = [
    "Episode_ID",
    "Episode_Attack_Family",
    "Calibration_Source_Normalized",
    "Precursor_Regime",
]


for c in [
    "Precursor_Length_Minutes",
    "Precursor_Max_Alpha",
]:

    if c in index_df.columns:

        episode_columns.append(
            c
        )


if PROFILE_COLUMN is not None:

    episode_columns.append(
        PROFILE_COLUMN
    )


episode_table = (

    index_df.loc[
        index_df[
            "Precursor_Regime"
        ].isin(
            SHORT_REGIMES
        ),
        episode_columns
    ]

    .groupby(
        "Episode_ID",
        as_index=False
    )

    .first()
)


episode_table.to_csv(

    OUTPUT_ROOT
    / "SHORT_REGIME_EPISODE_METADATA.csv",

    index=False
)


print(
    "\nEpisode counts:"
)


print(
    episode_table[
        "Precursor_Regime"
    ]
    .value_counts()
    .to_string()
)


# --------------------------------------------------------------------------------------------------
# Family composition
# --------------------------------------------------------------------------------------------------

family_composition = pd.crosstab(

    episode_table[
        "Precursor_Regime"
    ],

    episode_table[
        "Episode_Attack_Family"
    ]
)


family_composition.to_csv(

    OUTPUT_ROOT
    / "FAMILY_COMPOSITION_COUNTS.csv"
)


print("\nFamily composition:")
print(
    family_composition.to_string()
)


# --------------------------------------------------------------------------------------------------
# Source composition
# --------------------------------------------------------------------------------------------------

source_composition = pd.crosstab(

    episode_table[
        "Precursor_Regime"
    ],

    episode_table[
        "Calibration_Source_Normalized"
    ]
)


source_composition.to_csv(

    OUTPUT_ROOT
    / "SOURCE_COMPOSITION_COUNTS.csv"
)


print("\nSource composition:")
print(
    source_composition.to_string()
)


# --------------------------------------------------------------------------------------------------
# Family × source composition
# --------------------------------------------------------------------------------------------------

family_source_composition = (

    episode_table

    .groupby(
        [
            "Precursor_Regime",
            "Episode_Attack_Family",
            "Calibration_Source_Normalized"
        ],
        as_index=False
    )

    .agg(
        Episodes=(
            "Episode_ID",
            "nunique"
        )
    )
)


family_source_composition.to_csv(

    OUTPUT_ROOT
    / "FAMILY_SOURCE_COMPOSITION.csv",

    index=False
)


print("\nFamily × source composition:")

print(
    family_source_composition.to_string(
        index=False
    )
)


# ==================================================================================================
# LENGTH / ALPHA COMPOSITION
# ==================================================================================================

length_alpha_rows = []


for regime in SHORT_REGIMES:

    subset = episode_table.loc[
        episode_table[
            "Precursor_Regime"
        ]
        ==
        regime
    ]


    row = {
        "Precursor_Regime":
            regime,

        "Episodes":
            int(
                subset[
                    "Episode_ID"
                ].nunique()
            ),
    }


    if "Precursor_Length_Minutes" in subset.columns:

        length_values = pd.to_numeric(

            subset[
                "Precursor_Length_Minutes"
            ],

            errors="coerce"
        )


        row[
            "Length_Mean"
        ] = float(
            length_values.mean()
        )


        row[
            "Length_Median"
        ] = float(
            length_values.median()
        )


        row[
            "Length_Min"
        ] = float(
            length_values.min()
        )


        row[
            "Length_Max"
        ] = float(
            length_values.max()
        )


    if "Precursor_Max_Alpha" in subset.columns:

        alpha_values = pd.to_numeric(

            subset[
                "Precursor_Max_Alpha"
            ],

            errors="coerce"
        )


        row[
            "Alpha_Mean"
        ] = float(
            alpha_values.mean()
        )


        row[
            "Alpha_Median"
        ] = float(
            alpha_values.median()
        )


        row[
            "Alpha_Min"
        ] = float(
            alpha_values.min()
        )


        row[
            "Alpha_Max"
        ] = float(
            alpha_values.max()
        )


    length_alpha_rows.append(
        row
    )


length_alpha_summary = pd.DataFrame(
    length_alpha_rows
)


length_alpha_summary.to_csv(

    OUTPUT_ROOT
    / "LENGTH_ALPHA_SUMMARY.csv",

    index=False
)


print("\nLength / alpha summary:")

print(
    length_alpha_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# PROFILE COMPOSITION IF AVAILABLE
# ==================================================================================================

if PROFILE_COLUMN is not None:

    profile_composition = pd.crosstab(

        episode_table[
            "Precursor_Regime"
        ],

        normalize_text(
            episode_table[
                PROFILE_COLUMN
            ]
        )
    )


    profile_composition.to_csv(

        OUTPUT_ROOT
        / "CALIBRATION_PROFILE_COMPOSITION.csv"
    )


    print(
        f"\nCalibration profile composition ({PROFILE_COLUMN}):"
    )


    print(
        profile_composition.to_string()
    )


# ==================================================================================================
# COMPOSITION DISTANCES
# ==================================================================================================

weak_episodes = episode_table.loc[
    episode_table[
        "Precursor_Regime"
    ]
    ==
    "SHORT_WEAK"
]


strong_episodes = episode_table.loc[
    episode_table[
        "Precursor_Regime"
    ]
    ==
    "SHORT_STRONG"
]


weak_family_dist = (

    weak_episodes[
        "Episode_Attack_Family"
    ]

    .value_counts(
        normalize=True
    )
)


strong_family_dist = (

    strong_episodes[
        "Episode_Attack_Family"
    ]

    .value_counts(
        normalize=True
    )
)


weak_source_dist = (

    weak_episodes[
        "Calibration_Source_Normalized"
    ]

    .value_counts(
        normalize=True
    )
)


strong_source_dist = (

    strong_episodes[
        "Calibration_Source_Normalized"
    ]

    .value_counts(
        normalize=True
    )
)


FAMILY_TVD = total_variation_distance(

    weak_family_dist,

    strong_family_dist
)


SOURCE_TVD = total_variation_distance(

    weak_source_dist,

    strong_source_dist
)


print(
    "\nFamily composition TVD:",
    round(
        FAMILY_TVD,
        6
    )
)


print(
    "Source composition TVD:",
    round(
        SOURCE_TVD,
        6
    )
)


# ==================================================================================================
# PART B — BUILD DESCRIPTIVE TEMPORAL REPRESENTATION
#
# 14 current values
# +
# 14 W10 deltas
# =
# 28 dimensions
# ==================================================================================================

print("\n" + "=" * 120)
print("PART B — EPISODE-LEVEL SIGNAL SEPARATION")
print("=" * 120)


X = np.asarray(
    X_val,
    dtype=np.float32
)


current = X[
    :,
    -1,
    :
]


delta = (

    X[
        :,
        -1,
        :
    ]

    -

    X[
        :,
        0,
        :
    ]
)


representation = np.concatenate(

    [
        current,
        delta
    ],

    axis=1
)


representation_names = (

    [
        f"CURRENT__{f}"
        for f
        in FEATURES
    ]

    +

    [
        f"DELTA_W10__{f}"
        for f
        in FEATURES
    ]
)


episode_ids = index_df[
    "Episode_ID"
].to_numpy()


regimes = index_df[
    "Precursor_Regime"
].to_numpy()


families = index_df[
    "Episode_Attack_Family"
].to_numpy()


sources = index_df[
    "Calibration_Source_Normalized"
].to_numpy()


tto = index_df[
    "Minutes_To_Next_Onset"
].to_numpy(
    dtype=float
)


# ==================================================================================================
# EPISODE SEPARATION
#
# Near:
#       1–3 min before onset
#
# Far:
#       > H+5 min before onset
#
# ==================================================================================================

episode_separation_rows = []


short_episode_ids = episode_table[
    "Episode_ID"
].astype(str).tolist()


for H in HORIZONS:

    h_col = HORIZON_TO_COLUMN[
        H
    ]


    eligible_H = (
        eligible_val[
            :,
            h_col
        ]
        ==
        1
    )


    for episode_id in short_episode_ids:

        ep_mask = (
            episode_ids
            ==
            episode_id
        )


        if not ep_mask.any():

            continue


        first_position = np.flatnonzero(
            ep_mask
        )[0]


        regime = regimes[
            first_position
        ]


        family = families[
            first_position
        ]


        source = sources[
            first_position
        ]


        near_mask = (

            ep_mask

            &

            eligible_H

            &

            np.isfinite(
                tto
            )

            &

            (
                tto
                >
                0
            )

            &

            (
                tto
                <=
                3
            )
        )


        far_mask = (

            ep_mask

            &

            eligible_H

            &

            np.isfinite(
                tto
            )

            &

            (
                tto
                >
                (
                    H
                    +
                    5
                )
            )
        )


        if (
            near_mask.sum()
            ==
            0

            or

            far_mask.sum()
            ==
            0
        ):

            continue


        near_mean = representation[
            near_mask
        ].mean(
            axis=0
        )


        far_mean = representation[
            far_mask
        ].mean(
            axis=0
        )


        difference = (
            near_mean
            -
            far_mean
        )


        row = {
            "Horizon":
                H,

            "Episode_ID":
                episode_id,

            "Precursor_Regime":
                regime,

            "Attack_Family":
                family,

            "Calibration_Source":
                source,

            "Near_Origins":
                int(
                    near_mask.sum()
                ),

            "Far_Origins":
                int(
                    far_mask.sum()
                ),

            "Episode_Separation_L2":
                float(
                    np.linalg.norm(
                        difference
                    )
                ),

            "Episode_Mean_Absolute_Shift":
                float(
                    np.mean(
                        np.abs(
                            difference
                        )
                    )
                ),

            "Episode_Max_Absolute_Shift":
                float(
                    np.max(
                        np.abs(
                            difference
                        )
                    )
                ),
        }


        episode_metadata_row = episode_table.loc[
            episode_table[
                "Episode_ID"
            ]
            ==
            episode_id
        ].iloc[
            0
        ]


        for optional in [
            "Precursor_Length_Minutes",
            "Precursor_Max_Alpha",
        ]:

            if optional in episode_table.columns:

                row[
                    optional
                ] = episode_metadata_row[
                    optional
                ]


        if PROFILE_COLUMN is not None:

            row[
                "Calibration_Profile"
            ] = str(
                episode_metadata_row[
                    PROFILE_COLUMN
                ]
            )


        episode_separation_rows.append(
            row
        )


episode_separation = pd.DataFrame(
    episode_separation_rows
)


episode_separation.to_csv(

    OUTPUT_ROOT
    / "EPISODE_SIGNAL_SEPARATION.csv",

    index=False
)


print(
    "\nEpisode separation records:",
    len(
        episode_separation
    )
)


print(
    episode_separation
    .groupby(
        [
            "Horizon",
            "Precursor_Regime"
        ]
    )[
        "Episode_Separation_L2"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std"
        ]
    )
    .to_string()
)


# ==================================================================================================
# PART C — MATCHED SIGNAL COMPARISON
#
# Compare SHORT_WEAK and SHORT_STRONG only when the same composition
# cell contains both regimes.
# ==================================================================================================

print("\n" + "=" * 120)
print("PART C — MATCHED SIGNAL COMPARISON")
print("=" * 120)


def build_matched_comparison(
    df,
    match_columns,
    metric,
    level_name
):

    rows = []


    group_columns = (
        [
            "Horizon"
        ]
        +
        match_columns
    )


    for group_values, group in df.groupby(
        group_columns,
        dropna=False
    ):

        if not isinstance(
            group_values,
            tuple
        ):

            group_values = (
                group_values,
            )


        weak = group.loc[
            group[
                "Precursor_Regime"
            ]
            ==
            "SHORT_WEAK",
            metric
        ].dropna()


        strong = group.loc[
            group[
                "Precursor_Regime"
            ]
            ==
            "SHORT_STRONG",
            metric
        ].dropna()


        if (
            len(
                weak
            )
            ==
            0

            or

            len(
                strong
            )
            ==
            0
        ):

            continue


        row = {
            "Match_Level":
                level_name,

            "Horizon":
                int(
                    group_values[
                        0
                    ]
                ),
        }


        for index, column in enumerate(
            match_columns,
            start=1
        ):

            row[
                column
            ] = group_values[
                index
            ]


        weak_mean = float(
            weak.mean()
        )


        strong_mean = float(
            strong.mean()
        )


        weak_median = float(
            weak.median()
        )


        strong_median = float(
            strong.median()
        )


        row.update(
            {
                "Weak_N":
                    len(
                        weak
                    ),

                "Strong_N":
                    len(
                        strong
                    ),

                "Matched_Weight":
                    min(
                        len(
                            weak
                        ),
                        len(
                            strong
                        )
                    ),

                "Weak_Mean":
                    weak_mean,

                "Strong_Mean":
                    strong_mean,

                "Strong_Minus_Weak_Mean":
                    strong_mean
                    -
                    weak_mean,

                "Weak_Median":
                    weak_median,

                "Strong_Median":
                    strong_median,

                "Strong_Minus_Weak_Median":
                    strong_median
                    -
                    weak_median,
            }
        )


        rows.append(
            row
        )


    return pd.DataFrame(
        rows
    )


matched_family = build_matched_comparison(

    episode_separation,

    [
        "Attack_Family"
    ],

    "Episode_Separation_L2",

    "FAMILY"
)


matched_source = build_matched_comparison(

    episode_separation,

    [
        "Calibration_Source"
    ],

    "Episode_Separation_L2",

    "SOURCE"
)


matched_family_source = build_matched_comparison(

    episode_separation,

    [
        "Attack_Family",
        "Calibration_Source"
    ],

    "Episode_Separation_L2",

    "FAMILY_SOURCE"
)


matched_signal = pd.concat(

    [
        matched_family,
        matched_source,
        matched_family_source,
    ],

    ignore_index=True,

    sort=False
)


matched_signal.to_csv(

    OUTPUT_ROOT
    / "MATCHED_SIGNAL_COMPARISON.csv",

    index=False
)


print(
    matched_signal.to_string(
        index=False
    )
)


# ==================================================================================================
# OPTIONAL EXACT LENGTH MATCH
# ==================================================================================================

matched_length = pd.DataFrame()


if "Precursor_Length_Minutes" in episode_separation.columns:

    episode_separation[
        "Precursor_Length_Minutes"
    ] = pd.to_numeric(

        episode_separation[
            "Precursor_Length_Minutes"
        ],

        errors="coerce"
    )


    matched_length = build_matched_comparison(

        episode_separation,

        [
            "Attack_Family",
            "Calibration_Source",
            "Precursor_Length_Minutes"
        ],

        "Episode_Separation_L2",

        "FAMILY_SOURCE_LENGTH"
    )


    matched_length.to_csv(

        OUTPUT_ROOT
        / "MATCHED_SIGNAL_BY_FAMILY_SOURCE_LENGTH.csv",

        index=False
    )


    print(
        "\nExact family/source/length matched comparison:"
    )


    print(
        matched_length.to_string(
            index=False
        )
    )


# ==================================================================================================
# PART D — ORIGINAL LSTM PER-EPISODE PERFORMANCE
#
# We now ask:
#
# Does SHORT_WEAK still outperform SHORT_STRONG even within the same
# family/source composition cell?
# ==================================================================================================

print("\n" + "=" * 120)
print("PART D — ORIGINAL LSTM PER-EPISODE PERFORMANCE")
print("=" * 120)


performance_rows = []


metadata_merge = index_df[
    [
        "Episode_ID",
        "Minute_Index",
        "Precursor_Regime",
        "Episode_Attack_Family",
        "Calibration_Source_Normalized",
    ]
].copy()


for H in HORIZONS:

    for seed in FINAL_SEEDS:

        prediction_file = (
            MODEL_ROOT
            / f"H{H:02d}"
            / "final_replicates"
            / f"seed_{seed}"
            / "validation_predictions.parquet"
        )


        require_file(
            prediction_file
        )


        pred = pd.read_parquet(
            prediction_file
        )


        pred[
            "Episode_ID"
        ] = pred[
            "Episode_ID"
        ].astype(
            str
        )


        pred[
            "Minute_Index"
        ] = pd.to_numeric(

            pred[
                "Minute_Index"
            ],

            errors="raise"
        ).astype(
            np.int64
        )


        joined = pred.merge(

            metadata_merge,

            on=[
                "Episode_ID",
                "Minute_Index"
            ],

            how="left",

            validate="one_to_one"
        )


        joined = joined.loc[
            joined[
                "Precursor_Regime"
            ].isin(
                SHORT_REGIMES
            )
        ].copy()


        for episode_id, episode_df in joined.groupby(
            "Episode_ID"
        ):

            y = episode_df[
                "Y_True"
            ].to_numpy(
                dtype=np.int8
            )


            scores = episode_df[
                "Probability"
            ].to_numpy(
                dtype=float
            )


            prediction = episode_df[
                "Prediction"
            ].to_numpy(
                dtype=np.int8
            )


            regime = episode_df[
                "Precursor_Regime"
            ].iloc[
                0
            ]


            family = episode_df[
                "Episode_Attack_Family"
            ].iloc[
                0
            ]


            source = episode_df[
                "Calibration_Source_Normalized"
            ].iloc[
                0
            ]


            performance_rows.append(
                {
                    "Horizon":
                        H,

                    "Seed":
                        seed,

                    "Episode_ID":
                        episode_id,

                    "Precursor_Regime":
                        regime,

                    "Attack_Family":
                        family,

                    "Calibration_Source":
                        source,

                    "Origins":
                        len(
                            episode_df
                        ),

                    "Positive_Origins":
                        int(
                            (
                                y
                                ==
                                1
                            ).sum()
                        ),

                    "AP":
                        safe_ap(
                            y,
                            scores
                        ),

                    "ROC_AUC":
                        safe_roc(
                            y,
                            scores
                        ),

                    "Precision":
                        float(
                            precision_score(
                                y,
                                prediction,
                                zero_division=0
                            )
                        ),

                    "Recall":
                        float(
                            recall_score(
                                y,
                                prediction,
                                zero_division=0
                            )
                        ),

                    "F1":
                        float(
                            f1_score(
                                y,
                                prediction,
                                zero_division=0
                            )
                        ),

                    "Mean_Probability":
                        float(
                            scores.mean()
                        ),

                    "Max_Probability":
                        float(
                            scores.max()
                        ),
                }
            )


episode_performance_seed = pd.DataFrame(
    performance_rows
)


episode_performance_seed.to_csv(

    OUTPUT_ROOT
    / "PER_EPISODE_MODEL_PERFORMANCE_BY_SEED.csv",

    index=False
)


# ==================================================================================================
# Average each episode across the three seeds
# ==================================================================================================

episode_performance = (

    episode_performance_seed

    .groupby(
        [
            "Horizon",
            "Episode_ID",
            "Precursor_Regime",
            "Attack_Family",
            "Calibration_Source",
        ],
        as_index=False
    )

    .agg(
        AP=(
            "AP",
            "mean"
        ),

        ROC_AUC=(
            "ROC_AUC",
            "mean"
        ),

        Precision=(
            "Precision",
            "mean"
        ),

        Recall=(
            "Recall",
            "mean"
        ),

        F1=(
            "F1",
            "mean"
        ),

        Mean_Probability=(
            "Mean_Probability",
            "mean"
        ),

        Max_Probability=(
            "Max_Probability",
            "mean"
        ),
    )
)


episode_performance.to_csv(

    OUTPUT_ROOT
    / "PER_EPISODE_MODEL_PERFORMANCE.csv",

    index=False
)


print(
    "\nPer-episode AP summary:"
)


print(

    episode_performance

    .groupby(
        [
            "Horizon",
            "Precursor_Regime"
        ]
    )[
        "AP"
    ]

    .agg(
        [
            "count",
            "mean",
            "median",
            "std"
        ]
    )

    .to_string()
)


# ==================================================================================================
# PART E — MATCHED MODEL PERFORMANCE
# ==================================================================================================

print("\n" + "=" * 120)
print("PART E — MATCHED LSTM PERFORMANCE")
print("=" * 120)


matched_model_family = build_matched_comparison(

    episode_performance,

    [
        "Attack_Family"
    ],

    "AP",

    "FAMILY"
)


matched_model_source = build_matched_comparison(

    episode_performance,

    [
        "Calibration_Source"
    ],

    "AP",

    "SOURCE"
)


matched_model_family_source = build_matched_comparison(

    episode_performance,

    [
        "Attack_Family",
        "Calibration_Source"
    ],

    "AP",

    "FAMILY_SOURCE"
)


matched_model = pd.concat(

    [
        matched_model_family,
        matched_model_source,
        matched_model_family_source,
    ],

    ignore_index=True,

    sort=False
)


matched_model.to_csv(

    OUTPUT_ROOT
    / "MATCHED_MODEL_AP_COMPARISON.csv",

    index=False
)


print(
    matched_model.to_string(
        index=False
    )
)


# ==================================================================================================
# PART F — FEATURE-LEVEL MATCHED EFFECTS
#
# Within family/source cells, calculate near-vs-far mean shifts.
# Then compare the absolute magnitude of those shifts between weak/strong.
# ==================================================================================================

print("\n" + "=" * 120)
print("PART F — MATCHED FEATURE-EFFECT COMPARISON")
print("=" * 120)


feature_effect_rows = []


for H in HORIZONS:

    h_col = HORIZON_TO_COLUMN[
        H
    ]


    eligible_H = (
        eligible_val[
            :,
            h_col
        ]
        ==
        1
    )


    for family in sorted(
        episode_table[
            "Episode_Attack_Family"
        ].unique()
    ):

        for source in sorted(
            episode_table[
                "Calibration_Source_Normalized"
            ].unique()
        ):

            for regime in SHORT_REGIMES:

                stratum_episode_ids = set(

                    episode_table.loc[
                        (
                            episode_table[
                                "Episode_Attack_Family"
                            ]
                            ==
                            family
                        )

                        &

                        (
                            episode_table[
                                "Calibration_Source_Normalized"
                            ]
                            ==
                            source
                        )

                        &

                        (
                            episode_table[
                                "Precursor_Regime"
                            ]
                            ==
                            regime
                        ),
                        "Episode_ID"
                    ].astype(str)
                )


                if len(
                    stratum_episode_ids
                ) == 0:

                    continue


                stratum_mask = np.isin(

                    episode_ids,

                    list(
                        stratum_episode_ids
                    )
                )


                near_mask = (

                    stratum_mask

                    &

                    eligible_H

                    &

                    np.isfinite(
                        tto
                    )

                    &

                    (
                        tto
                        >
                        0
                    )

                    &

                    (
                        tto
                        <=
                        3
                    )
                )


                far_mask = (

                    stratum_mask

                    &

                    eligible_H

                    &

                    np.isfinite(
                        tto
                    )

                    &

                    (
                        tto
                        >
                        (
                            H
                            +
                            5
                        )
                    )
                )


                if (
                    near_mask.sum()
                    ==
                    0

                    or

                    far_mask.sum()
                    ==
                    0
                ):

                    continue


                near_mean = representation[
                    near_mask
                ].mean(
                    axis=0
                )


                far_mean = representation[
                    far_mask
                ].mean(
                    axis=0
                )


                difference = (
                    near_mean
                    -
                    far_mean
                )


                for feature_index, feature_name in enumerate(
                    representation_names
                ):

                    feature_effect_rows.append(
                        {
                            "Horizon":
                                H,

                            "Attack_Family":
                                family,

                            "Calibration_Source":
                                source,

                            "Precursor_Regime":
                                regime,

                            "Episodes":
                                len(
                                    stratum_episode_ids
                                ),

                            "Feature":
                                feature_name,

                            "Mean_Shift":
                                float(
                                    difference[
                                        feature_index
                                    ]
                                ),

                            "Absolute_Mean_Shift":
                                float(
                                    abs(
                                        difference[
                                            feature_index
                                        ]
                                    )
                                ),
                        }
                    )


feature_effects = pd.DataFrame(
    feature_effect_rows
)


feature_effects.to_csv(

    OUTPUT_ROOT
    / "MATCHED_FEATURE_EFFECTS.csv",

    index=False
)


# ==================================================================================================
# Compare strong vs weak feature effect magnitude within same H/family/source
# ==================================================================================================

feature_match_rows = []


for (
    H,
    family,
    source,
    feature
), group in feature_effects.groupby(
    [
        "Horizon",
        "Attack_Family",
        "Calibration_Source",
        "Feature"
    ]
):

    weak = group.loc[
        group[
            "Precursor_Regime"
        ]
        ==
        "SHORT_WEAK",
        "Absolute_Mean_Shift"
    ]


    strong = group.loc[
        group[
            "Precursor_Regime"
        ]
        ==
        "SHORT_STRONG",
        "Absolute_Mean_Shift"
    ]


    if (
        len(
            weak
        ) == 0

        or

        len(
            strong
        ) == 0
    ):

        continue


    feature_match_rows.append(
        {
            "Horizon":
                H,

            "Attack_Family":
                family,

            "Calibration_Source":
                source,

            "Feature":
                feature,

            "Weak_Absolute_Shift":
                float(
                    weak.iloc[
                        0
                    ]
                ),

            "Strong_Absolute_Shift":
                float(
                    strong.iloc[
                        0
                    ]
                ),

            "Strong_Minus_Weak":
                float(
                    strong.iloc[
                        0
                    ]
                    -
                    weak.iloc[
                        0
                    ]
                ),
        }
    )


feature_match = pd.DataFrame(
    feature_match_rows
)


feature_match.to_csv(

    OUTPUT_ROOT
    / "MATCHED_FEATURE_STRONG_MINUS_WEAK.csv",

    index=False
)


feature_direction_summary = (

    feature_match

    .groupby(
        "Horizon",
        as_index=False
    )

    .agg(
        Matched_Feature_Comparisons=(
            "Strong_Minus_Weak",
            "count"
        ),

        Mean_Strong_Minus_Weak=(
            "Strong_Minus_Weak",
            "mean"
        ),

        Median_Strong_Minus_Weak=(
            "Strong_Minus_Weak",
            "median"
        ),

        Fraction_Stronger_In_STRONG=(
            "Strong_Minus_Weak",
            lambda x:
                float(
                    (
                        x
                        >
                        0
                    ).mean()
                )
        ),
    )
)


print(
    feature_direction_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# PART G — AUTOMATIC INTERPRETATION
# ==================================================================================================

print("\n" + "=" * 120)
print("PART G — AUTOMATIC CELL 5C FINDINGS")
print("=" * 120)


findings = {
    "Family_Composition_TVD":
        FAMILY_TVD,

    "Source_Composition_TVD":
        SOURCE_TVD,

    "Horizons":
        {},
}


# ==================================================================================================
# Alpha sanity check
# ==================================================================================================

if "Precursor_Max_Alpha" in episode_table.columns:

    alpha_by_regime = (

        episode_table

        .groupby(
            "Precursor_Regime"
        )[
            "Precursor_Max_Alpha"
        ]

        .mean()
    )


    weak_alpha = float(
        alpha_by_regime[
            "SHORT_WEAK"
        ]
    )


    strong_alpha = float(
        alpha_by_regime[
            "SHORT_STRONG"
        ]
    )


    findings[
        "SHORT_WEAK_Alpha_Mean"
    ] = weak_alpha


    findings[
        "SHORT_STRONG_Alpha_Mean"
    ] = strong_alpha


    findings[
        "Strong_Alpha_Greater"
    ] = bool(
        strong_alpha
        >
        weak_alpha
    )


    print(
        "\nNominal precursor alpha:"
    )


    print(
        f"SHORT_WEAK   = {weak_alpha:.4f}"
    )


    print(
        f"SHORT_STRONG = {strong_alpha:.4f}"
    )


    print(
        "Strong alpha > weak alpha:",
        strong_alpha
        >
        weak_alpha
    )


# ==================================================================================================
# Horizon-level matched conclusions
# ==================================================================================================

for H in HORIZONS:

    findings[
        "Horizons"
    ][
        str(
            H
        )
    ] = {}


    # ----------------------------------------------------------------------------------------------
    # SIGNAL
    # ------------------------------------------------------------------------------------------

    fs = matched_family_source.loc[
        matched_family_source[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        fs
    ):

        matched_signal_difference = weighted_mean(

            fs[
                "Strong_Minus_Weak_Mean"
            ],

            fs[
                "Matched_Weight"
            ]
        )


        matched_signal_fraction_strong_better = float(

            (
                fs[
                    "Strong_Minus_Weak_Mean"
                ]
                >
                0
            ).mean()
        )


    else:

        matched_signal_difference = np.nan

        matched_signal_fraction_strong_better = np.nan


    # ----------------------------------------------------------------------------------------------
    # MODEL AP
    # ------------------------------------------------------------------------------------------

    fm = matched_model_family_source.loc[
        matched_model_family_source[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        fm
    ):

        matched_ap_difference = weighted_mean(

            fm[
                "Strong_Minus_Weak_Mean"
            ],

            fm[
                "Matched_Weight"
            ]
        )


        matched_ap_fraction_strong_better = float(

            (
                fm[
                    "Strong_Minus_Weak_Mean"
                ]
                >
                0
            ).mean()
        )


    else:

        matched_ap_difference = np.nan

        matched_ap_fraction_strong_better = np.nan


    findings[
        "Horizons"
    ][
        str(
            H
        )
    ] = {
        "Matched_Family_Source_Signal_Strong_Minus_Weak":
            (
                None

                if not np.isfinite(
                    matched_signal_difference
                )

                else float(
                    matched_signal_difference
                )
            ),

        "Matched_Cells_With_Stronger_STRONG_Signal_Fraction":
            (
                None

                if not np.isfinite(
                    matched_signal_fraction_strong_better
                )

                else float(
                    matched_signal_fraction_strong_better
                )
            ),

        "Matched_Family_Source_AP_Strong_Minus_Weak":
            (
                None

                if not np.isfinite(
                    matched_ap_difference
                )

                else float(
                    matched_ap_difference
                )
            ),

        "Matched_Cells_With_Higher_STRONG_AP_Fraction":
            (
                None

                if not np.isfinite(
                    matched_ap_fraction_strong_better
                )

                else float(
                    matched_ap_fraction_strong_better
                )
            ),
    }


    print(
        f"\nH{H}"
    )


    print(
        "Matched family/source signal "
        "STRONG - WEAK:",
        matched_signal_difference
    )


    print(
        "Fraction matched cells where STRONG signal > WEAK:",
        matched_signal_fraction_strong_better
    )


    print(
        "Matched family/source model AP "
        "STRONG - WEAK:",
        matched_ap_difference
    )


    print(
        "Fraction matched cells where STRONG AP > WEAK:",
        matched_ap_fraction_strong_better
    )


# ==================================================================================================
# Main interpretation
# ==================================================================================================

print("\n" + "-" * 120)


if (
    FAMILY_TVD
    <
    0.10

    and

    SOURCE_TVD
    <
    0.10
):

    print(
        "✓ SHORT_WEAK and SHORT_STRONG have broadly similar pooled "
        "family/source composition."
    )

else:

    print(
        "⚠ SHORT_WEAK and SHORT_STRONG differ materially in pooled "
        "family/source composition."
    )


for H in HORIZONS:

    h_result = findings[
        "Horizons"
    ][
        str(
            H
        )
    ]


    signal_difference = h_result[
        "Matched_Family_Source_Signal_Strong_Minus_Weak"
    ]


    if signal_difference is not None:

        if signal_difference > 0:

            print(
                f"✓ H{H}: after matching family/source, SHORT_STRONG "
                f"has stronger mean signal separation."
            )

        else:

            print(
                f"⚠ H{H}: even after matching family/source, "
                f"SHORT_STRONG still does NOT have stronger observed separation."
            )


write_json(

    findings,

    OUTPUT_ROOT
    / "CELL5C_AUTOMATIC_FINDINGS.json"
)


# ==================================================================================================
# PART H — PLOTS
# ==================================================================================================

# --------------------------------------------------------------------------------------------------
# Episode separation distributions
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = episode_separation.loc[
        episode_separation[
            "Horizon"
        ]
        ==
        H
    ]


    weak_values = plot_df.loc[
        plot_df[
            "Precursor_Regime"
        ]
        ==
        "SHORT_WEAK",
        "Episode_Separation_L2"
    ].to_numpy()


    strong_values = plot_df.loc[
        plot_df[
            "Precursor_Regime"
        ]
        ==
        "SHORT_STRONG",
        "Episode_Separation_L2"
    ].to_numpy()


    plt.figure(
        figsize=(
            7,
            5
        )
    )


    plt.boxplot(
        [
            weak_values,
            strong_values,
        ],
        labels=[
            "SHORT_WEAK",
            "SHORT_STRONG"
        ]
    )


    plt.ylabel(
        "Episode-Level Signal Separation"
    )


    plt.title(
        f"W10/H{H} — Episode Separation Distribution"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_episode_separation_boxplot.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# --------------------------------------------------------------------------------------------------
# Per-episode LSTM AP
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = episode_performance.loc[
        episode_performance[
            "Horizon"
        ]
        ==
        H
    ]


    weak_values = plot_df.loc[
        plot_df[
            "Precursor_Regime"
        ]
        ==
        "SHORT_WEAK",
        "AP"
    ].dropna().to_numpy()


    strong_values = plot_df.loc[
        plot_df[
            "Precursor_Regime"
        ]
        ==
        "SHORT_STRONG",
        "AP"
    ].dropna().to_numpy()


    plt.figure(
        figsize=(
            7,
            5
        )
    )


    plt.boxplot(
        [
            weak_values,
            strong_values,
        ],
        labels=[
            "SHORT_WEAK",
            "SHORT_STRONG"
        ]
    )


    plt.ylabel(
        "Per-Episode Average Precision"
    )


    plt.title(
        f"LSTM W10/H{H} — Per-Episode AP"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_per_episode_AP_boxplot.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# --------------------------------------------------------------------------------------------------
# Matched family/source signal differences
# --------------------------------------------------------------------------------------------------

for H in HORIZONS:

    plot_df = matched_family_source.loc[
        matched_family_source[
            "Horizon"
        ]
        ==
        H
    ].copy()


    if len(
        plot_df
    ) == 0:

        continue


    plot_df[
        "Cell"
    ] = (

        plot_df[
            "Attack_Family"
        ].astype(str)

        +

        " | "

        +

        plot_df[
            "Calibration_Source"
        ].astype(str)
    )


    plt.figure(
        figsize=(
            10,
            5
        )
    )


    plt.bar(

        plot_df[
            "Cell"
        ],

        plot_df[
            "Strong_Minus_Weak_Mean"
        ]
    )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.ylabel(
        "Signal Separation: STRONG - WEAK"
    )


    plt.title(
        f"W10/H{H} — Matched Family/Source Signal Difference"
    )


    plt.xticks(
        rotation=35,
        ha="right"
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_ROOT
        / f"H{H:02d}_matched_signal_difference.png",

        dpi=170,

        bbox_inches="tight"
    )


    plt.show()

    plt.close()


# ==================================================================================================
# COMPLETE
# ==================================================================================================

write_json(
    {
        "Cell":
            "5C",

        "Analysis":
            "SHORT_WEAK_VS_SHORT_STRONG_MATCHED_COMPOSITION_AUDIT",

        "Evaluation_Split":
            "VALIDATION",

        "Training_Performed":
            False,

        "Hyperparameter_Tuning_Performed":
            False,

        "Threshold_Changes":
            False,

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "Family_Composition_TVD":
            FAMILY_TVD,

        "Source_Composition_TVD":
            SOURCE_TVD,

        "Outputs": [
            "SHORT_REGIME_EPISODE_METADATA.csv",
            "FAMILY_COMPOSITION_COUNTS.csv",
            "SOURCE_COMPOSITION_COUNTS.csv",
            "FAMILY_SOURCE_COMPOSITION.csv",
            "LENGTH_ALPHA_SUMMARY.csv",
            "EPISODE_SIGNAL_SEPARATION.csv",
            "MATCHED_SIGNAL_COMPARISON.csv",
            "PER_EPISODE_MODEL_PERFORMANCE.csv",
            "MATCHED_MODEL_AP_COMPARISON.csv",
            "MATCHED_FEATURE_EFFECTS.csv",
            "CELL5C_AUTOMATIC_FINDINGS.json",
        ],

        "Next":
            "CLEAN_EPISODE_LEVEL_CV_LSTM_OPTIMIZATION",
    },

    COMPLETE_FILE
)


print("\n" + "=" * 120)
print("✓ CELL 5C COMPLETE")
print("=" * 120)


print(
    "\nSaved to:"
)

print(
    OUTPUT_ROOT
)


print(
    "\nNO TRAINING ✓"
)

print(
    "NO HYPERPARAMETER TUNING ✓"
)

print(
    "NO THRESHOLD CHANGES ✓"
)

print(
    "NO TEST DATA ✓"
)


print(
    "\nNEXT:"
)

print(
    "Clean episode-level CV + fold-specific preprocessing + serious LSTM optimization."
)


print("=" * 120)


gc.collect()

In [ ]:
# ==================================================================================================
# CELL 6A — CLEAN LSTM OPTIMIZATION INFRASTRUCTURE
#
# IMPORTANT:
#   - NO TEST PATH EXISTS IN THIS CELL.
#   - INTERNAL CV IS SPLIT BY WHOLE Episode_ID.
#   - EACH CV FOLD FITS ITS OWN log1p + StandardScaler USING ONLY THAT FOLD'S TRAIN EPISODES.
#   - ORIGINAL LSTM IS NEVER OVERWRITTEN.
#
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import random
import gc
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    precision_recall_curve,
    confusion_matrix,
    auc,
)

warnings.filterwarnings("ignore", category=FutureWarning)


# ==================================================================================================
# INSTALL OPTUNA IF NEEDED
# ==================================================================================================

try:
    import optuna
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "optuna",
        ]
    )
    import optuna


print("=" * 120)
print("CELL 6A — CLEAN EPISODE-CV LSTM OPTIMIZATION INFRASTRUCTURE")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = ROOT / "model_ready" / "12_0"

P12_2 = ROOT / "model_ready" / "12_2"

P12_3 = ROOT / "model_ready" / "12_3"

BASELINE_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
)

BASELINE_SUMMARY_FILE = (
    BASELINE_ROOT
    / "validation_diagnostics"
    / "LSTM_FULL_VALIDATION_SUMMARY.csv"
)

OPT_ROOT = (
    ROOT
    / "models"
    / "LSTM_W10_OPT_V2_CLEAN_CV"
)

OPT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# Phase 12.3 controls
# --------------------------------------------------------------------------------------------------

READY_12_3_FILE = (
    P12_3
    / "12_3_READY_FOR_MODELING.json"
)

LOCK_12_3_FILE = (
    P12_3
    / "12_3_PROTOCOL_LOCK.json"
)

LOCK_HASH_12_3_FILE = (
    P12_3
    / "12_3_PROTOCOL_LOCK_SHA256.txt"
)


# --------------------------------------------------------------------------------------------------
# TRAIN — RAW
#
# Deliberately use RAW Phase 12.0 data for CV.
# Each fold gets its own scaler.
# --------------------------------------------------------------------------------------------------

RAW_X_TRAIN_FILE = (
    P12_0
    / "W10"
    / "train"
    / "X.npy"
)

Y_TRAIN_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)

ELIGIBLE_TRAIN_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)

TRAIN_INDEX_FILE = (
    P12_0
    / "W10"
    / "train"
    / "index.parquet"
)


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

FEATURE_COUNT = 14

WINDOW = 10

HORIZONS = [
    5,
    10,
]

HORIZON_TO_COLUMN = {
    1: 0,
    5: 1,
    10: 2,
    15: 3,
    30: 4,
}

CV_FOLDS_COUNT = 3

CV_SPLIT_SEED = 20261022

HPO_SEED = 20260934

DEVELOPMENT_SEED = 20260930

FINAL_SEEDS = [
    20260930,
    20260931,
    20260932,
]

HPO_TRIALS_PER_HORIZON = 12

HPO_MAX_EPOCHS = 35

HPO_PATIENCE = 5

FINAL_MAX_EPOCHS = 60

FINAL_PATIENCE = 8

MIN_DELTA = 0.0001

GRAD_CLIP = 1.0

AP_TIE_TOLERANCE = 0.002

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = (
    DEVICE.type
    ==
    "cuda"
)


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required artifact:\n{path}"
        )


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def read_json(path):

    require_file(path)

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ==================================================================================================
# VERIFY 12.3 / TEST EMBARGO
# ==================================================================================================

for path in [
    READY_12_3_FILE,
    LOCK_12_3_FILE,
    LOCK_HASH_12_3_FILE,
    RAW_X_TRAIN_FILE,
    Y_TRAIN_FILE,
    ELIGIBLE_TRAIN_FILE,
    TRAIN_INDEX_FILE,
    BASELINE_SUMMARY_FILE,
]:

    require_file(path)


ready_12_3 = read_json(
    READY_12_3_FILE
)

protocol_hash = sha256_file(
    LOCK_12_3_FILE
)

recorded_protocol_hash = (
    LOCK_HASH_12_3_FILE
    .read_text(
        encoding="utf-8"
    )
    .strip()
)


if protocol_hash != recorded_protocol_hash:

    raise RuntimeError(
        "Phase 12.3 protocol lock mismatch."
    )


if ready_12_3.get(
    "phase_12_3_status"
) != "PASS":

    raise RuntimeError(
        "Phase 12.3 is not PASS."
    )


if ready_12_3.get(
    "training_authorized"
) is not True:

    raise RuntimeError(
        "Training is not authorized."
    )


if ready_12_3.get(
    "test_evaluation_authorized"
) is not False:

    raise RuntimeError(
        "Unexpected TEST authorization."
    )


print("✓ Phase 12.3 verified")
print("✓ TRAIN/VALIDATION modeling authorized")
print("✓ TEST remains embargoed")
print("✓ No TEST path is defined in this optimization pipeline")


# ==================================================================================================
# DEVICE
# ==================================================================================================

print(
    "\nDevice:",
    DEVICE
)

if DEVICE.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "Mixed precision:",
    AMP_ENABLED
)


# ==================================================================================================
# LOAD RAW TRAIN DATA
# ==================================================================================================

X_RAW = np.load(
    RAW_X_TRAIN_FILE,
    mmap_mode="r"
)

Y_ALL = np.load(
    Y_TRAIN_FILE,
    allow_pickle=False
)

ELIGIBLE_ALL = np.load(
    ELIGIBLE_TRAIN_FILE,
    allow_pickle=False
)

INDEX_ALL = pd.read_parquet(
    TRAIN_INDEX_FILE
)


if X_RAW.shape != (
    59837,
    10,
    14
):

    raise RuntimeError(
        f"Unexpected raw TRAIN shape: {X_RAW.shape}"
    )


if len(INDEX_ALL) != len(X_RAW):

    raise RuntimeError(
        "TRAIN index/X alignment failed."
    )


if (
    X_RAW.min()
    <
    0
):

    raise RuntimeError(
        "Raw Phase 12.0 X unexpectedly contains negative values."
    )


INDEX_ALL = INDEX_ALL.copy()

INDEX_ALL[
    "Episode_ID"
] = (
    INDEX_ALL[
        "Episode_ID"
    ]
    .astype(str)
)

INDEX_ALL[
    "Minute_Index"
] = pd.to_numeric(
    INDEX_ALL[
        "Minute_Index"
    ],
    errors="raise"
).astype(
    np.int64
)


EPISODE_IDS = INDEX_ALL[
    "Episode_ID"
].to_numpy()

MINUTE_INDEX = INDEX_ALL[
    "Minute_Index"
].to_numpy(
    dtype=np.int64
)


# ==================================================================================================
# EPISODE TABLE FOR STRATIFIED CV
# ==================================================================================================

for column in [
    "Episode_Type",
    "Episode_Attack_Family",
    "Calibration_Source",
]:

    if column not in INDEX_ALL.columns:

        INDEX_ALL[
            column
        ] = "UNKNOWN"

    INDEX_ALL[
        column
    ] = (
        INDEX_ALL[
            column
        ]
        .astype("string")
        .fillna("NONE")
        .astype(str)
    )


episode_table = (

    INDEX_ALL[
        [
            "Episode_ID",
            "Episode_Type",
            "Episode_Attack_Family",
            "Calibration_Source",
        ]
    ]

    .groupby(
        "Episode_ID",
        as_index=False
    )

    .first()
)


episode_table[
    "Stratum"
] = (

    episode_table[
        "Episode_Type"
    ]

    +

    " | "

    +

    episode_table[
        "Episode_Attack_Family"
    ]

    +

    " | "

    +

    episode_table[
        "Calibration_Source"
    ]
)


stratum_counts = episode_table[
    "Stratum"
].value_counts()


if (
    stratum_counts
    <
    CV_FOLDS_COUNT
).any():

    print(
        "Some family/source strata have fewer than "
        f"{CV_FOLDS_COUNT} episodes."
    )

    print(
        "Falling back to Episode_Type + family stratification."
    )

    episode_table[
        "Stratum"
    ] = (

        episode_table[
            "Episode_Type"
        ]

        +

        " | "

        +

        episode_table[
            "Episode_Attack_Family"
        ]
    )


print(
    "\nIndependent TRAIN episodes:",
    len(
        episode_table
    )
)


# ==================================================================================================
# CREATE 3 EPISODE-LEVEL FOLDS
# ==================================================================================================

skf = StratifiedKFold(
    n_splits=CV_FOLDS_COUNT,
    shuffle=True,
    random_state=CV_SPLIT_SEED,
)


CV_EPISODE_SPLITS = []


for fold_id, (
    train_episode_indices,
    dev_episode_indices,
) in enumerate(
    skf.split(
        episode_table[
            "Episode_ID"
        ],
        episode_table[
            "Stratum"
        ]
    ),
    start=1
):

    train_episodes = set(
        episode_table.iloc[
            train_episode_indices
        ][
            "Episode_ID"
        ].astype(str)
    )

    dev_episodes = set(
        episode_table.iloc[
            dev_episode_indices
        ][
            "Episode_ID"
        ].astype(str)
    )


    overlap = (
        train_episodes
        &
        dev_episodes
    )


    if overlap:

        raise RuntimeError(
            f"Fold {fold_id}: episode overlap detected."
        )


    CV_EPISODE_SPLITS.append(
        {
            "fold":
                fold_id,

            "train_episodes":
                train_episodes,

            "dev_episodes":
                dev_episodes,
        }
    )


    print(
        f"Fold {fold_id}: "
        f"TRAIN episodes={len(train_episodes)}, "
        f"DEV episodes={len(dev_episodes)}, "
        f"overlap={len(overlap)}"
    )


# ==================================================================================================
# SAVE CV EPISODE ASSIGNMENTS
# ==================================================================================================

cv_assignment_rows = []


for split in CV_EPISODE_SPLITS:

    fold_id = split[
        "fold"
    ]


    for episode_id in split[
        "dev_episodes"
    ]:

        cv_assignment_rows.append(
            {
                "Fold":
                    fold_id,

                "Episode_ID":
                    episode_id,

                "Role":
                    "DEV",
            }
        )


pd.DataFrame(
    cv_assignment_rows
).to_csv(

    OPT_ROOT
    / "CV_EPISODE_ASSIGNMENTS.csv",

    index=False
)


# ==================================================================================================
# FOLD-SPECIFIC SCALER
#
# IMPORTANT:
#
# The scaler fit set uses all sequence-covered predictor minutes from
# FOLD-TRAIN EPISODES ONLY.
#
# No label values and no fold-DEV episode contributes to scaler fitting.
#
# Overlapping sequence minutes are deduplicated before fitting.
# ==================================================================================================

EPISODE_CODES, _ = pd.factorize(
    EPISODE_IDS
)

EPISODE_CODES = EPISODE_CODES.astype(
    np.int64
)

KEY_OFFSET = int(
    max(
        MINUTE_INDEX.max()
        +
        100,
        10000
    )
)


def get_unique_raw_minutes_for_episode_mask(
    episode_mask
):

    sample_indices = np.flatnonzero(
        episode_mask
    )


    if len(sample_indices) == 0:

        raise RuntimeError(
            "No samples available for scaler fitting."
        )


    X_subset = np.asarray(
        X_RAW[
            sample_indices
        ],
        dtype=np.float32
    )


    episode_code_subset = EPISODE_CODES[
        sample_indices
    ]


    endpoint_subset = MINUTE_INDEX[
        sample_indices
    ]


    history_offsets = np.arange(
        WINDOW - 1,
        -1,
        -1,
        dtype=np.int64
    )


    history_minutes = (

        endpoint_subset[
            :,
            None
        ]

        -

        history_offsets[
            None,
            :
        ]
    )


    key_matrix = (

        episode_code_subset[
            :,
            None
        ]

        *
        KEY_OFFSET

        +

        history_minutes
    )


    flat_keys = key_matrix.reshape(
        -1
    )


    flat_values = X_subset.reshape(
        -1,
        FEATURE_COUNT
    )


    _, first_indices = np.unique(
        flat_keys,
        return_index=True
    )


    unique_values = flat_values[
        first_indices
    ]


    return unique_values


def fit_fold_scaler(
    train_episode_set
):

    train_episode_mask = np.isin(
        EPISODE_IDS,
        list(
            train_episode_set
        )
    )


    unique_raw_minutes = (
        get_unique_raw_minutes_for_episode_mask(
            train_episode_mask
        )
    )


    if not np.isfinite(
        unique_raw_minutes
    ).all():

        raise RuntimeError(
            "Non-finite raw value in fold scaler fit."
        )


    log_minutes = np.log1p(
        unique_raw_minutes
    ).astype(
        np.float64
    )


    scaler = StandardScaler()

    scaler.fit(
        log_minutes
    )


    return (
        scaler,
        len(
            unique_raw_minutes
        )
    )


def transform_full_raw_tensor(
    scaler
):

    transformed = np.log1p(
        np.asarray(
            X_RAW,
            dtype=np.float32
        )
    )


    flat = transformed.reshape(
        -1,
        FEATURE_COUNT
    )


    flat = scaler.transform(
        flat
    ).astype(
        np.float32
    )


    transformed = flat.reshape(
        X_RAW.shape
    )


    if not np.isfinite(
        transformed
    ).all():

        raise RuntimeError(
            "Non-finite transformed CV data."
        )


    return transformed


# ==================================================================================================
# BUILD FOLD-SPECIFIC TRANSFORMED ARRAYS ONCE
# ==================================================================================================

print("\n" + "=" * 120)
print("BUILDING CLEAN FOLD-SPECIFIC PREPROCESSING")
print("=" * 120)


CV_CACHE = []


for split in CV_EPISODE_SPLITS:

    fold_id = split[
        "fold"
    ]


    scaler, scaler_fit_minutes = fit_fold_scaler(
        split[
            "train_episodes"
        ]
    )


    X_scaled = transform_full_raw_tensor(
        scaler
    )


    # ----------------------------------------------------------------------------------------------
    # Hard leakage check:
    # no DEV episode may be part of scaler-fit episode set
    # ----------------------------------------------------------------------------------------------

    if split[
        "train_episodes"
    ].intersection(
        split[
            "dev_episodes"
        ]
    ):

        raise RuntimeError(
            f"Fold {fold_id}: scaler leakage."
        )


    CV_CACHE.append(
        {
            "fold":
                fold_id,

            "train_episodes":
                split[
                    "train_episodes"
                ],

            "dev_episodes":
                split[
                    "dev_episodes"
                ],

            "X_scaled":
                X_scaled,

            "scaler_mean":
                scaler.mean_.copy(),

            "scaler_scale":
                scaler.scale_.copy(),

            "scaler_fit_unique_minutes":
                scaler_fit_minutes,
        }
    )


    np.savez(
        OPT_ROOT
        / f"FOLD_{fold_id}_SCALER.npz",

        mean=scaler.mean_,

        scale=scaler.scale_,

        var=scaler.var_,

        n_features_in=np.array(
            [
                scaler.n_features_in_
            ]
        ),

        fit_unique_minutes=np.array(
            [
                scaler_fit_minutes
            ]
        ),
    )


    print(
        f"Fold {fold_id}: "
        f"scaler fit on {scaler_fit_minutes:,} unique fold-TRAIN minutes only"
    )


print(
    "✓ Fold-specific preprocessing complete"
)


# ==================================================================================================
# EPISODE-BALANCED SAMPLER
# ==================================================================================================

def episode_balanced_weights(
    y,
    episodes
):

    y = np.asarray(
        y,
        dtype=np.int8
    )

    episodes = np.asarray(
        episodes
    )


    weights = np.zeros(
        len(y),
        dtype=np.float64
    )


    for class_value in [
        0,
        1,
    ]:

        class_mask = (
            y
            ==
            class_value
        )


        class_episode_series = pd.Series(
            episodes[
                class_mask
            ]
        )


        counts = class_episode_series.value_counts()

        n_episodes = len(
            counts
        )


        if n_episodes == 0:

            raise RuntimeError(
                "Episode-balanced sampler found empty class."
            )


        per_origin_episode_count = (
            class_episode_series
            .map(
                counts
            )
            .to_numpy(
                dtype=np.float64
            )
        )


        weights[
            class_mask
        ] = (

            0.5

            /

            (
                n_episodes
                *
                per_origin_episode_count
            )
        )


    weights /= weights.sum()


    return weights


# ==================================================================================================
# LOSSES
# ==================================================================================================

class BinaryFocalLoss(
    nn.Module
):

    def __init__(
        self,
        gamma=2.0,
        alpha=0.5
    ):

        super().__init__()

        self.gamma = float(
            gamma
        )

        self.alpha = float(
            alpha
        )


    def forward(
        self,
        logits,
        targets
    ):

        targets = targets.float()


        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )


        probabilities = torch.sigmoid(
            logits
        )


        pt = (
            probabilities
            *
            targets

            +

            (
                1.0
                -
                probabilities
            )

            *
            (
                1.0
                -
                targets
            )
        )


        alpha_t = (
            self.alpha
            *
            targets

            +

            (
                1.0
                -
                self.alpha
            )

            *
            (
                1.0
                -
                targets
            )
        )


        focal_factor = (
            1.0
            -
            pt
        ) ** self.gamma


        loss = (
            alpha_t
            *
            focal_factor
            *
            bce
        )


        return loss.mean()


class ClassBalancedFocalLoss(
    nn.Module
):

    def __init__(
        self,
        n_negative,
        n_positive,
        beta=0.999,
        gamma=2.0
    ):

        super().__init__()


        self.gamma = float(
            gamma
        )


        effective_neg = (
            1.0
            -
            beta ** float(
                n_negative
            )
        )


        effective_pos = (
            1.0
            -
            beta ** float(
                n_positive
            )
        )


        weight_neg = (
            1.0
            -
            beta
        ) / max(
            effective_neg,
            1e-12
        )


        weight_pos = (
            1.0
            -
            beta
        ) / max(
            effective_pos,
            1e-12
        )


        normalization = (
            weight_neg
            +
            weight_pos
        )


        weight_neg = (
            2.0
            *
            weight_neg
            /
            normalization
        )


        weight_pos = (
            2.0
            *
            weight_pos
            /
            normalization
        )


        self.register_buffer(
            "weight_neg",
            torch.tensor(
                weight_neg,
                dtype=torch.float32
            )
        )


        self.register_buffer(
            "weight_pos",
            torch.tensor(
                weight_pos,
                dtype=torch.float32
            )
        )


    def forward(
        self,
        logits,
        targets
    ):

        targets = targets.float()


        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )


        probabilities = torch.sigmoid(
            logits
        )


        pt = (
            probabilities
            *
            targets

            +

            (
                1.0
                -
                probabilities
            )

            *
            (
                1.0
                -
                targets
            )
        )


        class_weight = (
            self.weight_pos
            *
            targets

            +

            self.weight_neg
            *
            (
                1.0
                -
                targets
            )
        )


        loss = (
            class_weight
            *
            (
                1.0
                -
                pt
            ) ** self.gamma
            *
            bce
        )


        return loss.mean()


def build_loss(
    config,
    y_train
):

    n_positive = int(
        (
            y_train
            ==
            1
        ).sum()
    )

    n_negative = int(
        (
            y_train
            ==
            0
        ).sum()
    )


    if config[
        "sampler"
    ] == "EPISODE_BALANCED":

        # Deliberately avoid double-balancing.
        return (
            nn.BCEWithLogitsLoss(),
            1.0
        )


    loss_family = config[
        "loss_family"
    ]


    if loss_family == "BCE":

        ratio = (
            n_negative
            /
            n_positive
        )


        pos_weight_power = config[
            "pos_weight_power"
        ]


        pos_weight = (
            ratio
            **
            pos_weight_power
        )


        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(
                pos_weight,
                dtype=torch.float32,
                device=DEVICE
            )
        )


        return (
            criterion,
            float(
                pos_weight
            )
        )


    if loss_family == "FOCAL":

        criterion = BinaryFocalLoss(
            gamma=config[
                "focal_gamma"
            ],
            alpha=config[
                "focal_alpha"
            ]
        ).to(
            DEVICE
        )


        return (
            criterion,
            1.0
        )


    if loss_family == "CB_FOCAL":

        criterion = ClassBalancedFocalLoss(
            n_negative=n_negative,
            n_positive=n_positive,
            beta=config[
                "cb_beta"
            ],
            gamma=config[
                "focal_gamma"
            ]
        ).to(
            DEVICE
        )


        return (
            criterion,
            1.0
        )


    raise ValueError(
        f"Unknown loss family: {loss_family}"
    )


# ==================================================================================================
# ADVANCED LSTM
# ==================================================================================================

class AdvancedLSTM(
    nn.Module
):

    def __init__(
        self,
        hidden_size,
        num_layers,
        dropout,
        bidirectional,
        head
    ):

        super().__init__()


        self.hidden_size = int(
            hidden_size
        )

        self.num_layers = int(
            num_layers
        )

        self.bidirectional = bool(
            bidirectional
        )

        self.head_type = str(
            head
        )


        self.direction_count = (
            2
            if self.bidirectional
            else 1
        )


        self.output_size = (
            self.hidden_size
            *
            self.direction_count
        )


        self.lstm = nn.LSTM(
            input_size=FEATURE_COUNT,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=(
                float(
                    dropout
                )

                if self.num_layers
                >
                1

                else 0.0
            ),
            bidirectional=self.bidirectional,
        )


        self.dropout = nn.Dropout(
            float(
                dropout
            )
        )


        if self.head_type == "LAST":

            self.classifier = nn.Linear(
                self.output_size,
                1
            )


        elif self.head_type == "POOL":

            representation_size = (
                self.output_size
                *
                3
            )


            self.norm = nn.LayerNorm(
                representation_size
            )


            self.classifier = nn.Sequential(
                nn.Linear(
                    representation_size,
                    self.output_size
                ),
                nn.GELU(),
                nn.Dropout(
                    float(
                        dropout
                    )
                ),
                nn.Linear(
                    self.output_size,
                    1
                ),
            )


        elif self.head_type == "ATTENTION":

            self.attention_score = nn.Linear(
                self.output_size,
                1
            )


            representation_size = (
                self.output_size
                *
                2
            )


            self.norm = nn.LayerNorm(
                representation_size
            )


            self.classifier = nn.Sequential(
                nn.Linear(
                    representation_size,
                    self.output_size
                ),
                nn.GELU(),
                nn.Dropout(
                    float(
                        dropout
                    )
                ),
                nn.Linear(
                    self.output_size,
                    1
                ),
            )


        else:

            raise ValueError(
                f"Unknown head: {self.head_type}"
            )


    def final_hidden_representation(
        self,
        hidden
    ):

        if not self.bidirectional:

            return hidden[
                -1
            ]


        batch_size = hidden.shape[
            1
        ]


        reshaped = hidden.view(
            self.num_layers,
            2,
            batch_size,
            self.hidden_size
        )


        forward_last = reshaped[
            -1,
            0
        ]


        backward_last = reshaped[
            -1,
            1
        ]


        return torch.cat(
            [
                forward_last,
                backward_last,
            ],
            dim=1
        )


    def forward(
        self,
        x
    ):

        sequence_output, (
            hidden,
            _
        ) = self.lstm(
            x
        )


        last_representation = (
            self.final_hidden_representation(
                hidden
            )
        )


        if self.head_type == "LAST":

            representation = self.dropout(
                last_representation
            )


        elif self.head_type == "POOL":

            mean_pool = sequence_output.mean(
                dim=1
            )


            max_pool = sequence_output.max(
                dim=1
            ).values


            representation = torch.cat(
                [
                    last_representation,
                    mean_pool,
                    max_pool,
                ],
                dim=1
            )


            representation = self.norm(
                representation
            )


        else:

            attention_logits = (
                self.attention_score(
                    sequence_output
                )
                .squeeze(-1)
            )


            attention_weights = torch.softmax(
                attention_logits,
                dim=1
            )


            context = torch.sum(
                sequence_output
                *
                attention_weights.unsqueeze(
                    -1
                ),
                dim=1
            )


            representation = torch.cat(
                [
                    last_representation,
                    context,
                ],
                dim=1
            )


            representation = self.norm(
                representation
            )


        logits = self.classifier(
            representation
        ).squeeze(-1)


        return logits


# ==================================================================================================
# DATA LOADERS
# ==================================================================================================

def make_loaders(
    X_train,
    y_train,
    episode_train,
    X_eval,
    y_eval,
    config,
    seed
):

    train_dataset = TensorDataset(
        torch.from_numpy(
            X_train
        ),
        torch.from_numpy(
            y_train
        ),
    )


    eval_dataset = TensorDataset(
        torch.from_numpy(
            X_eval
        ),
        torch.from_numpy(
            y_eval
        ),
    )


    generator = torch.Generator()

    generator.manual_seed(
        seed
    )


    if config[
        "sampler"
    ] == "EPISODE_BALANCED":

        weights = episode_balanced_weights(
            y_train,
            episode_train
        )


        sampler = WeightedRandomSampler(
            weights=torch.tensor(
                weights,
                dtype=torch.double
            ),
            num_samples=len(
                weights
            ),
            replacement=True,
            generator=generator,
        )


        train_loader = DataLoader(
            train_dataset,
            batch_size=config[
                "batch_size"
            ],
            sampler=sampler,
            shuffle=False,
            num_workers=0,
            pin_memory=AMP_ENABLED,
        )


    else:

        train_loader = DataLoader(
            train_dataset,
            batch_size=config[
                "batch_size"
            ],
            shuffle=True,
            generator=generator,
            num_workers=0,
            pin_memory=AMP_ENABLED,
        )


    eval_loader = DataLoader(
        eval_dataset,
        batch_size=max(
            512,
            config[
                "batch_size"
            ]
        ),
        shuffle=False,
        num_workers=0,
        pin_memory=AMP_ENABLED,
    )


    return (
        train_loader,
        eval_loader
    )


# ==================================================================================================
# PREDICTION
# ==================================================================================================

@torch.no_grad()
def predict_scores(
    model,
    loader
):

    model.eval()


    targets = []

    probabilities = []


    for xb, yb in loader:

        xb = xb.to(
            DEVICE,
            non_blocking=True
        )


        with torch.cuda.amp.autocast(
            enabled=AMP_ENABLED
        ):

            logits = model(
                xb
            )


        probs = torch.sigmoid(
            logits
        )


        targets.append(
            yb.cpu().numpy()
        )


        probabilities.append(
            probs.cpu().numpy()
        )


    return (
        np.concatenate(
            targets
        ).astype(
            np.int8
        ),

        np.concatenate(
            probabilities
        ).astype(
            np.float64
        ),
    )


# ==================================================================================================
# F1 THRESHOLD
# ==================================================================================================

def choose_f1_threshold(
    y_true,
    scores
):

    precision, recall, thresholds = (
        precision_recall_curve(
            y_true,
            scores
        )
    )


    if len(
        thresholds
    ) == 0:

        return 0.5


    precision = precision[
        :-1
    ]

    recall = recall[
        :-1
    ]


    denominator = (
        precision
        +
        recall
    )


    f1 = np.divide(
        2
        *
        precision
        *
        recall,
        denominator,
        out=np.zeros_like(
            denominator
        ),
        where=(
            denominator
            >
            0
        )
    )


    best_f1 = f1.max()


    candidates = np.flatnonzero(
        np.isclose(
            f1,
            best_f1,
            atol=1e-12,
            rtol=0.0
        )
    )


    # Phase 12.3 tie rule:
    # higher precision, then higher threshold

    candidate_table = pd.DataFrame(
        {
            "index":
                candidates,

            "precision":
                precision[
                    candidates
                ],

            "threshold":
                thresholds[
                    candidates
                ],
        }
    )


    candidate_table = (
        candidate_table

        .sort_values(
            [
                "precision",
                "threshold"
            ],
            ascending=[
                False,
                False
            ]
        )
    )


    best_index = int(
        candidate_table.iloc[
            0
        ][
            "index"
        ]
    )


    return float(
        thresholds[
            best_index
        ]
    )


# ==================================================================================================
# BASIC METRICS
# ==================================================================================================

def calculate_metrics(
    y_true,
    scores,
    threshold
):

    prediction = (
        scores
        >=
        threshold
    ).astype(
        np.int8
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[
            0,
            1
        ]
    ).ravel()


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_true,
            scores
        )
    )


    return {
        "Average_Precision":
            float(
                average_precision_score(
                    y_true,
                    scores
                )
            ),

        "PR_AUC":
            float(
                auc(
                    recall_curve,
                    precision_curve
                )
            ),

        "ROC_AUC":
            float(
                roc_auc_score(
                    y_true,
                    scores
                )
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    prediction
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    prediction
                )
            ),

        "Precision":
            float(
                precision_score(
                    y_true,
                    prediction,
                    zero_division=0
                )
            ),

        "Recall":
            float(
                recall_score(
                    y_true,
                    prediction,
                    zero_division=0
                )
            ),

        "Specificity":
            float(
                specificity
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    prediction,
                    zero_division=0
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y_true,
                    prediction
                )
            ),

        "Brier":
            float(
                brier_score_loss(
                    y_true,
                    scores
                )
            ),

        "Threshold":
            float(
                threshold
            ),

        "TP":
            int(tp),

        "FP":
            int(fp),

        "TN":
            int(tn),

        "FN":
            int(fn),
    }


# ==================================================================================================
# TRAIN ONE MODEL
# ==================================================================================================

def train_one_model(
    *,
    H,
    config,
    seed,
    X_train,
    y_train,
    episode_train,
    X_eval,
    y_eval,
    max_epochs,
    patience,
    verbose=False
):

    set_seed(
        seed
    )


    train_loader, eval_loader = make_loaders(
        X_train,
        y_train,
        episode_train,
        X_eval,
        y_eval,
        config,
        seed
    )


    model = AdvancedLSTM(
        hidden_size=config[
            "hidden_size"
        ],
        num_layers=config[
            "num_layers"
        ],
        dropout=config[
            "dropout"
        ],
        bidirectional=config[
            "bidirectional"
        ],
        head=config[
            "head"
        ]
    ).to(
        DEVICE
    )


    criterion, applied_pos_weight = build_loss(
        config,
        y_train
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config[
            "learning_rate"
        ],
        weight_decay=config[
            "weight_decay"
        ],
    )


    scheduler = None


    if config[
        "scheduler"
    ] == "PLATEAU":

        scheduler = (
            torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="max",
                factor=0.5,
                patience=2,
                min_lr=1e-6,
            )
        )


    amp_scaler = torch.cuda.amp.GradScaler(
        enabled=AMP_ENABLED
    )


    best_ap = -np.inf

    best_epoch = None

    best_state = None

    patience_counter = 0

    history = []


    for epoch in range(
        1,
        max_epochs
        +
        1
    ):

        model.train()


        running_loss = 0.0

        observations = 0


        for xb, yb in train_loader:

            xb = xb.to(
                DEVICE,
                non_blocking=True
            )

            yb = yb.to(
                DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.cuda.amp.autocast(
                enabled=AMP_ENABLED
            ):

                logits = model(
                    xb
                )

                loss = criterion(
                    logits,
                    yb
                )


            amp_scaler.scale(
                loss
            ).backward()


            amp_scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )


            amp_scaler.step(
                optimizer
            )


            amp_scaler.update()


            running_loss += (
                float(
                    loss.item()
                )
                *
                len(
                    xb
                )
            )


            observations += len(
                xb
            )


        train_loss = (
            running_loss
            /
            observations
        )


        eval_targets, eval_scores = predict_scores(
            model,
            eval_loader
        )


        eval_ap = float(
            average_precision_score(
                eval_targets,
                eval_scores
            )
        )


        if scheduler is not None:

            scheduler.step(
                eval_ap
            )


        history.append(
            {
                "Epoch":
                    epoch,

                "Train_Loss":
                    train_loss,

                "Eval_AP":
                    eval_ap,

                "Learning_Rate":
                    optimizer.param_groups[
                        0
                    ][
                        "lr"
                    ],
            }
        )


        improved = (
            eval_ap
            >
            best_ap
            +
            MIN_DELTA
        )


        if improved:

            best_ap = eval_ap

            best_epoch = epoch


            best_state = {
                key:
                    value.detach().cpu().clone()

                for key, value
                in model.state_dict().items()
            }


            patience_counter = 0


        else:

            patience_counter += 1


        if verbose and (
            epoch == 1
            or
            epoch % 5 == 0
            or
            improved
        ):

            print(
                f"H{H} | "
                f"epoch={epoch:02d} | "
                f"loss={train_loss:.5f} | "
                f"AP={eval_ap:.6f} | "
                f"best={best_ap:.6f}"
            )


        if patience_counter >= patience:

            break


    if best_state is None:

        raise RuntimeError(
            "No best state generated."
        )


    model.load_state_dict(
        best_state
    )

    model = model.to(
        DEVICE
    )


    eval_targets, eval_scores = predict_scores(
        model,
        eval_loader
    )


    threshold = choose_f1_threshold(
        eval_targets,
        eval_scores
    )


    metrics = calculate_metrics(
        eval_targets,
        eval_scores,
        threshold
    )


    return {
        "state_dict":
            best_state,

        "metrics":
            metrics,

        "scores":
            eval_scores,

        "targets":
            eval_targets,

        "best_epoch":
            int(
                best_epoch
            ),

        "history":
            pd.DataFrame(
                history
            ),

        "parameter_count":
            int(
                sum(
                    parameter.numel()

                    for parameter
                    in model.parameters()
                )
            ),

        "applied_pos_weight":
            float(
                applied_pos_weight
            ),
    }


# ==================================================================================================
# BUILD HORIZON/FOLD DATA
# ==================================================================================================

def get_cv_fold_data(
    H,
    fold_cache
):

    horizon_column = HORIZON_TO_COLUMN[
        H
    ]


    eligible = (
        ELIGIBLE_ALL[
            :,
            horizon_column
        ]
        ==
        1
    )


    train_episode_mask = np.isin(
        EPISODE_IDS,
        list(
            fold_cache[
                "train_episodes"
            ]
        )
    )


    dev_episode_mask = np.isin(
        EPISODE_IDS,
        list(
            fold_cache[
                "dev_episodes"
            ]
        )
    )


    train_mask = (
        eligible
        &
        train_episode_mask
    )


    dev_mask = (
        eligible
        &
        dev_episode_mask
    )


    if np.any(
        train_mask
        &
        dev_mask
    ):

        raise RuntimeError(
            "Sample overlap inside CV fold."
        )


    X_train = np.asarray(
        fold_cache[
            "X_scaled"
        ][
            train_mask
        ],
        dtype=np.float32
    )


    X_dev = np.asarray(
        fold_cache[
            "X_scaled"
        ][
            dev_mask
        ],
        dtype=np.float32
    )


    y_train = np.asarray(
        Y_ALL[
            train_mask,
            horizon_column
        ],
        dtype=np.float32
    )


    y_dev = np.asarray(
        Y_ALL[
            dev_mask,
            horizon_column
        ],
        dtype=np.float32
    )


    episode_train = EPISODE_IDS[
        train_mask
    ]


    episode_dev = EPISODE_IDS[
        dev_mask
    ]


    return {
        "X_train":
            X_train,

        "y_train":
            y_train,

        "episode_train":
            episode_train,

        "X_dev":
            X_dev,

        "y_dev":
            y_dev,

        "episode_dev":
            episode_dev,
    }


# ==================================================================================================
# VERIFY CV SUPPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("CV SUPPORT")
print("=" * 120)


for H in HORIZONS:

    for fold_cache in CV_CACHE:

        data = get_cv_fold_data(
            H,
            fold_cache
        )


        print(
            f"H{H} fold {fold_cache['fold']} | "
            f"TRAIN={len(data['y_train']):,} "
            f"pos={int(data['y_train'].sum()):,} | "
            f"DEV={len(data['y_dev']):,} "
            f"pos={int(data['y_dev'].sum()):,}"
        )


        del data

        gc.collect()


# ==================================================================================================
# RESOLVE OPTUNA CONFIGURATION
# ==================================================================================================

def suggest_config(
    trial
):

    config = {
        "hidden_size":
            trial.suggest_categorical(
                "hidden_size",
                [
                    32,
                    64,
                    96,
                    128,
                ]
            ),

        "num_layers":
            trial.suggest_int(
                "num_layers",
                1,
                3
            ),

        "bidirectional":
            trial.suggest_categorical(
                "bidirectional",
                [
                    False,
                    True,
                ]
            ),

        "head":
            trial.suggest_categorical(
                "head",
                [
                    "LAST",
                    "POOL",
                    "ATTENTION",
                ]
            ),

        "dropout":
            trial.suggest_float(
                "dropout",
                0.05,
                0.40,
                step=0.05
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                2e-4,
                2e-3,
                log=True
            ),

        "weight_decay":
            trial.suggest_float(
                "weight_decay",
                1e-6,
                3e-3,
                log=True
            ),

        "batch_size":
            trial.suggest_categorical(
                "batch_size",
                [
                    256,
                    512,
                ]
            ),

        "scheduler":
            trial.suggest_categorical(
                "scheduler",
                [
                    "NONE",
                    "PLATEAU",
                ]
            ),

        "sampler":
            trial.suggest_categorical(
                "sampler",
                [
                    "NATURAL",
                    "EPISODE_BALANCED",
                ]
            ),
    }


    if config[
        "sampler"
    ] == "EPISODE_BALANCED":

        # Avoid simultaneously resampling and applying strong class loss weights.

        config[
            "loss_family"
        ] = "BCE"

        config[
            "pos_weight_power"
        ] = 0.0


    else:

        config[
            "loss_family"
        ] = trial.suggest_categorical(
            "loss_family",
            [
                "BCE",
                "FOCAL",
                "CB_FOCAL",
            ]
        )


        if config[
            "loss_family"
        ] == "BCE":

            config[
                "pos_weight_power"
            ] = trial.suggest_categorical(
                "pos_weight_power",
                [
                    0.0,
                    0.5,
                    1.0,
                ]
            )


        elif config[
            "loss_family"
        ] == "FOCAL":

            config[
                "focal_gamma"
            ] = trial.suggest_categorical(
                "focal_gamma",
                [
                    1.0,
                    2.0,
                    3.0,
                ]
            )


            config[
                "focal_alpha"
            ] = trial.suggest_categorical(
                "focal_alpha",
                [
                    0.25,
                    0.50,
                    0.75,
                ]
            )


        else:

            config[
                "focal_gamma"
            ] = trial.suggest_categorical(
                "focal_gamma",
                [
                    1.0,
                    2.0,
                    3.0,
                ]
            )


            config[
                "cb_beta"
            ] = trial.suggest_categorical(
                "cb_beta",
                [
                    0.99,
                    0.999,
                    0.9999,
                ]
            )


    return config


# ==================================================================================================
# SAVE CELL 6A CONTRACT
# ==================================================================================================

write_json(
    {
        "Stage":
            "CELL_6A",

        "Optimizer":
            "LSTM_OPT_V2_CLEAN_CV",

        "History_Window":
            10,

        "Horizons":
            [
                5,
                10
            ],

        "CV_Folds":
            3,

        "CV_Split_Unit":
            "Episode_ID",

        "Fold_Preprocessing":
            "log1p + StandardScaler fit independently on fold-TRAIN unique minutes only",

        "HPO_Trials_Per_Horizon":
            HPO_TRIALS_PER_HORIZON,

        "HPO_Max_Epochs":
            HPO_MAX_EPOCHS,

        "HPO_Patience":
            HPO_PATIENCE,

        "Search_Axes": [
            "hidden size",
            "layer count",
            "bidirectionality",
            "temporal head",
            "dropout",
            "learning rate",
            "weight decay",
            "batch size",
            "scheduler",
            "sampler",
            "loss family",
            "class weighting/focal parameters",
        ],

        "TEST_Path_Defined":
            False,

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "Phase_12_3_Lock_SHA256":
            protocol_hash,
    },

    OPT_ROOT
    / "CELL6A_OPTIMIZATION_CONTRACT.json"
)


print("\n" + "=" * 120)
print("✓ CELL 6A COMPLETE")
print("=" * 120)

print(
    "Clean episode-level 3-fold CV prepared ✓"
)

print(
    "Fold-specific scaler fit only on fold-TRAIN episodes ✓"
)

print(
    "Losses ready: BCE / Focal / Class-Balanced Focal ✓"
)

print(
    "Heads ready: LAST / POOL / ATTENTION ✓"
)

print(
    "Bidirectional LSTM candidate ready ✓"
)

print(
    "Episode-balanced sampler ready ✓"
)

print(
    "TEST path defined: NO ✓"
)

print(
    "\nRun CELL 6B next."
)

In [ ]:
# ==================================================================================================
# CELL 6B — CLEAN 3-FOLD OPTUNA LSTM SEARCH
#
# MODEL SELECTION USES ONLY INTERNAL TRAIN CV.
#
# OFFICIAL VALIDATION IS NOT LOADED HERE.
# TEST IS NOT LOADED HERE.
# ==================================================================================================

print("=" * 120)
print("CELL 6B — 3-FOLD EPISODE-CV OPTUNA SEARCH")
print("=" * 120)


# ==================================================================================================
# OPTUNA STORAGE
#
# Stored on Drive so the study can resume after a Colab disconnect.
# ==================================================================================================

OPTUNA_DB = (
    OPT_ROOT
    / "optuna_lstm_v2.sqlite3"
)


STORAGE_URL = (
    "sqlite:///"
    +
    str(
        OPTUNA_DB
    )
)


# ==================================================================================================
# ANCHOR TRIALS
#
# These guarantee that the search explicitly evaluates several meaningful
# configurations instead of leaving everything to random TPE exploration.
# ==================================================================================================

ANCHOR_TRIALS = [

    # Original LSTM-like configuration
    {
        "hidden_size":
            64,

        "num_layers":
            2,

        "bidirectional":
            False,

        "head":
            "LAST",

        "dropout":
            0.30,

        "learning_rate":
            0.001,

        "weight_decay":
            0.001,

        "batch_size":
            256,

        "scheduler":
            "NONE",

        "sampler":
            "NATURAL",

        "loss_family":
            "BCE",

        "pos_weight_power":
            1.0,
    },


    # V1 suggested unweighted BCE may be better.
    {
        "hidden_size":
            64,

        "num_layers":
            2,

        "bidirectional":
            False,

        "head":
            "LAST",

        "dropout":
            0.30,

        "learning_rate":
            0.001,

        "weight_decay":
            0.001,

        "batch_size":
            256,

        "scheduler":
            "PLATEAU",

        "sampler":
            "NATURAL",

        "loss_family":
            "BCE",

        "pos_weight_power":
            0.0,
    },


    # Attention + intermediate weighting
    {
        "hidden_size":
            96,

        "num_layers":
            2,

        "bidirectional":
            False,

        "head":
            "ATTENTION",

        "dropout":
            0.25,

        "learning_rate":
            0.0008,

        "weight_decay":
            0.0003,

        "batch_size":
            256,

        "scheduler":
            "PLATEAU",

        "sampler":
            "NATURAL",

        "loss_family":
            "BCE",

        "pos_weight_power":
            0.5,
    },


    # Bidirectional + attention + focal
    {
        "hidden_size":
            64,

        "num_layers":
            2,

        "bidirectional":
            True,

        "head":
            "ATTENTION",

        "dropout":
            0.25,

        "learning_rate":
            0.0007,

        "weight_decay":
            0.0003,

        "batch_size":
            256,

        "scheduler":
            "PLATEAU",

        "sampler":
            "NATURAL",

        "loss_family":
            "FOCAL",

        "focal_gamma":
            2.0,

        "focal_alpha":
            0.50,
    },
]


# ==================================================================================================
# OBJECTIVE
# ==================================================================================================

def objective(
    trial,
    H
):

    config = suggest_config(
        trial
    )


    fold_aps = []

    fold_f1s = []

    fold_mccs = []

    fold_rocs = []

    fold_best_epochs = []

    parameter_count = None


    for fold_position, fold_cache in enumerate(
        CV_CACHE,
        start=1
    ):

        fold_data = get_cv_fold_data(
            H,
            fold_cache
        )


        result = train_one_model(
            H=H,
            config=config,
            seed=(
                DEVELOPMENT_SEED
                +
                fold_position
            ),
            X_train=fold_data[
                "X_train"
            ],
            y_train=fold_data[
                "y_train"
            ],
            episode_train=fold_data[
                "episode_train"
            ],
            X_eval=fold_data[
                "X_dev"
            ],
            y_eval=fold_data[
                "y_dev"
            ],
            max_epochs=HPO_MAX_EPOCHS,
            patience=HPO_PATIENCE,
            verbose=False,
        )


        fold_aps.append(
            result[
                "metrics"
            ][
                "Average_Precision"
            ]
        )


        fold_f1s.append(
            result[
                "metrics"
            ][
                "F1"
            ]
        )


        fold_mccs.append(
            result[
                "metrics"
            ][
                "MCC"
            ]
        )


        fold_rocs.append(
            result[
                "metrics"
            ][
                "ROC_AUC"
            ]
        )


        fold_best_epochs.append(
            result[
                "best_epoch"
            ]
        )


        parameter_count = result[
            "parameter_count"
        ]


        running_mean_ap = float(
            np.mean(
                fold_aps
            )
        )


        trial.report(
            running_mean_ap,
            step=fold_position
        )


        print(
            f"H{H} | "
            f"trial={trial.number:02d} | "
            f"fold={fold_position}/3 | "
            f"AP={fold_aps[-1]:.6f} | "
            f"running mean={running_mean_ap:.6f}"
        )


        del fold_data
        del result

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


        # Do not prune before seeing two independent episode folds.
        if (
            fold_position
            >=
            2

            and

            trial.should_prune()
        ):

            raise optuna.TrialPruned()


    mean_ap = float(
        np.mean(
            fold_aps
        )
    )


    std_ap = float(
        np.std(
            fold_aps,
            ddof=1
        )
    )


    mean_f1 = float(
        np.mean(
            fold_f1s
        )
    )


    mean_mcc = float(
        np.mean(
            fold_mccs
        )
    )


    mean_roc = float(
        np.mean(
            fold_rocs
        )
    )


    trial.set_user_attr(
        "resolved_config",
        config
    )


    trial.set_user_attr(
        "fold_aps",
        [
            float(x)
            for x
            in fold_aps
        ]
    )


    trial.set_user_attr(
        "mean_ap",
        mean_ap
    )


    trial.set_user_attr(
        "std_ap",
        std_ap
    )


    trial.set_user_attr(
        "mean_f1",
        mean_f1
    )


    trial.set_user_attr(
        "mean_mcc",
        mean_mcc
    )


    trial.set_user_attr(
        "mean_roc_auc",
        mean_roc
    )


    trial.set_user_attr(
        "best_epochs",
        [
            int(x)
            for x
            in fold_best_epochs
        ]
    )


    trial.set_user_attr(
        "parameter_count",
        int(
            parameter_count
        )
    )


    print(
        f"\nH{H} TRIAL {trial.number} COMPLETE | "
        f"mean AP={mean_ap:.6f} ± {std_ap:.6f} | "
        f"F1={mean_f1:.6f} | "
        f"MCC={mean_mcc:.6f}\n"
    )


    return mean_ap


# ==================================================================================================
# RUN H5 AND H10 SEARCHES
# ==================================================================================================

SELECTED_CONFIGS = {}

HPO_SUMMARY_ROWS = []


for H in HORIZONS:

    print("\n" + "#" * 120)
    print(
        f"OPTUNA CLEAN CV SEARCH — H{H}"
    )
    print("#" * 120)


    sampler = optuna.samplers.TPESampler(
        seed=HPO_SEED
        +
        H
    )


    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=4,
        n_warmup_steps=2,
    )


    study = optuna.create_study(
        study_name=f"LSTM_OPT_V2_H{H}",
        storage=STORAGE_URL,
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        load_if_exists=True,
    )


    if len(
        study.trials
    ) == 0:

        for anchor in ANCHOR_TRIALS:

            study.enqueue_trial(
                anchor
            )


    finished_trial_count = sum(
        trial.state
        in {
            optuna.trial.TrialState.COMPLETE,
            optuna.trial.TrialState.PRUNED,
            optuna.trial.TrialState.FAIL,
        }

        for trial
        in study.trials
    )


    remaining_trials = max(
        0,
        HPO_TRIALS_PER_HORIZON
        -
        finished_trial_count
    )


    print(
        "Existing finished trials:",
        finished_trial_count
    )

    print(
        "Trials still requested:",
        remaining_trials
    )


    if remaining_trials > 0:

        study.optimize(
            lambda trial:
                objective(
                    trial,
                    H
                ),
            n_trials=remaining_trials,
            gc_after_trial=True,
        )


    completed_trials = [
        trial

        for trial
        in study.trials

        if trial.state
        ==
        optuna.trial.TrialState.COMPLETE
    ]


    if not completed_trials:

        raise RuntimeError(
            f"H{H}: no completed Optuna trials."
        )


    # ==============================================================================================
    # SAVE RAW STUDY TABLE
    # ==============================================================================================

    study.trials_dataframe().to_csv(
        OPT_ROOT
        / f"H{H:02d}_OPTUNA_TRIALS.csv",
        index=False
    )


    # ==============================================================================================
    # SCIENTIFIC SELECTION RULE
    #
    # 1. max mean CV AP
    # 2. anything within 0.002 AP is considered practically tied
    # 3. among tied configurations:
    #       higher mean CV F1
    #       lower parameter count
    #       lower trial number
    # ==============================================================================================

    best_cv_ap = max(
        float(
            trial.value
        )

        for trial
        in completed_trials
    )


    candidates = [
        trial

        for trial
        in completed_trials

        if (
            best_cv_ap
            -
            float(
                trial.value
            )
        )
        <=
        AP_TIE_TOLERANCE
    ]


    candidates = sorted(
        candidates,
        key=lambda trial: (
            -float(
                trial.user_attrs.get(
                    "mean_f1",
                    -np.inf
                )
            ),
            int(
                trial.user_attrs.get(
                    "parameter_count",
                    10**12
                )
            ),
            int(
                trial.number
            ),
        )
    )


    selected_trial = candidates[
        0
    ]


    selected_config = selected_trial.user_attrs[
        "resolved_config"
    ]


    SELECTED_CONFIGS[
        H
    ] = selected_config


    selection_record = {
        "Horizon":
            H,

        "Selected_Trial":
            int(
                selected_trial.number
            ),

        "Selected_Mean_CV_AP":
            float(
                selected_trial.value
            ),

        "Selected_CV_AP_Std":
            float(
                selected_trial.user_attrs[
                    "std_ap"
                ]
            ),

        "Selected_Mean_CV_F1":
            float(
                selected_trial.user_attrs[
                    "mean_f1"
                ]
            ),

        "Selected_Mean_CV_MCC":
            float(
                selected_trial.user_attrs[
                    "mean_mcc"
                ]
            ),

        "Selected_Mean_CV_ROC_AUC":
            float(
                selected_trial.user_attrs[
                    "mean_roc_auc"
                ]
            ),

        "Parameter_Count":
            int(
                selected_trial.user_attrs[
                    "parameter_count"
                ]
            ),

        "Configuration":
            selected_config,
    }


    write_json(
        selection_record,
        OPT_ROOT
        / f"H{H:02d}_SELECTED_CV_CONFIGURATION.json"
    )


    HPO_SUMMARY_ROWS.append(
        {
            "Horizon":
                H,

            "Selected_Trial":
                selected_trial.number,

            "Mean_CV_AP":
                selected_trial.value,

            "CV_AP_Std":
                selected_trial.user_attrs[
                    "std_ap"
                ],

            "Mean_CV_F1":
                selected_trial.user_attrs[
                    "mean_f1"
                ],

            "Mean_CV_MCC":
                selected_trial.user_attrs[
                    "mean_mcc"
                ],

            "Parameter_Count":
                selected_trial.user_attrs[
                    "parameter_count"
                ],

            "Head":
                selected_config[
                    "head"
                ],

            "Hidden_Size":
                selected_config[
                    "hidden_size"
                ],

            "Layers":
                selected_config[
                    "num_layers"
                ],

            "Bidirectional":
                selected_config[
                    "bidirectional"
                ],

            "Loss":
                selected_config[
                    "loss_family"
                ],

            "Sampler":
                selected_config[
                    "sampler"
                ],
        }
    )


    print("\n" + "=" * 120)

    print(
        f"H{H} SELECTED CLEAN-CV CONFIGURATION"
    )


    print(
        json.dumps(
            selected_config,
            indent=4
        )
    )


    print(
        f"Mean CV AP = "
        f"{float(selected_trial.value):.6f}"
    )


    print(
        f"CV AP std  = "
        f"{float(selected_trial.user_attrs['std_ap']):.6f}"
    )


    print(
        f"Mean CV F1 = "
        f"{float(selected_trial.user_attrs['mean_f1']):.6f}"
    )


    print("=" * 120)


# ==================================================================================================
# GLOBAL HPO SUMMARY
# ==================================================================================================

HPO_SUMMARY = pd.DataFrame(
    HPO_SUMMARY_ROWS
)


HPO_SUMMARY.to_csv(
    OPT_ROOT
    / "CLEAN_CV_HPO_SUMMARY.csv",
    index=False
)


write_json(
    {
        "Stage":
            "CELL_6B",

        "Status":
            "CLEAN_CV_HPO_COMPLETE",

        "HPO_Selection_Source":
            "TRAIN_INTERNAL_3_FOLD_EPISODE_CV",

        "Official_Validation_Used_For_HPO":
            False,

        "Selected_Configurations":
            {
                str(H):
                    SELECTED_CONFIGS[
                        H
                    ]

                for H
                in HORIZONS
            },

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,
    },

    OPT_ROOT
    / "CELL6B_HPO_COMPLETE.json"
)


print("\n" + "=" * 120)
print("✓ CELL 6B COMPLETE")
print("=" * 120)

print(
    HPO_SUMMARY.to_string(
        index=False
    )
)

print(
    "\nOfficial VALIDATION used for HPO: NO ✓"
)

print(
    "TEST loaded: NO ✓"
)

print(
    "\nRun CELL 6C next."
)

In [ ]:
# ==================================================================================================
# CELL 6C — FULL TRAIN → OFFICIAL VALIDATION CONFIRMATION
#
# The architecture/loss/sampler were already chosen using TRAIN-internal CV.
#
# Official VALIDATION is now used for:
#   - early stopping
#   - final development metrics
#   - threshold selection
#
# This is permitted by Phase 12.3.
#
# TEST IS STILL NOT LOADED.
# ==================================================================================================

print("=" * 120)
print("CELL 6C — CLEAN-CV SELECTED LSTM → OFFICIAL VALIDATION")
print("=" * 120)


# ==================================================================================================
# OFFICIAL TRAIN-ONLY PREPROCESSED ARRAYS FROM PHASE 12.2
# ==================================================================================================

OFFICIAL_X_TRAIN_FILE = (
    P12_2
    / "transformed"
    / "W10"
    / "train"
    / "X_log1p_standardized.npy"
)

OFFICIAL_X_VAL_FILE = (
    P12_2
    / "transformed"
    / "W10"
    / "validation"
    / "X_log1p_standardized.npy"
)

OFFICIAL_Y_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "y_binary.npy"
)

OFFICIAL_ELIGIBLE_VAL_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "eligible.npy"
)

OFFICIAL_VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)


for path in [
    OFFICIAL_X_TRAIN_FILE,
    OFFICIAL_X_VAL_FILE,
    OFFICIAL_Y_VAL_FILE,
    OFFICIAL_ELIGIBLE_VAL_FILE,
    OFFICIAL_VAL_INDEX_FILE,
]:

    require_file(path)


X_TRAIN_OFFICIAL = np.load(
    OFFICIAL_X_TRAIN_FILE,
    mmap_mode="r"
)

X_VAL_OFFICIAL = np.load(
    OFFICIAL_X_VAL_FILE,
    mmap_mode="r"
)

Y_VAL_OFFICIAL = np.load(
    OFFICIAL_Y_VAL_FILE,
    allow_pickle=False
)

ELIGIBLE_VAL_OFFICIAL = np.load(
    OFFICIAL_ELIGIBLE_VAL_FILE,
    allow_pickle=False
)

VAL_INDEX_OFFICIAL = pd.read_parquet(
    OFFICIAL_VAL_INDEX_FILE
)


VAL_INDEX_OFFICIAL = (
    VAL_INDEX_OFFICIAL
    .copy()
)


VAL_INDEX_OFFICIAL[
    "Episode_ID"
] = (
    VAL_INDEX_OFFICIAL[
        "Episode_ID"
    ]
    .astype(str)
)


# ==================================================================================================
# LOAD SELECTED CONFIGURATIONS FROM DISK
# ==================================================================================================

SELECTED_CONFIGS = {}


for H in HORIZONS:

    selection_file = (
        OPT_ROOT
        / f"H{H:02d}_SELECTED_CV_CONFIGURATION.json"
    )


    selection = read_json(
        selection_file
    )


    SELECTED_CONFIGS[
        H
    ] = selection[
        "Configuration"
    ]


# ==================================================================================================
# EVENT METRICS
# ==================================================================================================

def calculate_event_metrics(
    y_true,
    scores,
    threshold,
    index_df
):

    prediction = (
        scores
        >=
        threshold
    )


    episodes = (
        index_df[
            "Episode_ID"
        ]
        .astype(str)
        .to_numpy()
    )


    tto = pd.to_numeric(
        index_df[
            "Minutes_To_Next_Onset"
        ],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )


    positive = (
        y_true
        ==
        1
    )


    negative = (
        y_true
        ==
        0
    )


    true_positive_warning = (
        prediction
        &
        positive
    )


    false_positive_warning = (
        prediction
        &
        negative
    )


    true_events = set(
        episodes[
            positive
        ]
    )


    recalled_events = set(
        episodes[
            true_positive_warning
        ]
    )


    warned_episodes = set(
        episodes[
            prediction
        ]
    )


    event_recall = (
        len(
            recalled_events
        )
        /
        len(
            true_events
        )
    )


    event_precision = (
        len(
            recalled_events
        )
        /
        len(
            warned_episodes
        )

        if len(
            warned_episodes
        )
        else np.nan
    )


    leads = []


    for episode_id in recalled_events:

        mask = (
            true_positive_warning

            &

            (
                episodes
                ==
                episode_id
            )
        )


        event_tto = tto[
            mask
        ]


        event_tto = event_tto[
            np.isfinite(
                event_tto
            )
        ]


        if len(
            event_tto
        ):

            leads.append(
                float(
                    np.max(
                        event_tto
                    )
                )
            )


    negative_hours = (
        negative.sum()
        /
        60.0
    )


    false_warning_rate = (
        false_positive_warning.sum()
        /
        negative_hours

        if negative_hours > 0
        else np.nan
    )


    return {
        "Event_Recall":
            float(
                event_recall
            ),

        "Event_Precision":
            float(
                event_precision
            ),

        "Median_First_Warning_Lead_Min":
            (
                float(
                    np.median(
                        leads
                    )
                )

                if len(
                    leads
                )
                else np.nan
            ),

        "Mean_First_Warning_Lead_Min":
            (
                float(
                    np.mean(
                        leads
                    )
                )

                if len(
                    leads
                )
                else np.nan
            ),

        "False_Warning_Origins_Per_Hour":
            float(
                false_warning_rate
            ),
    }


# ==================================================================================================
# FINAL THREE-SEED DEVELOPMENT RUN
# ==================================================================================================

FINAL_ROWS = []

HORIZON_SUMMARIES = []


for H in HORIZONS:

    print("\n" + "#" * 120)
    print(
        f"OPTIMIZED LSTM FINAL DEVELOPMENT — H{H}"
    )
    print("#" * 120)


    horizon_column = HORIZON_TO_COLUMN[
        H
    ]


    train_mask = (
        ELIGIBLE_ALL[
            :,
            horizon_column
        ]
        ==
        1
    )


    val_mask = (
        ELIGIBLE_VAL_OFFICIAL[
            :,
            horizon_column
        ]
        ==
        1
    )


    X_train = np.asarray(
        X_TRAIN_OFFICIAL[
            train_mask
        ],
        dtype=np.float32
    )


    y_train = np.asarray(
        Y_ALL[
            train_mask,
            horizon_column
        ],
        dtype=np.float32
    )


    episode_train = EPISODE_IDS[
        train_mask
    ]


    X_val = np.asarray(
        X_VAL_OFFICIAL[
            val_mask
        ],
        dtype=np.float32
    )


    y_val = np.asarray(
        Y_VAL_OFFICIAL[
            val_mask,
            horizon_column
        ],
        dtype=np.float32
    )


    val_index = (
        VAL_INDEX_OFFICIAL
        .loc[
            val_mask
        ]
        .reset_index(
            drop=True
        )
    )


    config = SELECTED_CONFIGS[
        H
    ]


    print(
        "Selected clean-CV configuration:"
    )

    print(
        json.dumps(
            config,
            indent=4
        )
    )


    print(
        "\nTRAIN:",
        X_train.shape,
        "positive=",
        int(
            y_train.sum()
        ),
        "negative=",
        int(
            (
                y_train
                ==
                0
            ).sum()
        )
    )


    print(
        "VALIDATION:",
        X_val.shape,
        "positive=",
        int(
            y_val.sum()
        ),
        "negative=",
        int(
            (
                y_val
                ==
                0
            ).sum()
        )
    )


    horizon_output = (
        OPT_ROOT
        / f"H{H:02d}"
        / "final_replicates"
    )


    horizon_output.mkdir(
        parents=True,
        exist_ok=True
    )


    horizon_rows = []


    for replicate_number, seed in enumerate(
        FINAL_SEEDS,
        start=1
    ):

        print("\n" + "-" * 120)

        print(
            f"H{H} | final replicate "
            f"{replicate_number}/3 | "
            f"seed={seed}"
        )

        print("-" * 120)


        result = train_one_model(
            H=H,
            config=config,
            seed=seed,
            X_train=X_train,
            y_train=y_train,
            episode_train=episode_train,
            X_eval=X_val,
            y_eval=y_val,
            max_epochs=FINAL_MAX_EPOCHS,
            patience=FINAL_PATIENCE,
            verbose=True,
        )


        metrics = result[
            "metrics"
        ]


        event_metrics = calculate_event_metrics(
            result[
                "targets"
            ],
            result[
                "scores"
            ],
            metrics[
                "Threshold"
            ],
            val_index
        )


        record = {
            "Model":
                "LSTM_OPT_V2_CLEAN_CV",

            "Horizon":
                H,

            "Replicate":
                replicate_number,

            "Seed":
                seed,

            "Best_Epoch":
                result[
                    "best_epoch"
                ],

            "Parameter_Count":
                result[
                    "parameter_count"
                ],

            "Applied_Pos_Weight":
                result[
                    "applied_pos_weight"
                ],

            **config,

            **metrics,

            **event_metrics,

            "TEST_Used":
                False,
        }


        FINAL_ROWS.append(
            record
        )


        horizon_rows.append(
            record
        )


        replicate_dir = (
            horizon_output
            / f"seed_{seed}"
        )


        replicate_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        # ------------------------------------------------------------------------------------------
        # MODEL
        # ------------------------------------------------------------------------------------------

        torch.save(
            {
                "model_type":
                    "LSTM_OPT_V2_CLEAN_CV",

                "history_window":
                    10,

                "horizon":
                    H,

                "seed":
                    seed,

                "configuration":
                    config,

                "state_dict":
                    result[
                        "state_dict"
                    ],

                "validation_threshold":
                    metrics[
                        "Threshold"
                    ],

                "phase_12_3_protocol_lock_sha256":
                    protocol_hash,

                "TEST_Used":
                    False,
            },

            replicate_dir
            / "model.pt"
        )


        # ------------------------------------------------------------------------------------------
        # HISTORY
        # ------------------------------------------------------------------------------------------

        result[
            "history"
        ].to_csv(
            replicate_dir
            / "training_history.csv",
            index=False
        )


        # ------------------------------------------------------------------------------------------
        # VALIDATION PREDICTIONS
        # ------------------------------------------------------------------------------------------

        prediction_df = val_index[
            [
                "Episode_ID",
                "Minute_Index",
                "Minutes_To_Next_Onset",
            ]
        ].copy()


        prediction_df[
            "Y_True"
        ] = result[
            "targets"
        ]


        prediction_df[
            "Probability"
        ] = result[
            "scores"
        ]


        prediction_df[
            "Threshold"
        ] = metrics[
            "Threshold"
        ]


        prediction_df[
            "Prediction"
        ] = (
            result[
                "scores"
            ]
            >=
            metrics[
                "Threshold"
            ]
        ).astype(
            np.int8
        )


        prediction_df.to_parquet(
            replicate_dir
            / "validation_predictions.parquet",
            index=False
        )


        write_json(
            record,
            replicate_dir
            / "validation_metrics.json"
        )


        print(
            f"\nAP          = "
            f"{metrics['Average_Precision']:.6f}"
        )

        print(
            f"PR-AUC      = "
            f"{metrics['PR_AUC']:.6f}"
        )

        print(
            f"ROC-AUC     = "
            f"{metrics['ROC_AUC']:.6f}"
        )

        print(
            f"Accuracy    = "
            f"{metrics['Accuracy']:.6f}"
        )

        print(
            f"BalancedAcc = "
            f"{metrics['Balanced_Accuracy']:.6f}"
        )

        print(
            f"Precision   = "
            f"{metrics['Precision']:.6f}"
        )

        print(
            f"Recall      = "
            f"{metrics['Recall']:.6f}"
        )

        print(
            f"F1          = "
            f"{metrics['F1']:.6f}"
        )

        print(
            f"MCC         = "
            f"{metrics['MCC']:.6f}"
        )

        print(
            f"EventRecall = "
            f"{event_metrics['Event_Recall']:.6f}"
        )

        print(
            f"EventPrec   = "
            f"{event_metrics['Event_Precision']:.6f}"
        )

        print(
            f"Median Lead = "
            f"{event_metrics['Median_First_Warning_Lead_Min']}"
        )

        print(
            f"FP origins/hour = "
            f"{event_metrics['False_Warning_Origins_Per_Hour']:.6f}"
        )


        del result

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    horizon_df = pd.DataFrame(
        horizon_rows
    )


    horizon_df.to_csv(
        OPT_ROOT
        / f"H{H:02d}"
        / "FINAL_VALIDATION_REPLICATES.csv",
        index=False
    )


    summary = {
        "Horizon":
            H,

        "Replicates":
            len(
                horizon_df
            ),

        "Average_Precision_Mean":
            float(
                horizon_df[
                    "Average_Precision"
                ].mean()
            ),

        "Average_Precision_Std":
            float(
                horizon_df[
                    "Average_Precision"
                ].std(
                    ddof=1
                )
            ),

        "PR_AUC_Mean":
            float(
                horizon_df[
                    "PR_AUC"
                ].mean()
            ),

        "ROC_AUC_Mean":
            float(
                horizon_df[
                    "ROC_AUC"
                ].mean()
            ),

        "Accuracy_Mean":
            float(
                horizon_df[
                    "Accuracy"
                ].mean()
            ),

        "Balanced_Accuracy_Mean":
            float(
                horizon_df[
                    "Balanced_Accuracy"
                ].mean()
            ),

        "Precision_Mean":
            float(
                horizon_df[
                    "Precision"
                ].mean()
            ),

        "Recall_Mean":
            float(
                horizon_df[
                    "Recall"
                ].mean()
            ),

        "F1_Mean":
            float(
                horizon_df[
                    "F1"
                ].mean()
            ),

        "MCC_Mean":
            float(
                horizon_df[
                    "MCC"
                ].mean()
            ),

        "Brier_Mean":
            float(
                horizon_df[
                    "Brier"
                ].mean()
            ),

        "Event_Recall_Mean":
            float(
                horizon_df[
                    "Event_Recall"
                ].mean()
            ),

        "Event_Precision_Mean":
            float(
                horizon_df[
                    "Event_Precision"
                ].mean()
            ),

        "Median_First_Warning_Lead_Min_Mean":
            float(
                horizon_df[
                    "Median_First_Warning_Lead_Min"
                ].mean()
            ),

        "False_Warning_Origins_Per_Hour_Mean":
            float(
                horizon_df[
                    "False_Warning_Origins_Per_Hour"
                ].mean()
            ),

        "Configuration":
            config,

        "TEST_Used":
            False,
    }


    HORIZON_SUMMARIES.append(
        summary
    )


    del X_train
    del y_train
    del episode_train
    del X_val
    del y_val
    del val_index

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ==================================================================================================
# SAVE OPTIMIZED FINAL SUMMARY
# ==================================================================================================

FINAL_DF = pd.DataFrame(
    FINAL_ROWS
)


FINAL_DF.to_csv(
    OPT_ROOT
    / "ALL_FINAL_VALIDATION_REPLICATES.csv",
    index=False
)


OPTIMIZED_SUMMARY = pd.DataFrame(
    HORIZON_SUMMARIES
)


OPTIMIZED_SUMMARY.to_csv(
    OPT_ROOT
    / "LSTM_OPT_V2_VALIDATION_SUMMARY.csv",
    index=False
)


# ==================================================================================================
# COMPARE AGAINST ORIGINAL BASELINE
# ==================================================================================================

BASELINE_SUMMARY = pd.read_csv(
    BASELINE_SUMMARY_FILE
)


comparison_rows = []


for H in HORIZONS:

    baseline_row = (
        BASELINE_SUMMARY
        .loc[
            BASELINE_SUMMARY[
                "Horizon"
            ]
            ==
            H
        ]
        .iloc[
            0
        ]
    )


    optimized_row = (
        OPTIMIZED_SUMMARY
        .loc[
            OPTIMIZED_SUMMARY[
                "Horizon"
            ]
            ==
            H
        ]
        .iloc[
            0
        ]
    )


    baseline_ap = float(
        baseline_row[
            "Average_Precision_Mean"
        ]
    )


    optimized_ap = float(
        optimized_row[
            "Average_Precision_Mean"
        ]
    )


    delta_ap = (
        optimized_ap
        -
        baseline_ap
    )


    baseline_f1 = float(
        baseline_row[
            "F1_Mean"
        ]
    )


    optimized_f1 = float(
        optimized_row[
            "F1_Mean"
        ]
    )


    baseline_mcc = float(
        baseline_row[
            "MCC_Mean"
        ]
    )


    optimized_mcc = float(
        optimized_row[
            "MCC_Mean"
        ]
    )


    baseline_event_recall = float(
        baseline_row[
            "Event_Recall_Mean"
        ]
    )


    optimized_event_recall = float(
        optimized_row[
            "Event_Recall_Mean"
        ]
    )


    if delta_ap > AP_TIE_TOLERANCE:

        recommendation = (
            "OPTIMIZED_LSTM_V2"
        )

    else:

        recommendation = (
            "ORIGINAL_LSTM_BASELINE"
        )


    comparison_rows.append(
        {
            "Horizon":
                H,

            "Baseline_AP":
                baseline_ap,

            "Optimized_AP":
                optimized_ap,

            "AP_Absolute_Change":
                delta_ap,

            "AP_Relative_Change_Percent":
                (
                    100.0
                    *
                    delta_ap
                    /
                    baseline_ap
                ),

            "Baseline_F1":
                baseline_f1,

            "Optimized_F1":
                optimized_f1,

            "F1_Change":
                (
                    optimized_f1
                    -
                    baseline_f1
                ),

            "Baseline_MCC":
                baseline_mcc,

            "Optimized_MCC":
                optimized_mcc,

            "MCC_Change":
                (
                    optimized_mcc
                    -
                    baseline_mcc
                ),

            "Baseline_Event_Recall":
                baseline_event_recall,

            "Optimized_Event_Recall":
                optimized_event_recall,

            "Event_Recall_Change":
                (
                    optimized_event_recall
                    -
                    baseline_event_recall
                ),

            "Recommended_LSTM":
                recommendation,
        }
    )


comparison = pd.DataFrame(
    comparison_rows
)


comparison.to_csv(
    OPT_ROOT
    / "ORIGINAL_VS_CLEAN_OPTIMIZED_LSTM.csv",
    index=False
)


# ==================================================================================================
# PRINT FINAL DEVELOPMENT SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("CLEAN OPTIMIZED LSTM — OFFICIAL VALIDATION SUMMARY")
print("=" * 120)


display_columns = [
    "Horizon",
    "Average_Precision_Mean",
    "Average_Precision_Std",
    "PR_AUC_Mean",
    "ROC_AUC_Mean",
    "Accuracy_Mean",
    "Balanced_Accuracy_Mean",
    "Precision_Mean",
    "Recall_Mean",
    "F1_Mean",
    "MCC_Mean",
    "Event_Recall_Mean",
    "Event_Precision_Mean",
    "Median_First_Warning_Lead_Min_Mean",
    "False_Warning_Origins_Per_Hour_Mean",
]


print(
    OPTIMIZED_SUMMARY[
        display_columns
    ].to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("ORIGINAL VS CLEAN OPTIMIZED LSTM")
print("=" * 120)


print(
    comparison.to_string(
        index=False
    )
)


# ==================================================================================================
# FINAL DEVELOPMENT LOCK
# ==================================================================================================

write_json(
    {
        "Model":
            "LSTM",

        "Optimization":
            "V2_CLEAN_EPISODE_CV",

        "Status":
            "PRE_TEST_OPTIMIZATION_COMPLETE",

        "Original_LSTM_Preserved":
            True,

        "Internal_Selection":
            "3-fold episode-level TRAIN CV",

        "Fold_Preprocessing":
            "log1p + StandardScaler fit on fold-TRAIN only",

        "Official_Validation_Used_For_HPO":
            False,

        "Official_Validation_Used_For_Final_Development":
            True,

        "Final_Seeds":
            FINAL_SEEDS,

        "Replacement_Rule":
            (
                "Optimized LSTM replaces baseline only "
                "if official validation AP gain > 0.002"
            ),

        "Recommendations":
            {
                str(
                    row[
                        "Horizon"
                    ]
                ):
                    row[
                        "Recommended_LSTM"
                    ]

                for _, row
                in comparison.iterrows()
            },

        "TEST_Path_Defined":
            False,

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "TEST_Evaluation_Performed":
            False,

        "Phase_12_3_Lock_SHA256":
            protocol_hash,
    },

    OPT_ROOT
    / "LSTM_OPT_V2_DEVELOPMENT_LOCK.json"
)


print("\n" + "=" * 120)
print("✓ CELL 6C COMPLETE")
print("=" * 120)


print(
    "\nOriginal LSTM preserved ✓"
)

print(
    "Clean fold-specific preprocessing used during HPO ✓"
)

print(
    "Architecture/loss selected with TRAIN-internal CV only ✓"
)

print(
    "Official validation used only after HPO selection ✓"
)

print(
    "Three frozen seeds completed ✓"
)

print(
    "TEST path defined: NO ✓"
)

print(
    "TEST loaded: NO ✓"
)

print(
    "TEST evaluated: NO ✓"
)


print(
    "\nSaved to:"
)

print(
    OPT_ROOT
)


print(
    "\nDO NOT RUN TEST."
)

print("=" * 120)

gc.collect()